# Public release note

This is a sanitized public copy of `omniASR_CTC_1B_VS_FN_6language_SOM(1).ipynb`. Cell outputs, execution counters,
Colab user metadata, widget state, embedded display payloads, private storage
paths, and internal run identifiers were removed before publication. The
experiment source is preserved, but public placeholders must be configured from
your own generated contracts before rerunning dependent stages. See the repository
README and `docs/REPRODUCIBILITY.md` for prerequisites, data access, execution
order, expected artifacts, and the English explanation of this experiment.

Never paste credentials into a notebook. Use Colab Secrets or environment
variables (`HF_TOKEN`, `KAGGLE_USERNAME`, and `KAGGLE_KEY`).


# AfriVoices East Africa ASR Hackathon — Prompt 2/2
## Fine-tuning multilingue joint `omniASR_CTC_1B_v2` (fairseq2)

**Entrée :** dataset curé du Prompt 1 (456 h, FLAC 16 kHz mono, `curated_manifest_all.csv`).
**Sortie :** checkpoint fine-tuné + `submission.csv` (colonnes `id,language,transcription`).

**Ordre :** 1→5 (préparation) → 6 (calibration ~15 min) → 7 (FAST ~1-2 h, GO/NO-GO)
→ 8 (run complet, multi-sessions) → 9 (évaluation) → 10 (soumission).

> ⚠️ Chaque phase coûteuse est bloquée par un drapeau `RUN_*` (section 2.2), `False` par défaut.
> Quatre points d'« auto-découverte » s'arrêtent proprement tant qu'ils ne sont pas résolus :
> tokenizer (4.2), format TSV (5.1→5.2), chargement de checkpoint (7.3), parseur test Kaggle (10.2),
> top-up depuis le checkpoint joint (9.2).
> **Session fraîche (reprise du run complet)** : exécuter 1→5, remettre `RUN_FULL_TRAINING=True`
> en 2.2, puis 8.1 → 8.2 — les fonctions d'entraînement vivent en 2.4, donc dans le chemin 1→5.


## 1 — Setup : Drive, dépendances, GPU, credentials Kaggle

In [ ]:
# 1.1 — Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1.2 — Dépendances : clone du dépôt PUIS installation DEPUIS le clone.
# Pourquoi pas PyPI ? La 0.2.0 publiée déclare Requires-Python <=3.12 (borne mal écrite) : le
# Python 3.12.12 de Colab la dépasse au sens strict et pip retombe silencieusement sur la 0.1.0,
# qui n'a PAS les cartes v2. Le dépôt a la borne corrigée (<3.14) et aligne code+cartes+recettes.
import subprocess, os
_torch_before, _numpy_before = None, None
try:
    import torch as _t0; _torch_before = _t0.__version__
except Exception:
    pass
try:
    import numpy as _n0; _numpy_before = _n0.__version__
except Exception:
    pass
OMNI_REPO_DIR = "/content/omnilingual-asr"
if not os.path.isdir(OMNI_REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/facebookresearch/omnilingual-asr.git", OMNI_REPO_DIR], check=True)

# PATCH idempotent du calculateur de WER du dépôt : une hypothèse CTC vide (que des blanks,
# légitime en cours d'entraînement) fait planter pad_seqs à la VALIDATION
# ("All lengths in seq_lens must be >= 1"). On substitue un unique token de padding
# (retiré au détokenisage -> chaîne vide, WER correct). À réappliquer à chaque clone frais.
import re as _re
_wc = os.path.join(OMNI_REPO_DIR, "workflows/recipes/wav2vec2/asr/wer_calculator.py")
_src = open(_wc, encoding="utf-8").read()
if "hyp_seq.new_full" in _src:
    print("PATCH wer_calculator : déjà appliqué.")
else:
    _pat = "( +)" + _re.escape("hyp_seq = hyp_seq[hyp_seq != self._blank_label]")
    _m = _re.search(_pat, _src)
    assert _m, "wer_calculator.py a changé : m'envoyer son contenu pour adapter le patch."
    _ind = _m.group(1)
    _NL = chr(10)
    _ins = (_m.group(0) + _NL + _ind + "if hyp_seq.numel() == 0:"
            + "  # patch: hypothese vide -> pad_seqs exige len >= 1" + _NL
            + _ind + "    hyp_seq = hyp_seq.new_full((1,), self._pad_idx)")
    open(_wc, "w", encoding="utf-8").write(_src.replace(_m.group(0), _ins))
    print("PATCH wer_calculator : appliqué (hypothèses vides gérées).")
!pip install -q /content/omnilingual-asr 'pyarrow>=20.0.0' soundfile jiwer pandas
import importlib.metadata as _md
# fairseq2 impose sa version de torch ; les torchaudio/torchvision préinstallés (compilés pour
# l'ancien torch) casseraient les imports du package (undefined symbol) -> alignement forcé.
_torch_base = _md.version("torch").split("+")[0]
_torch_mm = ".".join(_torch_base.split(".")[:2])
_TV_FOR_TORCH = {"2.8": "0.23.0", "2.9": "0.24.0", "2.10": "0.25.0", "2.11": "0.26.0"}
_targets = {"torchaudio": _torch_base, "torchvision": _TV_FOR_TORCH.get(_torch_mm)}
_realigned = False
for _pkg, _ver in _targets.items():
    if not _ver:
        continue
    try:
        _cur = _md.version(_pkg).split("+")[0]
    except Exception:
        continue
    if _cur != _ver:
        print(f"Alignement {_pkg} {_cur} -> {_ver} (torch {_torch_base})")
        subprocess.run(["python3", "-m", "pip", "install", "-q", f"{_pkg}=={_ver}"], check=False)
        _realigned = True
if _realigned:
    print("⚠️ torchaudio/torchvision réalignés : REDÉMARRER le runtime puis ré-exécuter depuis 1.1.")
_sha = subprocess.run(["git", "-C", OMNI_REPO_DIR, "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
_oa_ver = _md.version("omnilingual-asr")
print("omnilingual-asr (installé depuis le clone) :", _oa_ver, "| dépôt @", _sha)
assert _oa_ver != "0.1.0", ("Toujours en 0.1.0 : l'installation depuis le clone a échoué — "
                            "lire le log pip ci-dessus.")
print("fairseq2 :", _md.version("fairseq2"))
def _base_ver(v):
    return str(v).split("+")[0]   # '2.8.0+cu128' et '2.8.0' = même version, notations différentes
for _name, _before in (("torch", _torch_before), ("numpy", _numpy_before)):
    if _before and _base_ver(_md.version(_name)) != _base_ver(_before):
        print(f"⚠️ {_name} a changé ({_before} -> {_md.version(_name)}) : "
              "REDÉMARRER le runtime puis ré-exécuter depuis 1.1.")


In [ ]:
# 1.3 — Imports communs + rapport GPU
import io, json, math, os, re, shutil, subprocess, sys, time, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch

assert torch.cuda.is_available(), "Aucun GPU : Runtime -> Modifier le type d'exécution -> A100."
_gpu = torch.cuda.get_device_properties(0)
GPU_VRAM_GB = _gpu.total_memory / 1e9
print(f"GPU : {_gpu.name} — {GPU_VRAM_GB:.1f} GB VRAM — bf16 : {torch.cuda.is_bf16_supported()}")
if "A100" not in _gpu.name:
    print("⚠️ Pas un A100 : la calibration (section 6) ajustera, mais le débit sera plus faible.")

def now_iso():
    from datetime import datetime, timezone
    return datetime.now(timezone.utc).isoformat()

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    print("json ->", path)

_STEP_RE = re.compile(r"step[_-](\d+)")
def latest_ckpt(root):
    """Dernier RÉPERTOIRE de checkpoint step_N, par NUMÉRO de step (jamais par tri
    lexicographique : 'step_750' > 'step_5000' en tri de chaînes !). Structure fairseq2 0.6 :
    ws_*/checkpoints/step_N/{model/pp_00/tp_00/..., trainer/, optimizer/}. None si aucun."""
    dirs = [d for d in Path(root).rglob("step_*")
            if d.is_dir() and _STEP_RE.search(d.name) and (d / "model").exists()
            and not d.name.endswith(".tmp")]   # .tmp = checkpoint NON finalisé (crash en cours d'écriture)
    return str(max(dirs, key=lambda d: int(_STEP_RE.search(d.name).group(1)))) if dirs else None

def prune_checkpoints(output_dir, keep=2):
    """Ne garder que les `keep` derniers répertoires step_* (un checkpoint 1B + optimiseur
    pèse >10 GB : 20 checkpoints satureraient le Drive). Chercher aussi une option native :
    grep -rn 'keep_last' dans le dépôt (voir 6.1)."""
    dirs = [d for d in Path(output_dir).rglob("step_*") if d.is_dir() and _STEP_RE.search(d.name)]
    dirs = sorted(dirs, key=lambda d: int(_STEP_RE.search(d.name).group(1)))
    for d in dirs[:-keep] if len(dirs) > keep else []:
        shutil.rmtree(d, ignore_errors=True)
        print("checkpoint élagué :", d)


In [ ]:
# 1.4 — Credentials Kaggle (nécessaires en section 10 uniquement)
_KAGGLE_JSON_CANDIDATES = ["/content/afrivoices_project",
                           "/content/afrivoices_project/kaggle.json"]
_kg_dst = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(_kg_dst):
    for cand in _KAGGLE_JSON_CANDIDATES:
        if os.path.exists(cand):
            os.makedirs(os.path.dirname(_kg_dst), exist_ok=True)
            shutil.copyfile(cand, _kg_dst); os.chmod(_kg_dst, 0o600)
            print("kaggle.json installé depuis", cand); break
    else:
        print("⚠️ kaggle.json introuvable sur le Drive — le déposer avant la section 10 "
              "(ou uploader via l'onglet Fichiers vers ~/.kaggle/kaggle.json).")
else:
    print("kaggle.json déjà en place.")


## 2 — Configuration centrale + garde-fous

In [ ]:
# 2.1 — Configuration centrale (UNIQUE endroit à modifier) + rechargement de la calibration
DRIVE_ROOT = "/content/afrivoices_project"
FT_RUN_NAME = "experiment_run"

FT_CONFIG = {
    "run_name": FT_RUN_NAME,
    "seed": 42,
    "model_card": "omniASR_CTC_1B_v2",
    "fallback_model_card": "omniASR_CTC_300M_v2",
    "tokenizer_name": "omniASR_tokenizer_written_v2",  # tokenizer_ref de la carte omniASR_CTC_1B_v2
    "manifest_dir": os.path.join(DRIVE_ROOT, "manifests"),
    "curated_all": os.path.join(DRIVE_ROOT, "manifests", "curated_manifest_all.csv"),
    "parquet_root": "/content/experiment_run",
    # cache Drive du parquet COMPLET (~26 GB) : évite de relire 230k FLAC via FUSE à chaque
    # session (des heures). Mettre à False si le quota Drive ne le permet pas.
    "parquet_drive_cache": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "parquet_cache"),
    "use_parquet_drive_cache": True,
    "experiment_run": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME),
    "output_dir": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "train_output"),
    "fast_output_dir": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "fast_output"),
    "log_dir": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "logs"),
    "report_dir": os.path.join(DRIVE_ROOT, "finetune_runs", FT_RUN_NAME, "reports"),
    "lr": 1e-5,
    "num_steps": 5000,
    "validate_every_n_steps": 250,
    "checkpoint_every_n_steps": 250,
    "checkpoints_keep": 2,   # JAMAIS 1 : après un SIGINT mural le dernier step peut être partiel
    "kaggle_test_filename": "anv-test-data-nt",   # vérifié via l'API (seul fichier, 20 Ko)
    "max_audio_len": 640_000,        # 40 s x 16 kHz (contrainte d'inférence CTC < 40 s)
    "max_num_elements": 5_120_000,
    "grad_accum": 8,
    "freeze_encoder_for_n_steps": 0,
    "mixed_precision": "torch.bfloat16",   # la valeur exacte est relue du YAML officiel en 6.1
    "fast_hours_per_language": 2.0,
    "fast_num_steps": 400,
    "max_wall_hours_per_session": 22.5,
}

LANG_TO_OMNI = {"sw": "swh_Latn", "kik": "kik_Latn", "kln": "kln_Latn",
                "luo": "luo_Latn", "som": "som_Latn", "mas": "mas_Latn"}
LANG_FALLBACK_OMNI = {"mas": "saq_Latn"}   # samburu : langue maa la plus proche couverte
LANG_TO_SUBMISSION = {"sw": "swa", "kik": "kik", "kln": "kln",
                      "luo": "luo", "som": "som", "mas": "mas"}
SUBMISSION_TEXT_COLUMN = "transcription"   # consigne Kaggle ambiguë (prediction/transcription)
LANGUAGES = list(LANG_TO_OMNI)

for _d in ("experiment_run", "output_dir", "fast_output_dir", "log_dir", "report_dir"):
    Path(FT_CONFIG[_d]).mkdir(parents=True, exist_ok=True)

# Les décisions de calibration (section 6) SURVIVENT aux sessions : rechargement systématique.
_calib_path = os.path.join(FT_CONFIG["report_dir"], "calibration_report.json")
if os.path.exists(_calib_path):
    _c = json.load(open(_calib_path))
    if _c.get("chosen"):
        FT_CONFIG["model_card"] = _c["model_card"]
        FT_CONFIG["grad_accum"] = _c["chosen"]["grad_accum"]
        FT_CONFIG["max_num_elements"] = _c["chosen"]["max_num_elements"]
        print(f"Calibration rechargée : {FT_CONFIG['model_card']}, accum={FT_CONFIG['grad_accum']}, "
              f"max_elem={FT_CONFIG['max_num_elements']}")
    else:
        print("⚠️ Calibration précédente NON conclusive : relancer la section 6.")
else:
    print("Pas encore de calibration : section 6 obligatoire avant 7/8.")
save_json(FT_CONFIG, os.path.join(FT_CONFIG["report_dir"], "experiment_run"))


In [ ]:
# 2.2 — Garde-fous : activer UN drapeau à la fois, ICI, puis relancer la cellule.
RUN_CALIBRATION = False     # section 6
RUN_FAST_TRAINING = False   # section 7
RUN_FULL_TRAINING = False   # section 8
RUN_TEST_INFERENCE = True  # section 10

_flags = {k: v for k, v in [("RUN_CALIBRATION", RUN_CALIBRATION),
                            ("RUN_FAST_TRAINING", RUN_FAST_TRAINING),
                            ("RUN_FULL_TRAINING", RUN_FULL_TRAINING),
                            ("RUN_TEST_INFERENCE", RUN_TEST_INFERENCE)]}
_active = [k for k, v in _flags.items() if v]
print("Drapeaux actifs :", _active or "aucun (toutes les phases coûteuses bloquées)")
if len(_active) > 1:
    print("⚠️ Plusieurs drapeaux actifs à la fois : déconseillé, activer les phases une par une.")
import random
random.seed(FT_CONFIG["seed"]); np.random.seed(FT_CONFIG["seed"]); torch.manual_seed(FT_CONFIG["seed"])


In [ ]:
# 2.3 — Couverture des langues par le modèle : décision AUTOMATIQUE du repli maasai
# (mas_Latn est absent des 1672 langues supportées ; maa_Latn = mazatèque, PAS le maasai).
_supported = None
try:
    from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs as _supported
except Exception as exc:
    print(f"Import lang_ids différent dans cette version ({exc!r}) — repli sur le dépôt cloné :")
    try:
        _lang_py = os.path.join(OMNI_REPO_DIR, "src/omnilingual_asr/models/wav2vec2_llama/lang_ids.py")
        _ns = {}
        exec(open(_lang_py, encoding="utf-8").read(), _ns)
        _supported = _ns.get("supported_langs") or next(v for v in _ns.values()
                                                        if isinstance(v, (list, set, tuple)) and len(v) > 1000)
    except Exception as exc2:
        _h = subprocess.run(["grep", "-rn", "-e", "supported_langs", "-e", "mas_Latn",
                             os.path.join(OMNI_REPO_DIR, "src")], capture_output=True, text=True).stdout
        print(_h[:1500])
        raise SystemExit(f"Liste des langues introuvable ({exc2!r}) : adapter d'après le grep ci-dessus.")
    # NB : lang_ids.py vit sous wav2vec2_llama mais c'est l'unique liste du package ;
    # sa validité pour la carte CTC sera confirmée de facto au FAST run.
_supported = set(_supported)
print(f"{len(_supported)} langues supportées par le modèle.")
_use_mas_fallback = "mas_Latn" not in _supported
if _use_mas_fallback:
    assert LANG_FALLBACK_OMNI["mas"] in _supported, \
        f"Le repli {LANG_FALLBACK_OMNI['mas']} n'est pas supporté non plus — choisir un autre code."
    print(f"mas_Latn non supporté (attendu) -> étiquette interne '{LANG_FALLBACK_OMNI['mas']}' "
          "(samburu) pour l'entraînement ET l'inférence. Le code de SOUMISSION reste 'mas'.")
else:
    print("mas_Latn supporté (surprise !) : pas de repli.")
for _l, _code in LANG_TO_OMNI.items():
    if _l != "mas":
        assert _code in _supported, f"{_code} absent des langues supportées ?!"

def omni_lang(lang):
    if lang == "mas" and _use_mas_fallback:
        return LANG_FALLBACK_OMNI["mas"]
    return LANG_TO_OMNI[lang]

# Test d'import de la pipeline d'inférence MAINTENANT (et pas en 7.3 après des heures de GPU) :
# les torchaudio/torchvision préinstallés de Colab, compilés pour un autre torch, cassent ces
# imports (undefined symbol / operator does not exist) tant qu'ils ne sont pas réalignés (1.2).
try:
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline  # noqa: F401
    print("✓ Import ASRInferencePipeline OK — l'inférence sera disponible.")
except Exception as exc:
    raise SystemExit(f"Import de la pipeline d'inférence impossible : {exc!r}. "
                     "Réalignement nécessaire : ré-exécuter 1.2 (elle installe les bonnes versions "
                     "de torchaudio/torchvision), redémarrer le runtime, reprendre depuis 1.1.")

In [ ]:
# 2.4 — Fonctions d'entraînement communes (YAML, run_recipe) — DANS le chemin 1→5,
# pour que le flux session-fraîche « 1→5 puis 8.1→8.2 » soit autosuffisant.
# La valeur exacte de mixed_precision.dtype est relue du YAML OFFICIEL du dépôt cloné.
_official_yaml = os.path.join(OMNI_REPO_DIR, "workflows/recipes/wav2vec2/asr/configs/"
                                             "ctc-finetune-recommendation.yaml")
_dtype_val = FT_CONFIG["mixed_precision"]
if os.path.exists(_official_yaml):
    _m = re.search(r'mixed_precision:\s*\n\s*dtype:\s*"?([A-Za-z0-9_.]+)"?',
                   open(_official_yaml).read())
    if _m:
        _dtype_val = _m.group(1)
        print("dtype officiel relevé :", _dtype_val)
else:
    print(f"⚠️ YAML officiel introuvable ({_official_yaml}) : dtype par défaut '{_dtype_val}' — "
          "vérifier le chemin des configs dans le dépôt cloné.")
# Option native de rétention de checkpoints ? (sinon prune_checkpoints prend le relais)
_keep_hits = subprocess.run(["grep", "-rn", "--include=*.py", "--include=*.yaml", "-i",
                             "keep_last", os.path.join(OMNI_REPO_DIR, "workflows"),
                             os.path.join(OMNI_REPO_DIR, "src")],
                            capture_output=True, text=True).stdout
print("Option de rétention native :", (_keep_hits[:800] or "aucune trouvée -> prune_checkpoints"))

# Carte d'asset du DATASET : comme les modèles/tokenizers, les datasets fairseq2 se
# résolvent par carte — et c'est ELLE qui porte le chemin racine du parquet (dataset_config.data,
# cf. cards/datasets/example_dataset.yaml du dépôt). Écrite dans le dossier cards/ du package
# installé : déjà enregistré comme source d'assets, y compris pour le sous-processus recette.
import omnilingual_asr as _oa
_DATASET_CARD_PATH = os.path.join(os.path.dirname(_oa.__file__), "cards", "datasets",
                                  "afrivoices_dataset.yaml")

def write_dataset_card(data_root=None):
    data_root = data_root or os.path.join(FT_CONFIG["parquet_root"], "version=0")
    os.makedirs(os.path.dirname(_DATASET_CARD_PATH), exist_ok=True)
    with open(_DATASET_CARD_PATH, "w") as f:
        f.write(f"""name: afrivoices_dataset
dataset_family: mixture_parquet_asr_dataset
dataset_config:
  data: {data_root}
tokenizer_ref: {FT_CONFIG["tokenizer_name"]}
""")
    print("carte dataset ->", _DATASET_CARD_PATH, "| data:", data_root)

write_dataset_card()

def write_recipe_yaml(path, model_card, tsv_path, num_steps, grad_accum, max_num_elements,
                      lr=None, ckpt_every=None):
    ck = ckpt_every or FT_CONFIG["checkpoint_every_n_steps"]
    # contrainte fairseq2 : validate/checkpoint_every doivent être multiples de publish_metrics
    # example_shuffle_window: 1000 (le code interdit 0 = shuffle du dataset entier -> OOM ;
    # l'équilibre inter-langues vient déjà des poids hours^beta, la fenêtre ne fait que brasser)
    pub = 50 if ck % 50 == 0 else ck
    yaml_text = f"""model:
  name: "{model_card}"

dataset:
  name: "afrivoices_dataset"
  train_split: "train"
  valid_split: "dev"
  storage_mode: "MIXTURE_PARQUET"
  task_mode: "ASR"
  mixture_parquet_storage_config:
    dataset_summary_path: "{tsv_path}"
    beta_corpus: 0.5
    beta_language: 0.5
    fragment_loading:
      cache: True
  asr_task_config:
    max_audio_len: {FT_CONFIG["max_audio_len"]}
    max_num_elements: {max_num_elements}
    batch_shuffle_window: 1
    normalize_audio: true
    example_shuffle_window: 1000

tokenizer:
  name: "{FT_CONFIG["tokenizer_name"]}"

optimizer:
  config:
    lr: {lr if lr is not None else FT_CONFIG["lr"]}

trainer:
  freeze_encoder_for_n_steps: {FT_CONFIG["freeze_encoder_for_n_steps"]}
  mixed_precision:
    dtype: "{_dtype_val}"
  grad_accumulation:
    num_batches: {grad_accum}

regime:
  num_steps: {num_steps}
  validate_every_n_steps: {ck}
  validate_after_n_steps: {ck}
  checkpoint_every_n_steps: {ck}
  checkpoint_after_n_steps: {ck}
  publish_metrics_every_n_steps: {pub}
  publish_metrics_after_n_steps: {pub}
"""
    with open(path, "w") as f:
        f.write(yaml_text)
    print("yaml ->", path)
    return path

def load_finetuned_pipeline(ckpt_path=None, dtype=torch.bfloat16):
    """Pipeline d'inférence : carte officielle (ckpt_path=None) ou NOTRE checkpoint fine-tuné.
    Voie officielle (signature de ASRInferencePipeline) : pas de checkpoint_path, mais des
    kwargs model=/tokenizer= -> load_model(carte) + load_state_dict(checkpoint) + load_tokenizer."""
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
    if ckpt_path is None:
        return ASRInferencePipeline(FT_CONFIG["model_card"])
    from fairseq2.models.hub import load_model
    from fairseq2.data.tokenizers.hub import load_tokenizer
    model = load_model(FT_CONFIG["model_card"], device=torch.device("cuda"), dtype=dtype)
    _ck = Path(ckpt_path)
    if _ck.is_dir():   # répertoire step_N : les poids sont sous model/pp_00/tp_00/
        _wfiles = sorted(q for q in (_ck / "model").rglob("*") if q.is_file())
        assert _wfiles, f"Aucun fichier de poids sous {_ck}/model"
        if len(_wfiles) > 1:
            print("Fichiers de poids multiples :", [str(q) for q in _wfiles])
            raise SystemExit("Checkpoint shardé (plusieurs fichiers) : me montrer cette liste "
                             "pour adapter le chargement.") from None
        _wfile = _wfiles[0]
    else:
        _wfile = _ck
    print("Poids :", _wfile)
    if str(_wfile).endswith(".safetensors"):
        from safetensors.torch import load_file as _st_load
        state = _st_load(str(_wfile))
    else:
        state = torch.load(str(_wfile), map_location="cpu", weights_only=False)
    sd = state.get("model", state) if isinstance(state, dict) else state
    try:
        model.load_state_dict(sd)
    except Exception:
        try:   # certains checkpoints préfixent les clés (module., model., _orig_mod.)
            _strip = lambda k: k.split("module.", 1)[-1].split("_orig_mod.", 1)[-1]
            model.load_state_dict({_strip(k): v for k, v in sd.items()})
            print("state_dict chargé après retrait des préfixes de clés.")
        except Exception as exc:
            _keys = list(sd.keys())[:12] if hasattr(sd, "keys") else type(sd)
            print("Clés du checkpoint (extrait) :", _keys)
            raise SystemExit(f"Format de state_dict inattendu ({exc!r}) : me montrer les clés "
                             "affichées ci-dessus.") from None
    model.eval()
    tok = load_tokenizer(FT_CONFIG["model_card"])
    return ASRInferencePipeline(None, model=model, tokenizer=tok)

def run_recipe(output_dir, yaml_path, log_name, wall_limit_h=None, prune_keep=None):
    """Sous-processus + stream des logs + coupure murale PROPRE (SIGINT puis drain du pipe
    jusqu'à EOF — sinon le fils bloque sur un pipe plein et survit en zombie GPU).
    Relancer avec le même output_dir reprend au dernier checkpoint PÉRIODIQUE."""
    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(FT_CONFIG["log_dir"], log_name)
    cmd = [sys.executable, "-m", "workflows.recipes.wav2vec2.asr", output_dir,
           "--config-file", yaml_path]
    print("CMD :", " ".join(cmd))
    t0 = time.time(); interrupted = False; _sigint_t = None
    with open(log_path, "a", encoding="utf-8") as logf:
        proc = subprocess.Popen(cmd, cwd=OMNI_REPO_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            logf.write(line)
            if any(k in line.lower() for k in ("loss", "error", "step", "oom", "memory", "cuda")):
                print(line.rstrip()[:200])
            # élagage EN COURS de run : sans lui, jusqu'à 20 checkpoints >10 GB s'accumulent
            # sur Drive avant la fin (saturation de quota en plein entraînement)
            if prune_keep and "checkpoint" in line.lower():
                try:
                    prune_checkpoints(output_dir, keep=prune_keep)
                except Exception as _pexc:
                    print("prune en cours de run impossible :", repr(_pexc)[:120])
            if wall_limit_h and not interrupted and (time.time() - t0) > wall_limit_h * 3600:
                print(f"⏱️ Limite murale {wall_limit_h}h : SIGINT (reprise au dernier checkpoint "
                      f"périodique, <= {FT_CONFIG['checkpoint_every_n_steps']} steps perdus).")
                proc.send_signal(2); _sigint_t = time.time()
                interrupted = True   # on CONTINUE de drainer le pipe jusqu'à EOF
            if interrupted and _sigint_t and (time.time() - _sigint_t) > 900:
                print("SIGINT ignoré depuis 15 min : kill.")
                proc.kill()
                _sigint_t = None
        try:
            proc.wait(timeout=900)
        except subprocess.TimeoutExpired:
            print("Le processus ne se termine pas : kill.")
            proc.kill(); proc.wait()
    print(f"Recette : code {proc.returncode} en {(time.time()-t0)/60:.1f} min ; log : {log_path}")
    return proc.returncode, log_path


## 3 — Chargement et contrôle des manifests curés

In [ ]:
# 3.1 — Manifest curé + contrôles
manifest = pd.read_csv(FT_CONFIG["curated_all"], low_memory=False)
required_cols = {"sample_id", "split", "language", "curated_audio_path", "text", "duration"}
missing = required_cols - set(manifest.columns)
assert not missing, f"Colonnes manquantes : {missing}"
manifest = manifest[manifest["language"].isin(LANGUAGES)].reset_index(drop=True)
assert set(manifest["language"].unique()) == set(LANGUAGES), \
    f"Langues du manifest != attendues : {sorted(manifest['language'].unique())} vs {LANGUAGES}"
_hours_tot = manifest["duration"].sum() / 3600
assert 300 < _hours_tot < 600, f"Volume anormal : {_hours_tot:.0f} h (attendu ~456 h)"
print(f"{len(manifest)} lignes")
print(manifest.groupby(["split", "language"])["duration"].sum().div(3600).round(2).unstack())

# Contrôle EXHAUSTIF via listing par répertoire (18 listings, rapide — jamais 230k exists FUSE).
# La validation du Prompt 1 n'échantillonnait que 600 fichiers : des écritures Drive perdues
# en fin de session ont pu passer inaperçues.
BUDGET_H = {"train": 60.0, "valid": 10.0, "local_test": 6.0}
_missing_frames = []
for (_s, _l), grp in manifest.groupby(["split", "language"]):
    _d = os.path.dirname(grp["curated_audio_path"].iloc[0])
    _have = set(os.listdir(_d)) if os.path.isdir(_d) else set()
    _miss = grp[~grp["curated_audio_path"].map(os.path.basename).isin(_have)]
    if len(_miss):
        _missing_frames.append(_miss)
    _kept_h = (grp["duration"].sum() - _miss["duration"].sum()) / 3600
    print(f"  {_s}/{_l}: {len(_miss)}/{len(grp)} manquants ; restant {_kept_h:.2f}h "
          f"({_kept_h / BUDGET_H[_s]:.1%} du budget {BUDGET_H[_s]:.0f}h)")
missing_df = pd.concat(_missing_frames, ignore_index=True) if _missing_frames else manifest.iloc[0:0]
print(f"TOTAL fichiers manquants : {len(missing_df)}")
if len(missing_df):
    missing_df.to_csv(os.path.join(FT_CONFIG["report_dir"], "missing_audio_files.csv"), index=False)
    _ok_after = True
    for (_s, _l), grp in manifest.groupby(["split", "language"]):
        _m = missing_df[(missing_df["split"] == _s) & (missing_df["language"] == _l)]
        if (grp["duration"].sum() - _m["duration"].sum()) / 3600 < 0.95 * BUDGET_H[_s]:
            _ok_after = False
            print(f"⚠️ {_s}/{_l} passerait SOUS 95% du budget sans ces fichiers.")
    if not _ok_after:
        raise SystemExit("Trop de fichiers manquants : relancer la cellule 13.4 du notebook de "
                         "curation (sa reprise ne rematérialisera QUE les fichiers manquants), "
                         "puis revenir ici. Liste : reports/missing_audio_files.csv")
    manifest = manifest[~manifest["sample_id"].isin(set(missing_df["sample_id"]))].reset_index(drop=True)
    print(f"Fichiers manquants exclus du manifest (budgets ≥95% préservés) : "
          f"{len(manifest)} lignes restantes.")
else:
    print("✓ Tous les fichiers du manifest sont présents sur le Drive.")


## 4 — Normalisation du texte + audit du vocabulaire CTC

In [ ]:
# 4.1 — Normalisation (minuscules, ponctuation retirée, diacritiques CONSERVÉS)
_PUNCT_RE = re.compile(r"[^\w\s'’\-]", re.UNICODE)
_WS_RE = re.compile(r"\s+")

def normalize_text(t: str) -> str:
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = _PUNCT_RE.sub(" ", t)
    t = t.replace("_", " ")
    return _WS_RE.sub(" ", t).strip()

manifest["text_norm"] = manifest["text"].map(normalize_text)
_empty = int((manifest["text_norm"].str.len() == 0).sum())
manifest = manifest[manifest["text_norm"].str.len() > 0].reset_index(drop=True)
print(f"Textes vides exclus : {_empty}")

# Chiffres : la dataprep officielle les retire du vocabulaire -> intranscriptibles.
_dig = manifest["text_norm"].str.contains(r"\d")
print("Lignes avec chiffres par split :", manifest[_dig].groupby("split").size().to_dict() or 0)
# Exclusion du TRAIN par langue, sous garde du plancher 95% du budget (les chiffres sont
# intranscriptibles par le vocabulaire du modèle : ils n'apportent que du bruit CTC).
_drop_ids = []
for _lang, _grp in manifest[manifest["split"] == "train"].groupby("language"):
    _dg = _grp[_grp["text_norm"].str.contains(r"\d")]
    _kept_h = (_grp["duration"].sum() - _dg["duration"].sum()) / 3600
    if len(_dg) and _kept_h >= 0.95 * BUDGET_H["train"]:
        _drop_ids.extend(_dg["sample_id"].tolist())
        print(f"  {_lang}: {len(_dg)} lignes chiffrées exclues du train ; reste {_kept_h:.2f}h "
              f"({_kept_h / BUDGET_H['train']:.1%})")
    elif len(_dg):
        print(f"  ⚠️ {_lang}: exclusion impossible sans passer sous 95% du budget -> conservées ({len(_dg)})")
if _drop_ids:
    manifest = manifest[~manifest["sample_id"].isin(set(_drop_ids))].reset_index(drop=True)
    print(f"Total exclu du train : {len(_drop_ids)} lignes ; manifest : {len(manifest)} lignes.")
print("NB : les réfs valid/local_test chiffrées gonfleront mécaniquement le CER (chiffres "
      "intranscriptibles par le modèle) — comptes ci-dessus pour interpréter les scores.")

In [ ]:
# 4.2 — Audit : caractères du corpus vs vocabulaire CTC (via l'ENCODEUR officiel)
import glob
import omnilingual_asr
_pkg_dir = os.path.dirname(omnilingual_asr.__file__)
_cards = sorted(os.path.basename(c) for c in
                glob.glob(os.path.join(_pkg_dir, "cards", "**", "*.yaml"), recursive=True))
print("omnilingual_asr", getattr(omnilingual_asr, "__version__", "?"), "| cartes packagées :", _cards)
assert any("v2" in c for c in _cards), \
    "Aucune carte v2 dans le package installé : relancer 1.2 (omnilingual-asr==0.2.0) + redémarrer."
from fairseq2.data.tokenizers.hub import load_tokenizer
_tok = None
for _cand in (FT_CONFIG["tokenizer_name"], FT_CONFIG["model_card"]):
    try:
        _tok = load_tokenizer(_cand)
        print("Tokenizer chargé via la carte :", _cand,
              "| taille vocab :", getattr(getattr(_tok, "vocab_info", None), "size", "?"))
        break
    except Exception as exc:
        print(f"  {_cand} -> {type(exc).__name__}")
if _tok is None:
    raise SystemExit("Cartes v2 présentes mais tokenizer non résolu : l'extension fairseq2 du "
                     "package ne se charge pas — vérifier les versions fairseq2/omnilingual-asr "
                     "(1.2) et me montrer cette sortie.") from None

try:
    _enc = _tok.create_encoder()
except TypeError as exc:
    import inspect as _insp
    print("Signature :", _insp.signature(_tok.create_encoder))
    raise SystemExit(f"create_encoder exige des arguments ({exc!r}) : adapter d'après la "
                     "signature affichée, puis relancer.") from None
_unk = getattr(_tok.vocab_info, "unk_idx", None)

def _char_is_oov(c):
    try:
        ids = _enc(c)
        ids = ids.tolist() if hasattr(ids, "tolist") else list(ids)
    except Exception:
        return True
    return (not ids) or ((_unk is not None) and (_unk in ids))

_oov_report = {}
for lang, grp in manifest.groupby("language"):
    chars = sorted(set("".join(grp["text_norm"])) - {" "})
    oov = [c for c in chars if _char_is_oov(c)]
    _oov_report[lang] = oov
    print(f"  {lang}: {len(chars)} caractères distincts, {len(oov)} hors vocabulaire {oov[:15]}")
save_json(_oov_report, os.path.join(FT_CONFIG["report_dir"], "vocab_oov_report.json"))
if sum(len(v) for v in _oov_report.values()):
    print("⚠️ OOV détectés : compléter OOV_CHAR_MAP en 4.3 avant conversion.")
else:
    print("✓ Toutes les langues couvertes par le vocabulaire CTC.")

In [ ]:
# 4.3 — Mapping des caractères OOV (compléter selon vocab_oov_report.json ; no-op sinon)
OOV_CHAR_MAP = {}   # ex. {"ŋ": "ng'"}
if OOV_CHAR_MAP:
    _tr = str.maketrans(OOV_CHAR_MAP)
    manifest["text_norm"] = manifest["text_norm"].map(lambda t: t.translate(_tr))
    print("Mapping OOV appliqué :", OOV_CHAR_MAP)
else:
    print("Aucun mapping OOV appliqué.")


## 5 — Conversion au format parquet fairseq2 (MIXTURE_PARQUET)
`version=0/corpus=afrivoices/split={train,dev}/language=xxx_Latn/part-*.parquet` — colonnes
`text`, `audio_bytes` (FLAC brut), `audio_size` (échantillons exacts), `corpus`, `split`, `language`.

In [ ]:
# 5.1 — DÉCOUVERTE : format exact du TSV `dataset_summary_path` (non documenté publiquement).
# On lit le code du dépôt qui le PARSE — puis on valide en 5.2 (verrou TSV_FORMAT_CONFIRMED).
_hits = subprocess.run(["grep", "-rn", "--include=*.py", "-e", "dataset_summary_path",
                        "-e", "language_distribution",
                        os.path.join(OMNI_REPO_DIR, "src"), os.path.join(OMNI_REPO_DIR, "workflows")],
                       capture_output=True, text=True).stdout
print(_hits[:4000] if _hits else "Aucune occurrence : chercher à la main dans le dépôt.")
print("\n-> Ouvrir le(s) fichier(s) ci-dessus, noter les COLONNES attendues du TSV, ajuster")
print("   build_parquet en 5.2 si besoin, puis mettre TSV_FORMAT_CONFIRMED = True.")


In [ ]:
# 5.2 — Conversion manifests -> parquet + TSV (verrouillée tant que le format TSV n'est pas confirmé)
import pyarrow as pa
import pyarrow.parquet as pq

# Format VÉRIFIÉ dans le code source du dépôt (mixture_parquet_storage.py, l.339-351 :
# pd.read_csv(sep="\t") puis colonnes utilisées = corpus, language, hours ; et le générateur
# officiel hf_dataset_ingestion_example.py écrit exactement group_by(corpus, language)
# .agg(hours = sum(audio_size)/3600/16000)). Poids d'échantillonnage ∝ hours^beta.
TSV_FORMAT_CONFIRMED = True

PARQUET_ROOT = os.path.join(FT_CONFIG["parquet_root"], "version=0")
CORPUS = "afrivoices"
SPLIT_MAP = {"train": "train", "valid": "dev"}

def _prep_hash():
    import hashlib
    _SCHEMA_VERSION = 2   # v2 : colonnes de partition retirées des fichiers parquet
    sig = f"schema{_SCHEMA_VERSION}|{len(manifest)}|{int(manifest['text_norm'].str.len().sum())}|" \
          f"{json.dumps(globals().get('OOV_CHAR_MAP', {}), sort_keys=True)}"
    return hashlib.md5(sig.encode()).hexdigest()[:12]

def dataset_meta(root=None):
    _p = os.path.join(os.path.dirname(root or PARQUET_ROOT + "/"), "dataset_meta.json")
    _p = os.path.join(FT_CONFIG["parquet_root"], "dataset_meta.json") if root is None else _p
    return json.load(open(_p)) if os.path.exists(_p) else None

def build_parquet(df, max_rows_per_file=2000, subset_hours_per_lang=None, root=PARQUET_ROOT):
    if not TSV_FORMAT_CONFIRMED:
        raise SystemExit("Format du TSV non confirmé : faire 5.1, ajuster si besoin, "
                         "puis TSV_FORMAT_CONFIRMED = True (même verrou que les autres découvertes).")
    t0 = time.time(); n_files = 0; summary_rows = []
    for (split, lang), grp in df[df["split"].isin(SPLIT_MAP)].groupby(["split", "language"]):
        if subset_hours_per_lang:
            grp = grp.sample(frac=1.0, random_state=FT_CONFIG["seed"])
            keep, acc = [], 0.0
            for _, r in grp.iterrows():
                if acc >= subset_hours_per_lang * 3600: break
                keep.append(r); acc += float(r["duration"])
            grp = pd.DataFrame(keep)
        out_dir = os.path.join(root, f"corpus={CORPUS}", f"split={SPLIT_MAP[split]}",
                               f"language={omni_lang(lang)}")
        os.makedirs(out_dir, exist_ok=True)
        from concurrent.futures import ThreadPoolExecutor
        def _read_bytes(pth):
            with open(pth, "rb") as fh:
                return fh.read()
        with ThreadPoolExecutor(max_workers=32) as _pool:   # FUSE : latence/fichier -> paralléliser
            _blobs = list(_pool.map(_read_bytes, grp["curated_audio_path"].tolist()))
        rows, part, total_sz = [], 0, 0
        for (_, r), blob in zip(grp.iterrows(), _blobs):
            frames = int(sf.info(io.BytesIO(blob)).frames)   # exact, zéro I/O supplémentaire
            total_sz += frames
            # corpus/split/language UNIQUEMENT dans les chemins hive (le lecteur officiel les
            # infère du partitionnement ; en colonne aussi -> ArrowTypeError: Unable to merge)
            rows.append({"text": r["text_norm"], "audio_bytes": blob, "audio_size": frames})
            if len(rows) >= max_rows_per_file:
                pq.write_table(pa.Table.from_pylist(rows),
                               os.path.join(out_dir, f"part-{part:04d}.parquet"), row_group_size=100)
                part += 1; n_files += 1; rows = []
        if rows:
            pq.write_table(pa.Table.from_pylist(rows),
                           os.path.join(out_dir, f"part-{part:04d}.parquet"), row_group_size=100)
            n_files += 1
        summary_rows.append({"corpus": CORPUS, "split": SPLIT_MAP[split],
                             "language": omni_lang(lang), "num_examples": int(len(grp)),
                             "total_audio_size": int(total_sz)})
        print(f"  {split}/{lang} -> {omni_lang(lang)} : {len(grp)} clips")
    # TSV officiel : EXACTEMENT 3 colonnes (corpus, language, hours), agrégées SANS split
    _summary = pd.DataFrame(summary_rows)
    _tsv_df = (_summary.groupby(["corpus", "language"], as_index=False)["total_audio_size"].sum()
               .assign(hours=lambda d: d["total_audio_size"] / 16000 / 3600)
               [["corpus", "language", "hours"]])
    tsv_path = os.path.join(os.path.dirname(root), "language_distribution_0.tsv")
    _tsv_df.to_csv(tsv_path, sep="\t", index=False)
    print(_tsv_df.round(2).to_string(index=False))
    # méta : QUEL dataset est présent (empêche d'entraîner le run complet sur un sous-ensemble
    # laissé par la calibration/le FAST — même chemin de TSV) + hash de préparation (invalidation)
    save_json({"subset_hours_per_lang": subset_hours_per_lang, "prep_hash": _prep_hash(),
               "rows": int(sum(r["num_examples"] for r in summary_rows)), "timestamp": now_iso()},
              os.path.join(os.path.dirname(root), "dataset_meta.json"))
    print(f"{n_files} parquet en {time.time()-t0:.0f}s ; résumé -> {tsv_path}")
    return tsv_path

def restore_or_build_full_parquet():
    """Parquet COMPLET : restauré depuis le cache Drive si présent (rapide), sinon construit
    depuis les FLAC (lent : ~230k petits fichiers via FUSE) puis mis en cache."""
    cache = FT_CONFIG["parquet_drive_cache"]
    local = FT_CONFIG["parquet_root"]
    tsv_local = os.path.join(local, "language_distribution_0.tsv")
    _marker = os.path.join(cache, "_COMPLETE")          # écrit EN DERNIER : cache partiel = invalide
    _meta_c = os.path.join(cache, "dataset_meta.json")
    if FT_CONFIG["use_parquet_drive_cache"] and os.path.exists(_marker) and os.path.exists(_meta_c):
        _m = json.load(open(_meta_c))
        if _m.get("prep_hash") == _prep_hash() and _m.get("subset_hours_per_lang") is None:
            print("Cache parquet Drive valide : copie locale (1 flux, >> plus rapide que 230k FLAC)…")
            shutil.rmtree(local, ignore_errors=True)
            subprocess.run(["cp", "-r", cache, local], check=True)
            return tsv_local
        print(f"Cache présent mais périmé (hash {_m.get('prep_hash')} != {_prep_hash()} "
              "ou sous-ensemble) : reconstruction.")
    shutil.rmtree(local, ignore_errors=True)
    tsv = build_parquet(manifest)
    if FT_CONFIG["use_parquet_drive_cache"]:
        print("Mise en cache du parquet sur Drive (~26 GB — vérifier le quota)…")
        shutil.rmtree(cache, ignore_errors=True)
        subprocess.run(["cp", "-r", local, cache], check=True)
        open(_marker, "w").write(now_iso())            # marqueur de complétude, en tout dernier
    return tsv

print("Conversion prête (exécution réelle dans les sections 6/7/8).")


## 6 — Calibration VRAM/débit (obligatoire avant tout entraînement)

In [ ]:
# 6.1 — Calibration : mini-dataset (0.25 h/langue), 30 steps, échelle grad-accum
if not RUN_CALIBRATION:
    raise SystemExit("Calibration bloquée. Mettre RUN_CALIBRATION=True en 2.2, relancer 2.2, puis ici.")
_meta = dataset_meta()
if _meta and _meta.get("subset_hours_per_lang") == 0.25 and _meta.get("prep_hash") == _prep_hash():
    _tsv = os.path.join(FT_CONFIG["parquet_root"], "language_distribution_0.tsv")
    print("Mini-parquet de calibration déjà présent et valide : réutilisé.")
else:
    shutil.rmtree(FT_CONFIG["parquet_root"], ignore_errors=True)
    _tsv = build_parquet(manifest, subset_hours_per_lang=0.25)

_ladder = [
    {"grad_accum": 8,  "max_num_elements": FT_CONFIG["max_num_elements"]},
    {"grad_accum": 16, "max_num_elements": FT_CONFIG["max_num_elements"] // 2},
    {"grad_accum": 32, "max_num_elements": FT_CONFIG["max_num_elements"] // 4},
]
calibration = {"timestamp": now_iso(), "gpu": _gpu.name, "attempts": [], "chosen": None,
               "model_card": FT_CONFIG["model_card"]}
for att in _ladder:
    _out = f"/content/calib_{att['grad_accum']}"
    shutil.rmtree(_out, ignore_errors=True)
    _yaml = write_recipe_yaml(f"/content/calib_{att['grad_accum']}.yaml", FT_CONFIG["model_card"],
                              _tsv, num_steps=30, ckpt_every=30, **att)
    rc, log_path = run_recipe(_out, _yaml, f"calib_accum{att['grad_accum']}.log")
    _tail = open(log_path, encoding="utf-8").read()[-5000:]
    _oom = ("CUDA out of memory" in _tail) or ("OutOfMemoryError" in _tail)
    calibration["attempts"].append({**att, "returncode": rc, "oom": bool(_oom)})
    print("Essai :", calibration["attempts"][-1])
    if rc == 0 and not _oom:
        calibration["chosen"] = att
        break
    if rc != 0 and not _oom:
        # erreur de config/recette : descendre l'échelle d'accumulation n'y changera RIEN
        save_json(calibration, os.path.join(FT_CONFIG["report_dir"], "calibration_report.json"))
        raise SystemExit(f"Échec de RECETTE (pas un OOM) à grad_accum={att['grad_accum']} : "
                         f"lire {log_path} (dernières lignes ci-dessus) et me les montrer. "
                         "Ne PAS conclure sur la VRAM.") from None
save_json(calibration, os.path.join(FT_CONFIG["report_dir"], "calibration_report.json"))
if calibration["chosen"] is None:
    # Aucun réglage 1B ne tient : repli officiel = checkpoint plus petit. chosen reste None
    # tant que le 300M n'a pas été RÉELLEMENT calibré (pas de fausse validation).
    raise SystemExit(f"Le {FT_CONFIG['model_card']} ne tient pas en VRAM. "
                     f"Mettre FT_CONFIG['model_card']='{FT_CONFIG['fallback_model_card']}' en 2.1 "
                     "(défaut) et RELANCER 2.1 puis cette cellule pour calibrer le 300M.")
FT_CONFIG["grad_accum"] = calibration["chosen"]["grad_accum"]
FT_CONFIG["max_num_elements"] = calibration["chosen"]["max_num_elements"]
save_json(FT_CONFIG, os.path.join(FT_CONFIG["report_dir"], "experiment_run"))
print("CHOIX :", FT_CONFIG["model_card"], calibration["chosen"])


## 7 — FAST run : valider TOUTE la boucle (train → checkpoint → reprise → inférence → CER)

In [ ]:
# 7.1 — FAST training (2 h/langue, ~400 steps)
if not RUN_FAST_TRAINING:
    raise SystemExit("FAST bloqué. RUN_FAST_TRAINING=True en 2.2 puis relancer 2.2 et ici.")
shutil.rmtree(FT_CONFIG["parquet_root"], ignore_errors=True)
_fast_tsv = build_parquet(manifest, subset_hours_per_lang=FT_CONFIG["fast_hours_per_language"])
_fast_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], "fast_recipe.yaml"),
                               FT_CONFIG["model_card"], _fast_tsv,
                               num_steps=FT_CONFIG["fast_num_steps"], ckpt_every=100,
                               grad_accum=FT_CONFIG["grad_accum"],
                               max_num_elements=FT_CONFIG["max_num_elements"])
rc, _log = run_recipe(FT_CONFIG["fast_output_dir"], _fast_yaml, "fast_training.log",
                      prune_keep=FT_CONFIG["checkpoints_keep"])
assert rc == 0, f"FAST training en échec (code {rc}) — lire {_log} AVANT toute suite."
prune_checkpoints(FT_CONFIG["fast_output_dir"], keep=FT_CONFIG["checkpoints_keep"])
print("Dernier checkpoint FAST :", latest_ckpt(FT_CONFIG["fast_output_dir"]))


In [ ]:
# 7.2 — Test de REPRISE : relancer la même commande doit repartir du checkpoint (pas de 0)
if not RUN_FAST_TRAINING:
    raise SystemExit("FAST bloqué.")
_fast_tsv2 = os.path.join(FT_CONFIG["parquet_root"], "language_distribution_0.tsv")
_meta = dataset_meta()
assert os.path.exists(_fast_tsv2) and _meta, "Parquet absent : exécuter 7.1 d'abord dans cette session."
assert _meta["subset_hours_per_lang"] == FT_CONFIG["fast_hours_per_language"], \
    f"Le parquet présent n'est pas le sous-ensemble FAST ({_meta}) : relancer 7.1."
_fast_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], "fast_recipe.yaml"),
                               FT_CONFIG["model_card"], _fast_tsv2,
                               num_steps=FT_CONFIG["fast_num_steps"], ckpt_every=100,
                               grad_accum=FT_CONFIG["grad_accum"],
                               max_num_elements=FT_CONFIG["max_num_elements"])   # idempotent
rc2, _log2 = run_recipe(FT_CONFIG["fast_output_dir"], _fast_yaml, "fast_training_resume.log")
print("Code retour reprise :", rc2)
print("⚠️ VÉRIFIER dans le log que la reprise part du step sauvegardé "
      "(chercher 'restor'/'resum'/numéro de step) — garantie anti-24h du run complet.")


In [ ]:
# 7.3 — Inférence du checkpoint FAST + CER par langue (mini-échantillon valid)
if not RUN_FAST_TRAINING:
    raise SystemExit("FAST bloqué.")
import jiwer

_experiment_run = latest_ckpt(FT_CONFIG["fast_output_dir"])
assert _experiment_run is not None, ("Aucun checkpoint 'model*step_*.pt' sous fast_output_dir — "
                              "inspecter la structure réelle (ls -R) et adapter latest_ckpt.")
print("Checkpoint :", _experiment_run)
pipe_ft = load_finetuned_pipeline(_experiment_run)   # voie officielle model=/tokenizer= (cf. 2.4)

_sample = (manifest[manifest["split"] == "valid"]
           .sample(frac=1.0, random_state=FT_CONFIG["seed"])
           .groupby("language").head(20))              # pas de groupby.apply (déprécié pandas>=2.2)
_cer = {}
for lang, grp in _sample.groupby("language"):
    files, refs = grp["curated_audio_path"].tolist(), grp["text_norm"].tolist()
    hyps = [normalize_text(h) for h in
            pipe_ft.transcribe(files, lang=[omni_lang(lang)] * len(files), batch_size=8)]
    _cer[lang] = round(jiwer.cer(refs, hyps), 4)
    save_json({"timestamp": now_iso(), "cer_fast": _cer},   # sauvegarde INCRÉMENTALE par langue
              os.path.join(FT_CONFIG["report_dir"], "fast_eval_report.json"))
    print(f"  {lang}: CER={_cer[lang]} | réf «{refs[0][:60]}» / hyp «{hyps[0][:60]}»")
print("GO section 8 si : reprise OK (7.2) + CER plausibles (pas de charabia).")


## 8 — Entraînement complet (multi-sessions ; relancer 8.1→8.2 à chaque session)

In [ ]:
# 8.1 — Restaurer (cache Drive) ou construire le parquet complet
if not RUN_FULL_TRAINING:
    raise SystemExit("Run complet bloqué. RUN_FULL_TRAINING=True en 2.2 puis relancer 2.2 et ici.")
_full_tsv = restore_or_build_full_parquet()


In [ ]:
# 8.2 — Lancer / reprendre (autonome : régénère son YAML, indépendant de 8.1 pour les variables)
if not RUN_FULL_TRAINING:
    raise SystemExit("Run complet bloqué.")
_full_tsv = os.path.join(FT_CONFIG["parquet_root"], "language_distribution_0.tsv")
_meta = dataset_meta()
assert os.path.exists(_full_tsv) and _meta, "Parquet absent : exécuter 8.1 d'abord dans cette session."
assert _meta["subset_hours_per_lang"] is None, \
    (f"Le parquet présent est un SOUS-ENSEMBLE ({_meta['subset_hours_per_lang']} h/langue, laissé "
     "par la calibration ou le FAST) : exécuter 8.1 pour restaurer/construire le dataset complet.")
_full_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], "full_recipe.yaml"),
                               FT_CONFIG["model_card"], _full_tsv,
                               num_steps=FT_CONFIG["num_steps"],
                               grad_accum=FT_CONFIG["grad_accum"],
                               max_num_elements=FT_CONFIG["max_num_elements"])
rc, _log = run_recipe(FT_CONFIG["output_dir"], _full_yaml, "full_training.log",
                      wall_limit_h=FT_CONFIG["max_wall_hours_per_session"],
                      prune_keep=FT_CONFIG["checkpoints_keep"])
prune_checkpoints(FT_CONFIG["output_dir"], keep=FT_CONFIG["checkpoints_keep"])
print("Code :", rc, "| dernier checkpoint :", latest_ckpt(FT_CONFIG["output_dir"]))
print("Si coupé par la limite murale : nouvelle session -> 1→5 -> RUN_FULL_TRAINING=True -> 8.1 -> 8.2.")


## 9 — Évaluation finale : zéro-shot vs fine-tuné sur local_test

In [ ]:
# 9.1 — CER/WER par langue, zéro-shot vs fine-tuné — REPRENABLE (sessions Colab fragiles) :
# les couples (modèle, langue) déjà présents dans final_eval_report.json sont sautés, et un
# modèle dont la passe est complète n'est même pas chargé.
import jiwer

EVAL_CLIPS_PER_LANG = 250   # None = tout local_test (plus long)
_eval = (manifest[manifest["split"] == "local_test"]
         .sample(frac=1.0, random_state=FT_CONFIG["seed"])
         .groupby("language").head(EVAL_CLIPS_PER_LANG or 10**9))

_experiment_run = latest_ckpt(FT_CONFIG["output_dir"])
assert _experiment_run is not None, "Aucun checkpoint du run complet : section 8 d'abord."
print("Checkpoint évalué :", _experiment_run)

_report_path = os.path.join(FT_CONFIG["report_dir"], "final_eval_report.json")
results = {}
if os.path.exists(_report_path):
    try:
        results = json.load(open(_report_path)).get("results", {})
        print("Reprise éval — déjà calculé :", {k: sorted(v) for k, v in results.items()})
    except Exception as _exc:
        print("Rapport précédent illisible :", repr(_exc))

_pipe_factories = {"zeroshot": lambda: load_finetuned_pipeline(None),
                   "finetuned": lambda: load_finetuned_pipeline(_experiment_run)}
for name, factory in _pipe_factories.items():
    results.setdefault(name, {})
    _todo = [l for l in LANGUAGES if l not in results[name]]
    if not _todo:
        print(f"[{name}] déjà complet, modèle non chargé.")
        continue
    pipe = factory()
    for lang, grp in _eval.groupby("language"):
        if lang not in _todo:
            continue
        files, refs = grp["curated_audio_path"].tolist(), grp["text_norm"].tolist()
        hyps = [normalize_text(h) for h in
                pipe.transcribe(files, lang=[omni_lang(lang)] * len(files), batch_size=16)]
        results[name][lang] = {"cer": round(jiwer.cer(refs, hyps), 4),
                               "wer": round(jiwer.wer(refs, hyps), 4), "n": len(refs)}
        save_json({"timestamp": now_iso(), "results": results}, _report_path)  # incrémental
        print(f"[{name}] {lang}: CER={results[name][lang]['cer']} WER={results[name][lang]['wer']}")
    del pipe
    torch.cuda.empty_cache()

_tbl = pd.DataFrame({(n, m): {l: results[n][l][m] for l in results[n]}
                     for n in results for m in ("cer", "wer")})
print(_tbl)
_reg = [l for l in results.get("finetuned", {})
        if results["finetuned"][l]["cer"] > results["zeroshot"][l]["cer"]]
print(("⚠️ Sans amélioration : " + str(_reg) + " -> envisager le top-up 9.2 (surtout mas).")
      if _reg else "✓ Amélioration sur toutes les langues : GO soumission.")

In [ ]:
# 9.2 — (OPTIONNEL) Top-up mono-langue — VERROUILLÉ : partir du checkpoint JOINT n'est pas
# encore câblé (lancer la recette sur un output_dir vierge repartirait des poids de BASE,
# ce qui serait un top-up trompeur). Mécanisme à implémenter : copier le dernier checkpoint
# du run joint dans le nouvel output_dir selon la structure observée en 7.1, ou utiliser
# l'option de seed de la recette si elle existe (grep 'resume\|init_from' dans workflows/).
TOPUP_LANG = None   # ex. "mas" — mais lire le verrou ci-dessous d'abord
if TOPUP_LANG:
    raise SystemExit("Top-up non câblé : le seed depuis le checkpoint joint doit être implémenté "
                     "d'abord (voir commentaire). Me montrer la structure de train_output/ et "
                     "la sortie du grep pour que je l'écrive.")
print("Pas de top-up demandé.")


## 10 — Test Kaggle → `submission.csv`

In [ ]:
# 10.1 — Récupérer et INSPECTER le fichier test.
# Priorité 1 : fichier déposé MANUELLEMENT (l'API Kaggle peut renvoyer une page reCAPTCHA au
# lieu du fichier — anti-robot). Télécharger alors depuis kaggle.com (onglet Data) dans un
# navigateur, et le déposer dans DRIVE_ROOT/kaggle_test/.
if not RUN_TEST_INFERENCE:
    raise SystemExit("Inférence test bloquée. RUN_TEST_INFERENCE=True en 2.2 puis relancer 2.2 et ici.")
KAGGLE_TEST_DIR = "/content/kaggle_test"
os.makedirs(KAGGLE_TEST_DIR, exist_ok=True)
_manual_dir = os.path.join(DRIVE_ROOT, "kaggle_test")

def _looks_like_captcha(path):
    with open(path, "rb") as fh:
        head = fh.read(400).lower()
    return (b"<!doctype html" in head or b"<html" in head) and (b"recaptcha" in head or b"captcha" in head)

_found = None
if os.path.isdir(_manual_dir):
    for f in sorted(os.listdir(_manual_dir)):
        if f.startswith(FT_CONFIG["kaggle_test_filename"]):
            _found = os.path.join(KAGGLE_TEST_DIR, f)
            shutil.copyfile(os.path.join(_manual_dir, f), _found)
            print("Fichier test repris depuis le dépôt MANUEL :", _found)
            break
if _found is None:
    assert os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")), "kaggle.json absent : voir 1.4."
    rc = subprocess.run(["kaggle", "competitions", "download",
                         "afri-voices-east-africa-asr-hackathon",
                         "-f", FT_CONFIG["kaggle_test_filename"], "-p", KAGGLE_TEST_DIR],
                        capture_output=True, text=True)
    print(rc.stdout[-300:], rc.stderr[-300:])
    if rc.returncode != 0:
        _err = rc.stdout + rc.stderr
        if "401" in _err:
            raise SystemExit("401 : kaggle.json invalide (régénérer un token).")
        if "403" in _err:
            raise SystemExit("403 : ACCEPTER LES RÈGLES de la compétition sur kaggle.com.")
        raise SystemExit(f"Téléchargement en échec : {_err[-500:]}")
    import zipfile
    for f in os.listdir(KAGGLE_TEST_DIR):
        if f.endswith(".zip"):
            with zipfile.ZipFile(os.path.join(KAGGLE_TEST_DIR, f)) as z:
                z.extractall(KAGGLE_TEST_DIR)
    _cands = [os.path.join(KAGGLE_TEST_DIR, f) for f in sorted(os.listdir(KAGGLE_TEST_DIR))
              if not f.endswith(".zip")]
    _found = _cands[0] if _cands else None

assert _found and os.path.exists(_found), "Aucun fichier test récupéré."
if _looks_like_captcha(_found):
    raise SystemExit(
        "⚠️ Kaggle a renvoyé une page reCAPTCHA au lieu du fichier (anti-robot). Solution : "
        "télécharger anv-test-data-nt DANS LE NAVIGATEUR depuis l'onglet Data de la compétition, "
        f"le déposer dans {_manual_dir}/ (créer le dossier), puis relancer cette cellule — "
        "elle le reprendra en priorité.")
print(f"\n===== {os.path.basename(_found)} ({os.path.getsize(_found)} octets) =====")
with open(_found, "rb") as fh:
    print(fh.read(2500))
KAGGLE_TEST_FILE = _found

In [ ]:
!pip install -q kagglehub
import kagglehub, shutil, os

# Téléchargement du dataset public (canal datasets : pas de captcha, utilise votre kaggle.json)
_dl = kagglehub.dataset_download("digitalumuganda/anv-test-data-nt")
print("téléchargé dans :", _dl)

# Copie sur VOTRE DRIVE (dossier que la 10.1 lit en priorité)
_manual_dir = os.path.join(DRIVE_ROOT, "kaggle_test")
os.makedirs(_manual_dir, exist_ok=True)
for _root, _, _files in os.walk(_dl):
    for _f in _files:
        _dst = os.path.join(_manual_dir, _f)
        shutil.copyfile(os.path.join(_root, _f), _dst)
        print(f"copié sur Drive : {_dst} ({os.path.getsize(_dst)} octets)")

# Inspection brute de chaque fichier (c'est CETTE sortie qu'il me faut)
print("\n===== INSPECTION =====")
for _f in sorted(os.listdir(_manual_dir)):
    _p = os.path.join(_manual_dir, _f)
    print(f"\n----- {_f} ({os.path.getsize(_p)} octets) -----")
    with open(_p, "rb") as fh:
        print(fh.read(2500))

In [ ]:
# 1) Nettoyer la copie à plat (corrompue par les collisions de noms) — puis vider la corbeille Drive
import shutil, os
shutil.rmtree(os.path.join(DRIVE_ROOT, "kaggle_test"), ignore_errors=True)
print("copie à plat supprimée — pensez à vider la corbeille Drive")

# 2) Inspecter la VRAIE structure depuis le cache kagglehub (intact)
_dl = "/root/.cache/kagglehub/datasets/digitalumuganda/anv-test-data-nt/versions/3"
for _root, _dirs, _files in os.walk(_dl):
    _depth = _root[len(_dl):].count(os.sep)
    print("  " * _depth + (os.path.basename(_root) or ".") + "/",
          f"[{len(_files)} fichier(s)]" if _files else "")
    for _f in sorted(_files)[:3]:
        print("  " * (_depth + 1) + f"{_f} ({os.path.getsize(os.path.join(_root, _f))} o)")

# 3) Schéma + quelques lignes d'un shard (colonnes non-audio)
import pyarrow.parquet as pq, glob
_one = sorted(glob.glob(os.path.join(_dl, "**", "*.parquet"), recursive=True))[0]
print("\nShard inspecté :", _one)
_schema = pq.read_schema(_one)
print("Colonnes top-level :", _schema.names)
_pf = pq.ParquetFile(_one)
print("Lignes du shard :", _pf.metadata.num_rows)
_cols = [c for c in _schema.names if c.lower() not in ("audio", "bytes", "wav", "data")]
print(f"Parquet schema columns: {_cols}; row preview omitted.")

In [ ]:
# 10.2 — Inventaire du dataset test + CACHE DRIVE (structure {lang}/{Scripted,Unscripted}/*.parquet)
# Dossiers = codes de soumission (swa, kik, kln, luo, som, mas). Le cache Drive évite de
# re-télécharger les 37,5 GB à chaque nouvelle session (kagglehub met en cache sur la VM,
# effacée au redémarrage).
if not RUN_TEST_INFERENCE:
    raise SystemExit("RUN_TEST_INFERENCE=True requis (2.2).")
import glob, subprocess
import pyarrow.parquet as pq

USE_TEST_DRIVE_CACHE = True
TEST_DRIVE = os.path.join(DRIVE_ROOT, "kaggle_test_full")
_marker = os.path.join(TEST_DRIVE, "_COMPLETE")

if USE_TEST_DRIVE_CACHE and os.path.exists(_marker):
    TEST_ROOT = TEST_DRIVE
    print("Dataset test repris du cache Drive (aucun téléchargement) :", TEST_ROOT)
else:
    subprocess.run(["pip", "install", "-q", "kagglehub"], check=False)
    import kagglehub
    _dl = kagglehub.dataset_download("digitalumuganda/anv-test-data-nt")
    print("Téléchargé via kagglehub :", _dl)
    if USE_TEST_DRIVE_CACHE:
        print("Copie structurée sur Drive (~37,5 GB, une seule fois — patiente)…")
        _tmp = TEST_DRIVE + ".tmp"
        subprocess.run(["rm", "-rf", _tmp], check=False)
        os.makedirs(_tmp, exist_ok=True)
        subprocess.run(["cp", "-r"] + glob.glob(os.path.join(_dl, "*")) + [_tmp], check=True)
        subprocess.run(["rm", "-rf", TEST_DRIVE], check=False)
        subprocess.run(["mv", _tmp, TEST_DRIVE], check=True)
        open(_marker, "w").write(now_iso())      # marqueur écrit EN DERNIER : cache partiel = invalide
        TEST_ROOT = TEST_DRIVE
        print("Cache Drive prêt :", TEST_ROOT)
    else:
        TEST_ROOT = _dl

EXPECTED_TEST_LANGS = {"swa", "kik", "kln", "luo", "som", "mas"}
# code de dossier (= code de soumission) -> code interne omnilingual pour l'inférence
TEST_DIR_TO_OMNI = {"swa": "swh_Latn", "kik": "kik_Latn", "kln": "kln_Latn",
                    "luo": "luo_Latn", "som": "som_Latn",
                    "mas": (LANG_FALLBACK_OMNI["mas"] if _use_mas_fallback else "mas_Latn")}
_found = {d for d in os.listdir(TEST_ROOT) if os.path.isdir(os.path.join(TEST_ROOT, d))}
assert EXPECTED_TEST_LANGS <= _found, f"Langues manquantes : {EXPECTED_TEST_LANGS - _found} (trouvé {_found})"

test_shards, TEST_TOTAL_ROWS = [], 0
for _lang in sorted(EXPECTED_TEST_LANGS):
    _paths = sorted(glob.glob(os.path.join(TEST_ROOT, _lang, "**", "*.parquet"), recursive=True))
    _rows = 0
    for _p in _paths:
        _n = pq.ParquetFile(_p).metadata.num_rows
        test_shards.append((_lang, _p, _n))
        _rows += _n
    TEST_TOTAL_ROWS += _rows
    print(f"  {_lang}: {len(_paths)} shard(s), {_rows} clips")
print(f"TOTAL : {len(test_shards)} shards, {TEST_TOTAL_ROWS} clips à transcrire")
save_json({"timestamp": now_iso(), "root": TEST_ROOT, "total_rows": TEST_TOTAL_ROWS,
           "shards": [{"lang": l, "path": p, "rows": n} for l, p, n in test_shards]},
          os.path.join(FT_CONFIG["report_dir"], "test_inventory.json"))

In [ ]:
# 10.3 — Transcription du test (REPRENABLE par shard) + submission.csv
if not RUN_TEST_INFERENCE:
    raise SystemExit("Inférence test bloquée.")
assert "test_shards" in globals(), "Inventaire absent : exécuter 10.2 d'abord."
import pyarrow.parquet as pq

_experiment_run = latest_ckpt(FT_CONFIG["output_dir"])
assert _experiment_run is not None, "Aucun checkpoint du run complet."
pipe = load_finetuned_pipeline(_experiment_run)

_pred_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions")
os.makedirs(_pred_dir, exist_ok=True)
_MAX_S = 38.0          # limite d'inférence CTC : < 40 s -> découpage des clips plus longs
_BATCH = 16

_B64_CHARS = set(b"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=\n\r")
def _ffmpeg_decode(raw, sr_out=16000):
    import subprocess
    try:
        proc = subprocess.run(["ffmpeg", "-v", "quiet", "-nostdin", "-i", "pipe:0",
                               "-f", "f32le", "-ac", "1", "-ar", str(sr_out), "pipe:1"],
                              input=raw, capture_output=True, check=True)
        wav = np.frombuffer(proc.stdout, dtype=np.float32).copy()
        return (wav, sr_out) if len(wav) else (None, None)
    except Exception:
        return None, None

def _decode_audio(cell):
    """Décodage TOLÉRANT : dict{bytes,path} / bytes / numpy int8-uint8 / memoryview ;
    soundfile puis repli ffmpeg ; (None, None) si le clip est réellement indécodable
    (le test contient probablement ~1% de clips corrompus, comme le train)."""
    raw = cell.get("bytes") if isinstance(cell, dict) else cell
    if raw is None:
        return None, None
    if hasattr(raw, "tobytes"):
        raw = raw.tobytes()
    elif isinstance(raw, (memoryview, bytearray)):
        raw = bytes(raw)
    # Certains clips stockent l'audio en BASE64 (texte) au lieu d'octets bruts
    # (ex: som/test_scripted_000 : 257 clips, header 'UklGR...' = base64 de 'RIFF').
    if raw[:5] == b"UklGR" or (len(raw) > 32 and set(raw[:80]) <= _B64_CHARS):
        try:
            import base64 as _b64
            _dec = _b64.b64decode(raw, validate=False)
            if _dec[:4] in (b"RIFF", b"fLaC", b"OggS") or _dec[:3] == b"ID3":
                raw = _dec
        except Exception:
            pass
    try:
        wav, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
    except Exception:
        return _ffmpeg_decode(raw)
    if getattr(wav, "ndim", 1) > 1:
        wav = wav.mean(axis=1)
    return wav, sr

_printed_type = False

for _lang, _shard, _nrows in test_shards:
    _out_csv = os.path.join(_pred_dir, f"{_lang}__{os.path.basename(os.path.dirname(_shard))}__"
                                       f"{os.path.basename(_shard)}.csv")
    if os.path.exists(_out_csv):
        continue   # reprise : shard déjà transcrit
    _t0 = time.time()
    _tbl = pq.read_table(_shard, columns=["id", "audio"]).to_pandas()
    # découpage éventuel des clips >= 40 s en segments <= 38 s (rejoints par espace)
    global _printed_type
    if not _printed_type:
        _c0 = _tbl.iloc[0]["audio"]
        print("type de la colonne audio :", type(_c0).__name__,
              "| clés :" if isinstance(_c0, dict) else "",
              list(_c0.keys()) if isinstance(_c0, dict) else "")
        _printed_type = True
    _items, _owner, _bad_rows = [], [], []
    for _ri, _row in _tbl.iterrows():
        wav, sr = _decode_audio(_row["audio"])
        if wav is None:
            _bad_rows.append(_ri)          # indécodable -> placeholder à l'assemblage
            continue
        if len(wav) < int(0.2 * sr):        # clip quasi vide : segment minimal (évite un rejet)
            wav = np.pad(wav, (0, int(0.2 * sr) - len(wav)))
        _limit = int(39.5 * sr)
        if len(wav) <= _limit:
            _items.append({"waveform": wav, "sample_rate": sr}); _owner.append(_ri)
        else:
            _step = int(_MAX_S * sr)
            for _off in range(0, len(wav), _step):
                _seg = wav[_off:_off + _step]
                if len(_seg) < int(0.2 * sr):
                    continue
                _items.append({"waveform": _seg, "sample_rate": sr}); _owner.append(_ri)
    if len(_bad_rows) > 0.5 * len(_tbl):
        _c0 = _tbl.iloc[_bad_rows[0]]["audio"]
        raise SystemExit(f"{len(_bad_rows)}/{len(_tbl)} clips indécodables dans ce shard : "
                         f"format inattendu (type={type(_c0).__name__}) — me montrer cette sortie.")
    _hyps = pipe.transcribe(_items, lang=[TEST_DIR_TO_OMNI[_lang]] * len(_items),
                            batch_size=_BATCH)
    _per_row = {}
    for _o, _h in zip(_owner, _hyps):
        _per_row.setdefault(_o, []).append(str(_h))
    _out = pd.DataFrame({"id": _tbl["id"],
                         "language": _lang,
                         "text": [normalize_text(" ".join(_per_row.get(_ri, [""])))
                                  for _ri in _tbl.index]})
    _out.to_csv(_out_csv, index=False)
    _done = len(glob.glob(os.path.join(_pred_dir, "*.csv")))
    print(f"[{_done}/{len(test_shards)}] {_lang}/{os.path.basename(_shard)} : {len(_tbl)} clips "
          f"({len(_bad_rows)} indécodables) en {(time.time()-_t0)/60:.1f} min")

# ---- Assemblage du submission.csv ----
_parts = [pd.read_csv(f, dtype=str) for f in sorted(glob.glob(os.path.join(_pred_dir, "*.csv")))]
submission = pd.concat(_parts, ignore_index=True)
submission["text"] = submission["text"].fillna("")
_empty = submission["text"].str.strip().eq("")
if int(_empty.sum()):
    print(f"⚠️ {int(_empty.sum())} prédiction(s) vide(s) remplacée(s) par un mot de remplissage.")
    submission.loc[_empty, "text"] = "a"
submission = submission.rename(columns={"text": SUBMISSION_TEXT_COLUMN})
submission = submission[["id", "language", SUBMISSION_TEXT_COLUMN]]
assert submission["language"].isin(set(LANG_TO_SUBMISSION.values())).all()
assert submission["id"].is_unique, "IDs dupliqués entre shards ?!"
assert len(submission) == TEST_TOTAL_ROWS, f"{len(submission)} lignes != {TEST_TOTAL_ROWS} attendues"
sub_path = os.path.join(FT_CONFIG["report_dir"], "submission.csv")
submission.to_csv(sub_path, index=False)
print(f"submission.csv : {len(submission)} lignes -> {sub_path}")
print(f"Submission assembled: {len(submission)} rows; preview omitted.")
print(f"Rappel : en-tête texte = '{SUBMISSION_TEXT_COLUMN}' (changer en 2.1 si Kaggle rejette) ; "
      "si les IDs sont refusés, variante sans extension : submission['id'].str.replace('.wav','')")

In [ ]:
# DIAGNOSTIC : récupération des clips som indécodables
import pyarrow.parquet as pq, io, os, subprocess, tempfile
import numpy as np, soundfile as sf
from collections import Counter

_shard = next(p for l, p, n in test_shards if l == "som" and "scripted_000" in p)
_tbl = pq.read_table(_shard, columns=["id", "audio"]).to_pandas()

def _bytes_of(cell):
    raw = cell.get("bytes") if isinstance(cell, dict) else cell
    if raw is None: return None
    if hasattr(raw, "tobytes"): raw = raw.tobytes()
    return bytes(raw) if isinstance(raw, (memoryview, bytearray)) else raw

def _decode_orig(raw):   # exactement ce que fait 10.3 (soundfile BytesIO -> ffmpeg pipe)
    if raw is None: return None
    try:
        return sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)[0]
    except Exception:
        try:
            p = subprocess.run(["ffmpeg","-v","quiet","-nostdin","-i","pipe:0","-f","f32le",
                                "-ac","1","-ar","16000","pipe:1"], input=raw, capture_output=True, check=True)
            w = np.frombuffer(p.stdout, dtype=np.float32)
            return w if len(w) else None
        except Exception:
            return None

fails = [(i, _bytes_of(r["audio"])) for i, r in _tbl.iterrows()]
fails = [(i, raw) for i, raw in fails if _decode_orig(raw) is None]
print(f"{len(fails)} échecs sur {len(_tbl)}")
for i, raw in fails[:5]:
    print(f"  row {i}: {'bytes=None' if raw is None else f'{len(raw)} o, header={raw[:24]}'}")

def _recover(raw):
    if raw is None: return None, "no_bytes"
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(raw); tmp = f.name
    try:
        try:
            w = sf.read(tmp, dtype="float32", always_2d=False)[0]
            return (w.mean(axis=1) if getattr(w,"ndim",1)>1 else w), "file_soundfile"
        except Exception: pass
        for extra in (["-ignore_length","1"], ["-err_detect","ignore_err"]):
            try:
                p = subprocess.run(["ffmpeg","-v","quiet","-nostdin",*extra,"-i",tmp,"-f","f32le",
                                    "-ac","1","-ar","16000","pipe:1"], capture_output=True, check=True)
                w = np.frombuffer(p.stdout, dtype=np.float32).copy()
                if len(w): return w, "ffmpeg" + extra[0]
            except Exception: pass
        if raw[:4] == b"RIFF":   # dernier repli : lire les octets après le header comme PCM s16 brut
            pcm = raw[44:]; w = np.frombuffer(pcm[:len(pcm)//2*2], dtype=np.int16).astype(np.float32)/32768
            if len(w): return w, "raw_pcm_s16"
        return None, "unrecoverable"
    finally:
        os.remove(tmp)

out, lens = Counter(), []
for i, raw in fails:
    w, how = _recover(raw); out[how] += 1
    if w is not None: lens.append(len(w)/44100)   # sr natif ~44.1k
print("récupération :", dict(out))
if lens: print(f"durées récupérées : {min(lens):.1f}–{max(lens):.1f}s (médiane {sorted(lens)[len(lens)//2]:.1f}s)")

In [ ]:
# SONDE v3 : RawSentencePieceTokenizer -> trouver id_to_piece + nature des tokens
import torch
print("type:", type(_tok).__name__)
print("attrs:", [a for a in dir(_tok) if not a.startswith("__")])

# 1) chercher le processeur SentencePiece (qq part dans _tok)
_spm = None
for a in dir(_tok):
    o = getattr(_tok, a, None)
    if hasattr(o, "id_to_piece") or hasattr(o, "IdToPiece"):
        _spm = o; print("SPM trouvé dans _tok.%s (%s)" % (a, type(o).__name__)); break

def _id2piece(i):
    if _spm is not None:
        try: return (_spm.id_to_piece(int(i)) if hasattr(_spm,"id_to_piece")
                     else _spm.IdToPiece(int(i)))
        except Exception: return ""
    return None

if _spm is not None:
    print("pieces 0-6:", [repr(_id2piece(i)) for i in range(7)])
    print("pieces argmax:", [(i, repr(_id2piece(i))) for i in {_am[10],_am[20],_am[30],1489}])
    _lens = [len(_id2piece(i) or "") for i in range(_tok.vocab_info.size)]
    import collections
    print("longueurs de pièces:", dict(collections.Counter(_lens).most_common(6)),
          "| max:", max(_lens))

# 2) repli : create_decoder par id (méthode utilisée par le pipeline)
_decf = _tok.create_decoder()
def id2s_dec(i):
    for dt in (torch.int64, torch.int32):
        try:
            r = _decf(torch.tensor([int(i)], dtype=dt))
            return r if isinstance(r, str) else (str(r[0]) if len(r) else "")
        except Exception: continue
    return None
print("\ndecoder([i]) 0,4,1489,char:",
      {i: repr(id2s_dec(i)) for i in (0, 4, 1489, _am[10])})

# 3) reconstruire le greedy avec la source qui marche (SPM en priorité) et comparer
def piece_to_text(p):
    return (p or "").replace("\u2581", " ")   # ▁ SentencePiece -> espace
_src = _id2piece if _spm is not None else id2s_dec
_ded = [t for j, t in enumerate(_am) if j == 0 or t != _am[j-1]]
_manual = "".join(piece_to_text(_src(t)) for t in _ded if t != 0)
print("\ngreedy manuel:", _manual[:90])
print("greedy pipe  :", _txt[0][:90])
print("MATCH:", _manual.strip() == _txt[0].strip())

## 11 — Résumé final du run

In [ ]:
# 11.1 — Synthèse
_summary = {
    "timestamp": now_iso(),
    "run": FT_RUN_NAME,
    "model": FT_CONFIG["model_card"],
    "grad_accum": FT_CONFIG["grad_accum"],
    "max_num_elements": FT_CONFIG["max_num_elements"],
    "mas_fallback": bool(_use_mas_fallback),
    "reports": sorted(os.listdir(FT_CONFIG["report_dir"])),
    "last_checkpoint": latest_ckpt(FT_CONFIG["output_dir"]),
}
save_json(_summary, os.path.join(FT_CONFIG["report_dir"], "run_summary.json"))
for k, v in _summary.items():
    print(f"{k}: {v}")


In [ ]:
# 12.1 — Construire un KenLM par langue (sur les transcriptions du TRAIN)
import subprocess, glob

KENLM_DIR = "/content/kenlm"
LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")   # persistant sur Drive
os.makedirs(LM_DRIVE, exist_ok=True)
KENLM_ORDER = 5

subprocess.run(["pip", "install", "-q", "pyctcdecode"], check=False)

# Outils KenLM (lmplz, build_binary) : compilés une seule fois sur la VM
_lmplz = os.path.join(KENLM_DIR, "build", "bin", "lmplz")
_bbin = os.path.join(KENLM_DIR, "build", "bin", "build_binary")
if not os.path.exists(_lmplz):
    subprocess.run("apt-get install -y libeigen3-dev libboost-all-dev",
                   shell=True, capture_output=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/kpu/kenlm.git", KENLM_DIR], check=True)
    _r = subprocess.run(f"cd {KENLM_DIR} && mkdir -p build && cd build && cmake .. && make -j4",
                        shell=True, capture_output=True, text=True)
    if not os.path.exists(_lmplz):
        print(_r.stdout[-1500:], _r.stderr[-1500:])
        raise SystemExit("Compilation KenLM échouée (lmplz introuvable) — voir logs ci-dessus.")
print("KenLM prêt :", _lmplz)

# Un LM par langue depuis le TRAIN (déjà normalisé : text_norm)
LM_PATHS = {}
for lang in LANGUAGES:
    _bin = os.path.join(LM_DRIVE, f"{lang}.binary")
    if os.path.exists(_bin):
        LM_PATHS[lang] = _bin
        print(f"  {lang}: LM déjà présent ({os.path.getsize(_bin)//1024} Ko)")
        continue
    _sents = manifest[(manifest.split == "train") & (manifest.language == lang)]["text_norm"].tolist()
    _txt = f"/content/lm_{lang}.txt"
    with open(_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(_sents))
    _arpa = f"/content/lm_{lang}.arpa"
    _order = KENLM_ORDER
    _r = subprocess.run(f"{_lmplz} -o {_order} --discount_fallback < {_txt} > {_arpa}",
                        shell=True, capture_output=True, text=True)
    if _r.returncode != 0:            # repli ordre 3 si le corpus est trop petit
        _order = 3
        _r = subprocess.run(f"{_lmplz} -o {_order} --discount_fallback < {_txt} > {_arpa}",
                            shell=True, capture_output=True, text=True)
    assert _r.returncode == 0, f"lmplz {lang} échec : {_r.stderr[-600:]}"
    subprocess.run(f"{_bbin} {_arpa} {_bin}", shell=True, check=True)
    LM_PATHS[lang] = _bin
    print(f"  {lang}: {len(_sents)} phrases -> {_order}-gramme -> {os.path.getsize(_bin)//1024} Ko")

# Vocabulaire par langue (unigrammes) : aide pyctcdecode à mieux borner les mots
LM_UNIGRAMS = {}
for lang in LANGUAGES:
    _words = set()
    for _t in manifest[(manifest.split == "train") & (manifest.language == lang)]["text_norm"]:
        _words.update(str(_t).split())
    LM_UNIGRAMS[lang] = sorted(_words)
    print(f"  {lang}: {len(LM_UNIGRAMS[lang])} mots-vocabulaire")

save_json({"lm_paths": LM_PATHS, "order": KENLM_ORDER,
           "unigram_counts": {l: len(v) for l, v in LM_UNIGRAMS.items()}},
          os.path.join(FT_CONFIG["report_dir"], "kenlm_build_report.json"))
print("LMs prêts :", {k: os.path.basename(v) for k, v in LM_PATHS.items()})

In [ ]:
# 12.1b — KenLM v3 : corpus enrichi = train + valid par langue, local_test EXCLU car c'est
# l'échantillon de réglage de 12.2c — l'inclure ferait mémoriser au LM les phrases de
# référence du tuning (fuite constatée avec les binaires v2 : WER fictifs, alpha au plafond).
# Binaires sous kenlm_models_v3/ ; les v1 (train seul) restent — 12.2c choisit par langue.
import subprocess, glob

KENLM_DIR = "/content/kenlm"
LM_DRIVE_V3 = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_v3")
os.makedirs(LM_DRIVE_V3, exist_ok=True)
_lmplz = os.path.join(KENLM_DIR, "build", "bin", "lmplz")
_bbin = os.path.join(KENLM_DIR, "build", "bin", "build_binary")
if not os.path.exists(_lmplz):
    subprocess.run("apt-get install -y libeigen3-dev libboost-all-dev",
                   shell=True, capture_output=True)
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/kpu/kenlm.git", KENLM_DIR], check=True)
    _r = subprocess.run(f"cd {KENLM_DIR} && mkdir -p build && cd build && cmake .. && make -j4",
                        shell=True, capture_output=True, text=True)
    if not os.path.exists(_lmplz):
        print(_r.stdout[-1500:], _r.stderr[-1500:])
        raise SystemExit("Compilation KenLM échouée (lmplz introuvable) — voir logs ci-dessus.")
print("KenLM prêt :", _lmplz)

LM_PATHS_V3 = {}
for lang in LANGUAGES:
    _bin = os.path.join(LM_DRIVE_V3, f"{lang}.binary")
    if os.path.exists(_bin):
        LM_PATHS_V3[lang] = _bin
        print(f"  {lang}: LM v3 déjà présent ({os.path.getsize(_bin)//1024} Ko)")
        continue
    _sents = manifest[manifest.split.isin(["train", "valid"])
                  & (manifest.language == lang)]["text_norm"].tolist()
    _txt, _arpa = f"/content/lm2_{lang}.txt", f"/content/lm2_{lang}.arpa"
    with open(_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(str(s) for s in _sents))
    def _lmplz_run(order):
        with open(_txt, "rb") as fi, open(_arpa, "wb") as fo:
            return subprocess.run([_lmplz, "-o", str(order), "--discount_fallback"],
                                  stdin=fi, stdout=fo, stderr=subprocess.PIPE, text=False)
    _order = 5
    _r = _lmplz_run(_order)
    if _r.returncode != 0:            # repli ordre 3 si le corpus est trop petit
        _order = 3
        _r = _lmplz_run(_order)
    assert _r.returncode == 0, f"lmplz {lang} échec : {_r.stderr[-600:].decode(errors='replace')}"
    subprocess.run([_bbin, _arpa, _bin], check=True)
    LM_PATHS_V3[lang] = _bin
    print(f"  {lang}: {len(_sents)} phrases -> {_order}-gramme v3 -> {os.path.getsize(_bin)//1024} Ko")
print("LMs v3 prêts -> 12.2c")

In [ ]:
# 12.2 — Décodeurs pyctcdecode + réglage alpha/beta sur local_test (valid)
import numpy as np, jiwer
from pyctcdecode import build_ctcdecoder

LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")
if "LM_PATHS" not in globals():
    LM_PATHS = {l: os.path.join(LM_DRIVE, f"{l}.binary") for l in LANGUAGES}
assert all(os.path.exists(p) for p in LM_PATHS.values()), "LMs manquants : exécuter 12.1."
if "LM_UNIGRAMS" not in globals():
    LM_UNIGRAMS = {}
    for l in LANGUAGES:
        _w = set()
        for t in manifest[(manifest.split == "train") & (manifest.language == l)]["text_norm"]:
            _w.update(str(t).split())
        LM_UNIGRAMS[l] = sorted(_w)

# Pipeline + labels pour pyctcdecode : pièces SentencePiece (UNIQUES par construction ;
# create_decoder produisait des doublons "" pour bos/pad/eos/unk). blank = index 0 -> "".
_pipe_lm = load_finetuned_pipeline(latest_ckpt(FT_CONFIG["output_dir"]))
_tkm = getattr(_pipe_lm.tokenizer, "_model", None)
LABELS = None
for _meth in ("index_to_token", "id_to_piece", "IdToPiece"):
    if _tkm is not None and hasattr(_tkm, _meth):
        try:
            LABELS = [str(getattr(_tkm, _meth)(i))
                      for i in range(_pipe_lm.tokenizer.vocab_info.size)]
            print("vocab via _model.%s" % _meth)
            break
        except Exception:
            LABELS = None
assert LABELS is not None, "Vocab SentencePiece inaccessible (voir _pipe_lm.tokenizer._model)."
_dups = len(LABELS) - len(set(LABELS))
assert _dups == 0, f"{_dups} pièces SP dupliquées — me montrer un échantillon."
LABELS[0] = ""   # blank CTC (index 0, dominant en argmax)

# Capture des log-probs CTC (spy sur le forward ; ordre garanti par batch_size=1)
def capture_logprobs(inputs, omni_code):
    caps = []
    _orig = _pipe_lm.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    _pipe_lm.model.forward = _spy
    try:
        _pipe_lm.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        _pipe_lm.model.forward = _orig
    return caps

# Test de cohérence : pyctcdecode SANS LM doit reproduire le greedy du pipeline (sinon
# labels/blank à revoir) — quelques secondes, évite un réglage/inférence sur une base fausse.
_pr = manifest[manifest.split == "valid"].iloc[0]
_lp0 = capture_logprobs([_pr["curated_audio_path"]], omni_lang(_pr["language"]))[0]
_g_ctc = normalize_text(build_ctcdecoder(LABELS).decode(_lp0))
_g_pipe = normalize_text(_pipe_lm.transcribe([_pr["curated_audio_path"]],
                         lang=[omni_lang(_pr["language"])], batch_size=1)[0])
print("sanity ctc :", _g_ctc[:70])
print("sanity pipe:", _g_pipe[:70])
assert jiwer.wer([_g_pipe], [_g_ctc]) < 0.15, \
    "pyctcdecode ne reproduit pas le greedy pipeline -> labels/blank/▁ à revoir (me montrer ces 2 lignes)."

# Grille alpha (poids LM) / beta (insertion) sur ~120 clips valid par langue
_val = (manifest[manifest.split == "valid"].sample(frac=1.0, random_state=FT_CONFIG["seed"])
        .groupby("language").head(120))
ALPHAS = [0.0, 0.3, 0.5, 0.8]
BETAS = [0.0, 1.0]
LM_BEST = {}
for lang in LANGUAGES:
    sub = _val[_val.language == lang]
    lp = capture_logprobs(sub["curated_audio_path"].tolist(), omni_lang(lang))
    refs = sub["text_norm"].tolist()
    dec0 = build_ctcdecoder(LABELS, unigrams=LM_UNIGRAMS[lang])
    wer_greedy = jiwer.wer(refs, [normalize_text(dec0.decode(x)) for x in lp])
    best = (wer_greedy, 0.0, 0.0)
    for a in ALPHAS:
        for b in BETAS:
            if a == 0.0 and b == 0.0:
                continue
            dec = build_ctcdecoder(LABELS, LM_PATHS[lang], unigrams=LM_UNIGRAMS[lang], alpha=a, beta=b)
            w = jiwer.wer(refs, [normalize_text(dec.decode(x)) for x in lp])
            if w < best[0]:
                best = (w, a, b)
    LM_BEST[lang] = {"alpha": best[1], "beta": best[2],
                     "wer_greedy": round(wer_greedy, 4), "wer_best": round(best[0], 4)}
    print(f"{lang}: greedy {wer_greedy:.4f} -> LM {best[0]:.4f}  (alpha={best[1]}, beta={best[2]})")
save_json(LM_BEST, os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json"))
_gain = float(np.mean([LM_BEST[l]["wer_greedy"] - LM_BEST[l]["wer_best"] for l in LANGUAGES]))
print(f"\nGain WER moyen (valid) : {_gain:+.4f}  -> "
      f"{'LM UTILE, lancer 12.3' if _gain > 0.002 else 'gain faible, me montrer les chiffres'}")

In [ ]:
# 12.2b — Raffinage alpha/beta autour de l'optimum 12.2 + largeur de beam (rejouable)
# La grille 12.2 était grossière (alpha max 0.8, atteint par certaines langues = optimum au bord).
# Ici : grille LOCALE par langue centrée sur kenlm_tuning.json, échantillon valid élargi
# 120->200 clips (même seed : les 120 premiers sont identiques + 80 nouveaux, réglage plus stable),
# puis essai beam_width=250 au meilleur couple (décodage ~2,5x plus lent : gardé si gain réel).
# À la fin : sauvegarde kenlm_tuning.json (backup v1) et INVALIDE les CSV test des langues dont
# les paramètres ont changé -> relancer 12.3 (la reprise ne refera que ces langues-là).
import glob, json, shutil, subprocess
try:      # kenlm (module python) requis par pyctcdecode pour charger les .binary — il n'est
    import kenlm, pyctcdecode          # installé par AUCUNE cellule (venait d'une sonde) -> auto
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import numpy as np, jiwer
from pyctcdecode import build_ctcdecoder

_tune_path = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
assert os.path.exists(_tune_path), "kenlm_tuning.json absent : exécuter 12.2 d'abord."
_prev = json.load(open(_tune_path))

LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")
if "LM_PATHS" not in globals():
    LM_PATHS = {l: os.path.join(LM_DRIVE, f"{l}.binary") for l in LANGUAGES}
assert all(os.path.exists(p) for p in LM_PATHS.values()), "LMs manquants : exécuter 12.1."
if "LM_UNIGRAMS" not in globals():
    LM_UNIGRAMS = {}
    for l in LANGUAGES:
        _w = set()
        for t in manifest[(manifest.split == "train") & (manifest.language == l)]["text_norm"]:
            _w.update(str(t).split())
        LM_UNIGRAMS[l] = sorted(_w)

if "capture_logprobs" not in globals() or "LABELS" not in globals():
    # session neuve : pipeline + labels + capture reconstruits (repris de 12.2, mêmes garde-fous)
    _pipe_lm = load_finetuned_pipeline(latest_ckpt(FT_CONFIG["output_dir"]))
    _tkm = getattr(_pipe_lm.tokenizer, "_model", None)
    LABELS = None
    for _meth in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _meth):
            try:
                LABELS = [str(getattr(_tkm, _meth)(i))
                          for i in range(_pipe_lm.tokenizer.vocab_info.size)]
                break
            except Exception:
                LABELS = None
    assert LABELS is not None and len(LABELS) == len(set(LABELS)), "Vocab SP inaccessible/dupliqué."
    LABELS[0] = ""   # blank CTC
    def capture_logprobs(inputs, omni_code):
        caps = []
        _orig = _pipe_lm.model.forward
        def _spy(src, bl, *a, **kw):
            out = _orig(src, bl, *a, **kw)
            lg = out[0] if isinstance(out, tuple) else out
            caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
            return out
        _pipe_lm.model.forward = _spy
        try:
            _pipe_lm.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
        finally:
            _pipe_lm.model.forward = _orig
        return caps

_val = (manifest[manifest.split == "valid"].sample(frac=1.0, random_state=FT_CONFIG["seed"])
        .groupby("language").head(200))
LM_BEST = {}
for lang in LANGUAGES:
    _t0 = time.time()
    a0, b0 = float(_prev[lang]["alpha"]), float(_prev[lang]["beta"])
    _as = sorted({round(max(0.0, a0 + d), 2) for d in (-0.15, 0.0, 0.15, 0.3, 0.5)})
    _bs = sorted({round(max(0.0, b0 + d), 2) for d in (-1.0, 0.0, 1.0, 2.0)})
    sub = _val[_val.language == lang]
    refs = sub["text_norm"].tolist()
    lp = capture_logprobs(sub["curated_audio_path"].tolist(), omni_lang(lang))
    best = None                       # (a0,b0) est dans la grille -> baseline comparable incluse
    for a in _as:
        for b in _bs:
            dec = build_ctcdecoder(LABELS, LM_PATHS[lang], unigrams=LM_UNIGRAMS[lang],
                                   alpha=a, beta=b)
            w = jiwer.wer(refs, [normalize_text(dec.decode(x)) for x in lp])
            if best is None or w < best[0]:
                best = (w, a, b)
    w100, a1, b1 = best
    dec = build_ctcdecoder(LABELS, LM_PATHS[lang], unigrams=LM_UNIGRAMS[lang], alpha=a1, beta=b1)
    w250 = jiwer.wer(refs, [normalize_text(dec.decode(x, beam_width=250)) for x in lp])
    beam = 250 if w250 < w100 - 0.002 else 100
    LM_BEST[lang] = {"alpha": a1, "beta": b1, "beam": beam,
                     "wer_best": round(w250 if beam == 250 else w100, 4),
                     "wer_12_2": _prev[lang].get("wer_best"),
                     "wer_greedy": _prev[lang].get("wer_greedy")}
    print(f"{lang}: (a={a0}, b={b0}) wer12.2={_prev[lang].get('wer_best')} -> "
          f"(a={a1}, b={b1}, beam={beam}) wer={LM_BEST[lang]['wer_best']}"
          f"  [{(time.time()-_t0)/60:.1f} min]")
    del lp

_v1 = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_v1.json")
if not os.path.exists(_v1):
    shutil.copy(_tune_path, _v1)      # garder la version 12.2 (celle de la soumission 0.38888)
save_json(LM_BEST, _tune_path)

# Invalidation des prédictions test des langues dont les paramètres ont CHANGÉ (critère =
# paramètres, pas WER : les WER 12.2/12.2b ne sont pas sur le même nombre de clips).
_MANIFEST_TO_DIR = {"sw": "swa"}      # seuls les CSV utilisent les codes des dossiers test
_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_stale, _changed = 0, []
for lang in LANGUAGES:
    _same = (LM_BEST[lang]["alpha"] == float(_prev[lang]["alpha"])
             and LM_BEST[lang]["beta"] == float(_prev[lang]["beta"])
             and LM_BEST[lang]["beam"] == 100)
    if not _same:
        _changed.append(lang)
        for f in glob.glob(os.path.join(_lm_dir, f"{_MANIFEST_TO_DIR.get(lang, lang)}__*.csv")):
            os.remove(f); _stale += 1
_gain = float(np.mean([float(_prev[l]["wer_best"]) - LM_BEST[l]["wer_best"] for l in LANGUAGES]))
print(f"\nGain WER moyen supplémentaire (valid) : {_gain:+.4f}")
if _changed:
    print(f"Langues modifiées : {_changed} -> {_stale} CSV de shards invalidés.")
    print("RELANCER 12.3 : la reprise saute les langues inchangées et refait celles-ci.")
else:
    print("Aucun paramètre modifié : l'optimum 12.2 était déjà le bon, ne pas relancer 12.3.")

In [ ]:
# 12.2c — POLISH sans fuite : re-réglage alpha/beta/beam par langue sur les logits ACTUELS
# (routage top-ups) + choix v1 (train) vs v3 (train+valid) par langue. Réglage sur LOCAL_TEST,
# jamais vu par les LM ni par aucune décision antérieure — régler sur valid avec un LM qui
# contient valid mémorise les références (fuite v2 : WER fictifs 0.04-0.23, alpha au plafond).
# ~1h30. Invalide les CSV des langues modifiées -> relancer 12.3 puis SOUMETTRE.
import subprocess, json, glob, shutil
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import numpy as np, jiwer
from pyctcdecode import build_ctcdecoder

assert "LM_PATHS_V3" in globals(), "Exécuter 12.1b d'abord."
LM_PATHS_V1 = {l: os.path.join(FT_CONFIG["experiment_run"], "kenlm_models", f"{l}.binary")
               for l in LANGUAGES}

# Routage joint/top-ups : TOUJOURS resynchronisé depuis les verdicts 13.3 — le nettoyage GPU
# de 13.2 vide TOPUP_PIPES, un dict partiel ou un capture_logprobs non routé (12.2b) peuvent
# subsister dans la session -> tout est redéfini ici, sans condition.
if "TOPUP_PIPES" not in globals():
    TOPUP_PIPES = {}
if "_pipe_joint" not in globals():
    _pipe_joint = load_finetuned_pipeline(latest_ckpt(FT_CONFIG["output_dir"]))
for _f in glob.glob(os.path.join(FT_CONFIG["report_dir"], "topup_choice_*.json")):
    _c = json.load(open(_f))
    if _c["chosen"] != "joint" and omni_lang(_c["language"]) not in TOPUP_PIPES:
        TOPUP_PIPES[omni_lang(_c["language"])] = load_finetuned_pipeline(_c["ckpt"])
def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""
                return L
            except Exception:
                pass
    raise SystemExit("Vocab SentencePiece inaccessible.")
def _capture(pipe, inputs, omni_code):
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        hyps = pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps, hyps
LABELS = _labels_from(_pipe_joint)
def capture_logprobs(inputs, omni_code):
    return _capture(TOPUP_PIPES.get(omni_code, _pipe_joint), inputs, omni_code)[0]
print("routage :", (", ".join(TOPUP_PIPES) + " -> top-up, ") if TOPUP_PIPES else "", "autres -> joint")

if "LM_UNIGRAMS" not in globals():                       # unigrammes v1 : train seul
    LM_UNIGRAMS = {}
    for l in LANGUAGES:
        _w = set()
        for t in manifest[(manifest.split == "train") & (manifest.language == l)]["text_norm"]:
            _w.update(str(t).split())
        LM_UNIGRAMS[l] = sorted(_w)
_uni3 = {}                                               # unigrammes v3 : train + valid
for l in LANGUAGES:
    _w = set()
    for t in manifest[manifest.split.isin(["train", "valid"])
                      & (manifest.language == l)]["text_norm"]:
        _w.update(str(t).split())
    _uni3[l] = sorted(_w)

_prev_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_pre_polish.json")
# base de comparaison = derniers réglages SAINS (le backup pré-polish si la 12.2c fuyarde a tourné)
_prev = json.load(open(_prev_p if os.path.exists(_prev_p)
                       else os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
ALPHAS = [0.3, 0.5, 0.65, 0.8, 1.0]
_val = (manifest[manifest.split == "local_test"]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).groupby("language").head(200))
assert len(_val), "local_test absent du manifest ?"
LM_BEST = {}
for lang in LANGUAGES:
    _t0 = time.time()
    sub = _val[_val.language == lang]
    refs = sub["text_norm"].tolist()
    lp = capture_logprobs(sub["curated_audio_path"].tolist(), omni_lang(lang))
    _b0 = float(_prev[lang]["beta"])
    BETAS = sorted({max(0.0, _b0 - 1.0), _b0, _b0 + 1.0})
    best = None
    for _tag, _bin, _uni in (("v1", LM_PATHS_V1[lang], LM_UNIGRAMS[lang]),
                             ("v3", LM_PATHS_V3[lang], _uni3[lang])):
        for a in ALPHAS:
            for b in BETAS:
                dec = build_ctcdecoder(LABELS, _bin, unigrams=_uni, alpha=a, beta=b)
                w = jiwer.wer(refs, [normalize_text(dec.decode(x)) for x in lp])
                if best is None or w < best[0]:
                    best = (w, _tag, _bin, a, b)
    w1, _tag, _bin, a1, b1 = best
    dec = build_ctcdecoder(LABELS, _bin, unigrams=(_uni3[lang] if _tag == "v3" else LM_UNIGRAMS[lang]),
                           alpha=a1, beta=b1)
    w250 = jiwer.wer(refs, [normalize_text(dec.decode(x, beam_width=250)) for x in lp])
    beam = 250 if w250 < w1 - 0.002 else 100
    LM_BEST[lang] = {"alpha": a1, "beta": b1, "beam": beam, "lm": _tag, "lm_bin": _bin,
                     "wer_best": round(min(w1, w250) if beam == 250 else w1, 4),
                     "wer_avant_polish": _prev[lang].get("wer_best")}
    print(f"{lang}: LM {_tag} (a={a1}, b={b1}, beam={beam})  WER {LM_BEST[lang]['wer_best']}"
          f"  (avant polish {_prev[lang].get('wer_best')})  [{(time.time()-_t0)/60:.1f} min]")
    del lp

if not os.path.exists(_prev_p):        # ne jamais écraser le backup sain d'origine
    shutil.copy(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json"), _prev_p)
save_json(LM_BEST, os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json"))
LM_PATHS = {l: LM_BEST[l]["lm_bin"] for l in LANGUAGES}          # utilisés par 12.3 (_DECODERS)
LM_UNIGRAMS = {l: (_uni3[l] if LM_BEST[l]["lm"] == "v3" else LM_UNIGRAMS[l]) for l in LANGUAGES}

# invalider les CSV test des langues dont les réglages ont changé
_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_MAN2DIR = {"sw": "swa"}
_n, _changed = 0, []
for lang in LANGUAGES:
    _o = _prev[lang]
    if (LM_BEST[lang]["alpha"], LM_BEST[lang]["beta"], LM_BEST[lang]["beam"], LM_BEST[lang]["lm"]) \
       != (float(_o["alpha"]), float(_o["beta"]), int(_o.get("beam", 100)), _o.get("lm", "v1")):
        _changed.append(lang)
        for f in glob.glob(os.path.join(_lm_dir, f"{_MAN2DIR.get(lang, lang)}__*.csv")):
            os.remove(f)
            _n += 1
print(f"\nLangues modifiées : {_changed} -> {_n} CSV invalidés. RELANCER 12.3 puis soumettre.")

In [ ]:
# 12.2d — RESTAURATION de session pour l'inférence (AUCUN re-réglage) : reconstruit routage,
# LABELS, capture_logprobs et la config LM depuis les fichiers d'état du Drive
# (topup_choice_*.json + kenlm_tuning.json). Après une déconnexion : préambule -> 12.2d -> 12.3
# (la reprise par shard conserve tout ce qui était déjà transcrit).
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
from pyctcdecode import build_ctcdecoder

if "_pipe_joint" not in globals():
    _pipe_joint = load_finetuned_pipeline(latest_ckpt(FT_CONFIG["output_dir"]))
if "TOPUP_PIPES" not in globals():
    TOPUP_PIPES = {}
for _f in glob.glob(os.path.join(FT_CONFIG["report_dir"], "topup_choice_*.json")):
    _c = json.load(open(_f))
    if _c["chosen"] != "joint" and omni_lang(_c["language"]) not in TOPUP_PIPES:
        TOPUP_PIPES[omni_lang(_c["language"])] = load_finetuned_pipeline(_c["ckpt"])

def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""
                return L
            except Exception:
                pass
    raise SystemExit("Vocab SentencePiece inaccessible.")
def _capture(pipe, inputs, omni_code):
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        hyps = pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps, hyps
LABELS = _labels_from(_pipe_joint)
def capture_logprobs(inputs, omni_code):
    return _capture(TOPUP_PIPES.get(omni_code, _pipe_joint), inputs, omni_code)[0]

LM_BEST = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")
LM_PATHS = {l: LM_BEST.get(l, {}).get("lm_bin") or os.path.join(LM_DRIVE, f"{l}.binary")
            for l in LANGUAGES}
assert all(os.path.exists(p) for p in LM_PATHS.values()), "LMs manquants sur Drive."
LM_UNIGRAMS = {}
for l in LANGUAGES:
    _uf = LM_BEST.get(l, {}).get("uni_file")    # v4 : fichier d'unigrammes dédié (14.5/14.6)
    if _uf and os.path.exists(_uf):
        LM_UNIGRAMS[l] = open(_uf, encoding="utf-8").read().splitlines()
        continue
    _tag = LM_BEST.get(l, {}).get("lm", "v1")   # v1=train, v3=train+valid, web=fichier dédié
    if _tag == "web":
        _p = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_web", f"{l}_unigrams.txt")
        LM_UNIGRAMS[l] = open(_p, encoding="utf-8").read().splitlines()
        continue
    _rows = (manifest if _tag == "v2"
             else manifest[manifest.split.isin(["train", "valid"])] if _tag == "v3"
             else manifest[manifest.split == "train"])
    _w = set()
    for t in _rows[_rows.language == l]["text_norm"]:
        _w.update(str(t).split())
    LM_UNIGRAMS[l] = sorted(_w)

_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_done = len(glob.glob(os.path.join(_lm_dir, "*.csv")))
print("routage :", (", ".join(TOPUP_PIPES) + " -> top-up, ") if TOPUP_PIPES else "", "autres -> joint")
print("LM par langue :", {l: LM_BEST.get(l, {}).get("lm", "v1") for l in LANGUAGES})
print(f"état inférence restauré ; {_done} CSV de shards déjà présents -> 12.3 reprendra le reste.")

In [ ]:
# 12.3 — Inférence test KenLM -> submission_lm.csv (REPRENABLE par shard) — V2 RECOUVREMENT :
# les clips >38 s sont découpés en fenêtres de 38 s avec 6 s de recouvrement ; les frames de
# logits des bords (3 s de chaque côté intérieur) sont TRONQUÉES puis les logits concaténés et
# décodés EN UNE SEULE passe beam par clip — le LM garde la continuité aux frontières
# (méthode standard HF/wav2vec2, cf. verdict_recherche.md point 1).
if not RUN_TEST_INFERENCE:
    raise SystemExit("RUN_TEST_INFERENCE=True requis (2.2).")
assert "test_shards" in globals(), "Exécuter 10.2 d'abord."
assert "LM_BEST" in globals(), "Exécuter 12.2d d'abord."
import pyarrow.parquet as pq, glob, subprocess

_B64_CHARS = set(b"ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=\n\r")
def _ffmpeg_decode(raw, sr_out=16000):
    try:
        proc = subprocess.run(["ffmpeg", "-v", "quiet", "-nostdin", "-i", "pipe:0",
                               "-f", "f32le", "-ac", "1", "-ar", str(sr_out), "pipe:1"],
                              input=raw, capture_output=True, check=True)
        w = np.frombuffer(proc.stdout, dtype=np.float32).copy()
        return (w, sr_out) if len(w) else (None, None)
    except Exception:
        return None, None
def _decode_audio(cell):
    raw = cell.get("bytes") if isinstance(cell, dict) else cell
    if raw is None:
        return None, None
    if hasattr(raw, "tobytes"):
        raw = raw.tobytes()
    elif isinstance(raw, (memoryview, bytearray)):
        raw = bytes(raw)
    if raw[:5] == b"UklGR" or (len(raw) > 32 and set(raw[:80]) <= _B64_CHARS):
        try:
            import base64 as _b64
            _d = _b64.b64decode(raw, validate=False)
            if _d[:4] in (b"RIFF", b"fLaC", b"OggS") or _d[:3] == b"ID3":
                raw = _d
        except Exception:
            pass
    try:
        wav, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
    except Exception:
        return _ffmpeg_decode(raw)
    if getattr(wav, "ndim", 1) > 1:
        wav = wav.mean(axis=1)
    return wav, sr

_DECODERS = {l: build_ctcdecoder(LABELS, LM_PATHS[l], unigrams=LM_UNIGRAMS[l],
                                 alpha=LM_BEST[l]["alpha"], beta=LM_BEST[l]["beta"])
             for l in LANGUAGES}
TEST_DIR_TO_MANIFEST = {"swa": "sw", "kik": "kik", "kln": "kln",
                        "luo": "luo", "som": "som", "mas": "mas"}
_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
os.makedirs(_lm_dir, exist_ok=True)
_WIN_S, _OVL_S = 38.0, 6.0                 # fenêtre / recouvrement total entre voisines
_HOP_S = _WIN_S - _OVL_S
_TRIM_S = _OVL_S / 2                       # tronqué de chaque côté intérieur

def _windows(wav, sr):
    """[(segment, tronquer_début, tronquer_fin)] avec recouvrement."""
    n = len(wav)
    if n <= int(39.5 * sr):
        return [(wav, False, False)]
    out, off = [], 0
    while off < n:
        seg = wav[off:off + int(_WIN_S * sr)]
        last = off + int(_WIN_S * sr) >= n
        if len(seg) >= int(0.2 * sr):
            out.append((seg, off > 0, not last))
        if last:
            break
        off += int(_HOP_S * sr)
    return out

def _decode_long(_segs, _omni, _dec, _kw, _sr):
    """Capture chaque fenêtre, tronque les frames de bord, concatène, décode UNE fois."""
    _lp = capture_logprobs([{"waveform": s, "sample_rate": _sr} for s, _, _ in _segs], _omni)
    _parts = []
    for (s, t0, t1), lp in zip(_segs, _lp):
        _fps = len(lp) / (len(s) / float(_sr))
        a = int(round(_TRIM_S * _fps)) if t0 else 0
        b = len(lp) - (int(round(_TRIM_S * _fps)) if t1 else 0)
        _parts.append(lp[a:b])
    _full = _parts[0] if len(_parts) == 1 else np.concatenate(_parts, axis=0)
    _hyp = _dec.decode(_full, **_kw)
    del _lp, _parts, _full
    return _hyp

for _lang, _shard, _n in test_shards:
    _out = os.path.join(_lm_dir, f"{_lang}__{os.path.basename(os.path.dirname(_shard))}__"
                                 f"{os.path.basename(_shard)}.csv")
    if os.path.exists(_out):
        continue
    _t0 = time.time()
    _mcode = TEST_DIR_TO_MANIFEST[_lang]
    _cfg = LM_BEST[_mcode]
    _kw = {"beam_width": int(_cfg.get("beam", 100))}
    if _cfg.get("tml") is not None:
        _kw["token_min_logp"] = float(_cfg["tml"])
    if _cfg.get("bpl") is not None:
        _kw["beam_prune_logp"] = float(_cfg["bpl"])
    _dec = _DECODERS[_mcode]
    _omni = TEST_DIR_TO_OMNI[_lang]
    _tbl = pq.read_table(_shard, columns=["id", "audio"]).to_pandas()
    _short_in, _short_own, _texts = [], [], {}
    for _ri, _row in _tbl.iterrows():
        wav, sr = _decode_audio(_row["audio"])
        if wav is None:
            continue
        if len(wav) < int(0.2 * sr):
            wav = np.pad(wav, (0, int(0.2 * sr) - len(wav)))
        _sw = _windows(wav, sr)
        if len(_sw) == 1:
            _short_in.append({"waveform": _sw[0][0], "sample_rate": sr})
            _short_own.append(_ri)
        else:                                   # long : logits concaténés, décodage unique
            _texts[_ri] = _decode_long(_sw, _omni, _dec, _kw, sr)
    _MINI = 24
    for _s in range(0, len(_short_in), _MINI):
        _lp = capture_logprobs(_short_in[_s:_s + _MINI], _omni)
        for _o, x in zip(_short_own[_s:_s + _MINI], _lp):
            _texts[_o] = _dec.decode(x, **_kw)
        del _lp
    _col = [normalize_text(_texts.get(_ri, "")) for _ri in _tbl.index]
    pd.DataFrame({"id": _tbl["id"], "language": _lang, "text": _col}).to_csv(_out, index=False)
    _done = len(glob.glob(os.path.join(_lm_dir, "*.csv")))
    print(f"[{_done}/{len(test_shards)}] {_lang}/{os.path.basename(_shard)} : "
          f"{len(_tbl)} clips en {(time.time()-_t0)/60:.1f} min")

_parts = [pd.read_csv(f, dtype=str) for f in sorted(glob.glob(os.path.join(_lm_dir, "*.csv")))]
sub = pd.concat(_parts, ignore_index=True)
sub["text"] = sub["text"].fillna("")
_e = sub["text"].str.strip().eq("")
if int(_e.sum()):
    print(f"{int(_e.sum())} prédiction(s) vide(s) -> mot de remplissage")
    sub.loc[_e, "text"] = "a"
sub = sub.rename(columns={"text": SUBMISSION_TEXT_COLUMN})[["id", "language", SUBMISSION_TEXT_COLUMN]]
assert sub["id"].is_unique and len(sub) == TEST_TOTAL_ROWS, (len(sub), TEST_TOTAL_ROWS)
_p = os.path.join(FT_CONFIG["report_dir"], "submission_lm.csv")
sub.to_csv(_p, index=False)
print(f"submission_lm.csv : {len(sub)} lignes -> {_p}")
print(sub.head(6).to_string())

In [ ]:
# 13.1 — SONDE : carte d'asset "omniASR_CTC_1B_v2_ft5000" pointant sur NOTRE checkpoint step_5000.
# Le top-up doit PARTIR du modèle joint fine-tuné (pas du modèle Meta) ; la voie fairseq2 propre
# est une carte héritant de omniASR_CTC_1B_v2 avec un checkpoint local. La vérification tourne en
# SOUS-PROCESSUS (scan des cartes garanti frais, le processus courant peut avoir mis en cache la
# liste) et compare des tenseurs carte-chargée vs fichier checkpoint. ~2-4 min, CPU seulement.
import subprocess, sys, glob
import omnilingual_asr as _oa

_ck = latest_ckpt(FT_CONFIG["output_dir"])
assert _ck, "Aucun checkpoint du run joint : section 8 d'abord."
TOPUP_SEED_CKPT = _ck
if os.path.isdir(_ck):
    _w = [q for q in glob.glob(os.path.join(_ck, "model", "**", "*"), recursive=True)
          if os.path.isfile(q)]
    assert len(_w) == 1, f"Poids multiples/absents sous {_ck}/model : {_w}"
    TOPUP_SEED_CKPT = _w[0]
TOPUP_SEED_CARD = "omniASR_CTC_1B_v2_ft5000"
_card_path = os.path.join(os.path.dirname(_oa.__file__), "cards", "models",
                          "afrivoices_topup.yaml")
os.makedirs(os.path.dirname(_card_path), exist_ok=True)

_PROBE = "/content/probe_topup_card.py"
open(_PROBE, "w").write('''
import sys, torch
ckpt_file, card = sys.argv[1], sys.argv[2]
import omnilingual_asr                                  # enregistre les cartes du package
from fairseq2.models.hub import load_model
try:                                                    # notre propre artefact ; True d'abord
    st = torch.load(ckpt_file, map_location="cpu", weights_only=True)
except Exception:
    st = torch.load(ckpt_file, map_location="cpu", weights_only=False)
sd = st.get("model", st) if isinstance(st, dict) else st
m = load_model(card, device=torch.device("cpu"), dtype=torch.float32)
msd = m.state_dict()
common = [k for k in sd.keys() if k in msd]
assert common, f"Aucune cle commune. ckpt: {list(sd)[:5]} / modele: {list(msd)[:5]}"
bad = 0
for k in (common[0], common[len(common) // 2], common[-1]):
    d = (msd[k].float() - sd[k].float()).abs().max().item()
    print(f"  {k}: ecart max {d:.2e}")
    bad += d > 1e-3
sys.exit(1 if bad else 0)
''')

_ok = False
for _uri in (f"file://{TOPUP_SEED_CKPT}", TOPUP_SEED_CKPT):   # deux formes de chemin local
    with open(_card_path, "w") as f:
        f.write(f'name: {TOPUP_SEED_CARD}\nbase: {FT_CONFIG["model_card"]}\n'
                f'checkpoint: "{_uri}"\n')
    _r = subprocess.run([sys.executable, _PROBE, TOPUP_SEED_CKPT, TOPUP_SEED_CARD],
                        capture_output=True, text=True)
    print(f"— essai checkpoint: {_uri}\n{_r.stdout}")
    if _r.returncode == 0:
        _ok = True
        break
    print(_r.stderr[-1200:])
assert _ok, ("La carte ne charge pas nos poids : me montrer TOUTE la sortie ci-dessus "
             "(plan B possible : copie du checkpoint dans le nouveau output_dir).")
print(f"✅ Carte '{TOPUP_SEED_CARD}' opérationnelle -> {_card_path}")
print(f"   poids = {TOPUP_SEED_CKPT}")

In [ ]:
# 13.2 — TOP-UP mono-langue : parquet {lang} (train+valid), carte dataset repointée, recette
# depuis la carte ft5000 (13.1), sortie Drive topup_{lang}/. kln et mas = pires langues (WER
# valid LM 0.554/0.521) : un top-up SPÉCIALISE le modèle ; chaque langue utilisera son propre
# checkpoint à l'inférence (13.4), donc l'oubli des autres langues est sans importance.
# ~30-40 min de build (FLAC via FUSE) + ~2h30 de train (1000 steps). REPRENABLE : relancer la
# même cellule reprend au dernier checkpoint (et saute le build si le parquet local est là).
TOPUP_LANG = "kln"          # "kln" puis "mas" — vide = cellule inerte (protège « Tout exécuter »)
TOPUP_STEPS = 2000       # checkpoints/validation à 500 et 1000 -> choix du meilleur en 13.3
if not TOPUP_LANG:
    raise SystemExit("TOPUP_LANG vide : mettre 'kln' ou 'mas' puis relancer.")
assert TOPUP_LANG in LANGUAGES, f"Langue inconnue : {TOPUP_LANG}"
assert "TOPUP_SEED_CARD" in globals(), "Exécuter 13.1 d'abord (carte + sonde)."
assert "build_parquet" in globals(), "Exécuter 5.2 d'abord (définitions build_parquet/TSV)."

# LIBÉRER LE GPU DU NOYAU : les pipelines d'inférence (13.3/13.4/12.3) laissent ~8-11 GB sur
# cuda:0, or la recette tourne dans un SOUS-PROCESSUS qui a besoin des 40 GB entiers (OOM sinon).
import gc
for _v in ("_pipe_joint", "_pipe_lm", "_pipe"):
    if _v in globals():
        del globals()[_v]
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.clear()
gc.collect()
torch.cuda.empty_cache()
print(f"GPU noyau libéré : {torch.cuda.memory_allocated() / 2**30:.1f} GB restants alloués")

_t_root = f"/content/topup_{TOPUP_LANG}"
_t_parq = os.path.join(_t_root, "version=0")
_t_tsv = os.path.join(_t_root, "language_distribution_0.tsv")
if os.path.exists(_t_tsv) and (dataset_meta(_t_parq) or {}).get("rows"):
    print("Parquet top-up déjà présent :", _t_parq)
else:
    shutil.rmtree(_t_root, ignore_errors=True)
    _tsv = build_parquet(manifest[manifest.language == TOPUP_LANG].reset_index(drop=True),
                         root=_t_parq)
    assert _tsv == _t_tsv, (_tsv, _t_tsv)
write_dataset_card(data_root=_t_parq)   # la carte GLOBALE pointe maintenant sur le top-up ;
                                        # 2.4 la remet sur le dataset complet à chaque session

_t_out = os.path.join(FT_CONFIG["experiment_run"], f"topup_{TOPUP_LANG}")
_t_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], f"topup_{TOPUP_LANG}_recipe.yaml"),
                            TOPUP_SEED_CARD, _t_tsv, num_steps=TOPUP_STEPS,
                            grad_accum=FT_CONFIG["grad_accum"],
                            max_num_elements=FT_CONFIG["max_num_elements"], ckpt_every=500)
rc, _log = run_recipe(_t_out, _t_yaml, f"topup_{TOPUP_LANG}.log",
                      wall_limit_h=FT_CONFIG["max_wall_hours_per_session"])
print("Code :", rc, "| dernier checkpoint :", latest_ckpt(_t_out))
print("(~12-15 GB par checkpoint sur Drive : vider la corbeille Drive si tu en supprimes.)")
print("Suite : 13.3 (comparer joint vs step_500 vs step_1000 sur valid).")

In [ ]:
# 13.3 — Choisir le checkpoint : joint step_5000 vs top-up step_500 vs step_1000, WER greedy et
# WER+LM (alpha/beta/beam de kenlm_tuning.json) sur 200 clips valid de la langue. ~15-20 min.
# Verdict sauvé dans reports/topup_choice_{lang}.json (lu par 13.4).
import subprocess, json
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
from pathlib import Path
from pyctcdecode import build_ctcdecoder

TOPUP_LANG_EVAL = ""     # vide = reprendre TOPUP_LANG de 13.2
_tl = TOPUP_LANG_EVAL or globals().get("TOPUP_LANG", "")
assert _tl in LANGUAGES, "Préciser TOPUP_LANG_EVAL (ex. 'kln') ou exécuter 13.2 d'abord."

def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""            # blank CTC
                return L
            except Exception:
                pass
    raise SystemExit("Vocab SentencePiece inaccessible (voir pipe.tokenizer._model).")

def _capture(pipe, inputs, omni_code):
    """(logprobs, hypothèses greedy) — spy sur le forward, batch_size=1 = ordre garanti."""
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        hyps = pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps, hyps

_tune = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
_a, _b = float(_tune[_tl]["alpha"]), float(_tune[_tl]["beta"])
_bw = int(_tune[_tl].get("beam", 100))
_lm_bin = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models", f"{_tl}.binary")
assert os.path.exists(_lm_bin), f"LM manquant : {_lm_bin}"
_uni = set()
for t in manifest[(manifest.split == "train") & (manifest.language == _tl)]["text_norm"]:
    _uni.update(str(t).split())
_uni = sorted(_uni)

_t_out = os.path.join(FT_CONFIG["experiment_run"], f"topup_{_tl}")
_steps = sorted((d for d in Path(_t_out).rglob("step_*")
                 if d.is_dir() and not d.name.endswith(".tmp") and (d / "model").exists()),
                key=lambda d: int(d.name.split("_")[1]))
assert _steps, f"Aucun checkpoint top-up sous {_t_out} : exécuter 13.2 d'abord."
# un changement de num_steps crée un NOUVEAU workspace ws_* (hash de config) : des step_N
# homonymes peuvent coexister entre l'ancien et le nouveau -> désambiguïser par le hash
_names = [d.name for d in _steps]
def _lbl(d):
    return d.name if _names.count(d.name) == 1 \
        else f"{d.name}@{d.parent.parent.name.split('.')[-1]}"
_cands = [("joint", latest_ckpt(FT_CONFIG["output_dir"]))] + [(_lbl(d), str(d)) for d in _steps]

_val = (manifest[(manifest.split == "valid") & (manifest.language == _tl)]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
refs = _val["text_norm"].tolist()
_res = {}
for _name, _ckpt in _cands:
    _pipe = load_finetuned_pipeline(_ckpt)
    L = _labels_from(_pipe)
    caps, hyps = _capture(_pipe, _val["curated_audio_path"].tolist(), omni_lang(_tl))
    _dec = build_ctcdecoder(L, _lm_bin, unigrams=_uni, alpha=_a, beta=_b)
    wg = jiwer.wer(refs, [normalize_text(h) for h in hyps])
    wl = jiwer.wer(refs, [normalize_text(_dec.decode(x, beam_width=_bw)) for x in caps])
    _res[_name] = {"ckpt": _ckpt, "wer_greedy": round(wg, 4), "wer_lm": round(wl, 4)}
    print(f"{_tl}/{_name}: greedy {wg:.4f} | +LM {wl:.4f}")
    del _pipe, caps
    torch.cuda.empty_cache()

_best = min(_res, key=lambda n: _res[n]["wer_lm"])
save_json({"language": _tl, "chosen": _best, **_res[_best], "all": _res},
          os.path.join(FT_CONFIG["report_dir"], f"topup_choice_{_tl}.json"))
if _best == "joint":
    print(f"⚠️ Le top-up ne bat PAS le joint sur valid ({_tl}) : NE PAS faire 13.4. "
          "Me montrer le tableau ci-dessus.")
else:
    _d = _res["joint"]["wer_lm"] - _res[_best]["wer_lm"]
    print(f"✅ {_best} retenu pour {_tl} : WER LM {_res[_best]['wer_lm']} "
          f"(joint {_res['joint']['wer_lm']}, gain {_d:+.4f}) -> exécuter 13.4.")

In [ ]:
TOPUP_LANG_EVAL = "mas"

In [ ]:
# 13.4 — Brancher le checkpoint top-up choisi (13.3) pour la ré-inférence test, avec ROUTAGE PAR
# LANGUE : capture_logprobs sert le pipeline top-up pour les langues topées (TOPUP_PIPES,
# cumulatif : kln puis mas) et le pipeline JOINT pour toutes les autres — indispensable car des
# CSV d'autres langues peuvent manquer (ex. kik/som invalidés par 12.2b) et seront régénérés par
# le même 12.3. Fournit aussi LABELS/LM_BEST/LM_PATHS/LM_UNIGRAMS -> RELANCER 12.3 ensuite.
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
from pyctcdecode import build_ctcdecoder   # requis par 12.3 (_DECODERS)

_tl = globals().get("TOPUP_LANG_EVAL", "") or globals().get("TOPUP_LANG", "")
assert _tl in LANGUAGES, "Exécuter 13.2/13.3 d'abord (TOPUP_LANG)."
assert "_capture" in globals() and "_labels_from" in globals(), "Exécuter 13.3 d'abord."
_choice_p = os.path.join(FT_CONFIG["report_dir"], f"topup_choice_{_tl}.json")
assert os.path.exists(_choice_p), "Exécuter 13.3 d'abord (topup_choice manquant)."
_choice = json.load(open(_choice_p))
assert _choice["chosen"] != "joint", \
    "13.3 a retenu le JOINT : rien à ré-inférer pour cette langue, ne pas relancer 12.3."

# pipelines : joint (par défaut) + top-ups par code omnilingual (~2 GB pièce en bf16, cumul OK)
if "_pipe_joint" not in globals():
    _pipe_joint = load_finetuned_pipeline(latest_ckpt(FT_CONFIG["output_dir"]))
if "TOPUP_PIPES" not in globals():
    TOPUP_PIPES = {}
TOPUP_PIPES[omni_lang(_tl)] = load_finetuned_pipeline(_choice["ckpt"])
# resynchroniser TOUTES les langues topées (13.2 vide TOPUP_PIPES pour libérer le GPU :
# sans cette boucle, les autres langues retomberaient silencieusement sur le joint)
for _f in glob.glob(os.path.join(FT_CONFIG["report_dir"], "topup_choice_*.json")):
    _c2 = json.load(open(_f))
    if _c2["chosen"] != "joint" and omni_lang(_c2["language"]) not in TOPUP_PIPES:
        TOPUP_PIPES[omni_lang(_c2["language"])] = load_finetuned_pipeline(_c2["ckpt"])
LABELS = _labels_from(_pipe_joint)         # même tokenizer pour tous les checkpoints
def capture_logprobs(inputs, omni_code):
    _r = _capture(TOPUP_PIPES.get(omni_code, _pipe_joint), inputs, omni_code)
    return _r[0] if isinstance(_r, tuple) else _r   # _capture de 13.3 (tuple) OU de 16.3 (liste)

LM_BEST = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
LM_DRIVE = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models")
# après le polish 12.2c, kenlm_tuning.json porte le binaire choisi (v1/v2) par langue
LM_PATHS = {l: LM_BEST.get(l, {}).get("lm_bin") or os.path.join(LM_DRIVE, f"{l}.binary")
            for l in LANGUAGES}
assert all(os.path.exists(p) for p in LM_PATHS.values()), "LMs manquants sur Drive."
LM_UNIGRAMS = {}
for l in LANGUAGES:
    _uf = LM_BEST.get(l, {}).get("uni_file")    # v4 : fichier d'unigrammes dédié (14.5/14.6)
    if _uf and os.path.exists(_uf):
        LM_UNIGRAMS[l] = open(_uf, encoding="utf-8").read().splitlines()
        continue
    _tag = LM_BEST.get(l, {}).get("lm", "v1")   # v1=train, v3=train+valid, web=fichier dédié
    if _tag == "web":
        _p = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_web", f"{l}_unigrams.txt")
        LM_UNIGRAMS[l] = open(_p, encoding="utf-8").read().splitlines()
        continue
    _rows = (manifest if _tag == "v2"
             else manifest[manifest.split.isin(["train", "valid"])] if _tag == "v3"
             else manifest[manifest.split == "train"])
    _w = set()
    for t in _rows[_rows.language == l]["text_norm"]:
        _w.update(str(t).split())
    LM_UNIGRAMS[l] = sorted(_w)

_dir_code = {"sw": "swa"}.get(_tl, _tl)                 # codes des dossiers test
_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_n = 0
for f in glob.glob(os.path.join(_lm_dir, f"{_dir_code}__*.csv")):
    os.remove(f)
    _n += 1
print(f"{_n} CSV {_dir_code} invalidés | routage : "
      + ", ".join(f"{c}->top-up" for c in TOPUP_PIPES) + ", autres->joint")
print(f"   {_tl}: {_choice['chosen']} (WER valid LM {_choice['wer_lm']})")
print("RELANCER 12.3 maintenant : elle régénère TOUTES les langues sans CSV, chacune avec le bon "
      "modèle, puis assemble submission_lm.csv.")

In [ ]:
# 14.1 — A/B swa : le modèle DE BASE (Meta, non fine-tuné) bat-il notre fine-tuné sur le test ?
# Notre swa d'entraînement = 100 % agriculture SCRIPTÉE (Afrivoice) ; le test swa = 100 %
# SPONTANÉ. Le fine-tuning a pu dégrader le swahili spontané que le pré-entraîné (tier H,
# données massives) maîtrisait. Aucune donnée interne ne peut le mesurer -> A/B par SOUMISSION :
# on régénère SEULEMENT swa avec le modèle de base (mêmes LM/alpha/beta), on soumet, on compare.
import shutil, glob
assert "TOPUP_PIPES" in globals() and "capture_logprobs" in globals(), "Exécuter 12.2c d'abord."

_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_bak = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm_swa_ft")
os.makedirs(_bak, exist_ok=True)
_moved = 0
for f in glob.glob(os.path.join(_lm_dir, "swa__*.csv")):
    shutil.move(f, os.path.join(_bak, os.path.basename(f)))   # prédictions FT mises de côté
    _moved += 1
assert _moved or glob.glob(os.path.join(_bak, "swa__*.csv")), \
    "Aucun CSV swa à mettre de côté : exécuter 12.3 (v7) d'abord."
print(f"{_moved} CSV swa (fine-tuné) sauvegardés -> {_bak}")

_pipe_base = load_finetuned_pipeline(None)     # carte officielle = poids Meta d'origine
TOPUP_PIPES["swh_Latn"] = _pipe_base           # swa -> BASE ; les autres langues inchangées
print("swa routé vers le modèle DE BASE (les LM/réglages ne changent pas).")
print("RELANCER 12.3 (26 shards swa, ~50 min) -> soumettre -> comparer au score v7 :")
print("  si MEILLEUR : l'oubli de domaine est confirmé, on garde et on creuse ;")
print("  si PIRE : exécuter 14.2 pour restaurer les prédictions fine-tunées.")

In [ ]:
# 14.2 — (SEULEMENT si l'A/B 14.1 est perdant) restaurer les prédictions swa fine-tunées
import shutil, glob
_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_bak = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm_swa_ft")
for f in glob.glob(os.path.join(_lm_dir, "swa__*.csv")):
    os.remove(f)
_n = 0
for f in glob.glob(os.path.join(_bak, "swa__*.csv")):
    shutil.copy(f, os.path.join(_lm_dir, os.path.basename(f)))
    _n += 1
assert _n, f"Aucune sauvegarde sous {_bak} ?"
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.pop("swh_Latn", None)          # swa re-routé vers le joint
print(f"{_n} CSV swa fine-tunés restaurés ; relancer 12.3 (tout est présent : "
      "elle ne fera que réassembler submission_lm.csv).")

In [ ]:
# 14.3 — LM swa « web » : Wikipedia swahili (donnée externe PUBLIQUE et gratuite, autorisée par
# les règles « External Tools & Data ») + notre texte train+valid. Notre LM swa ne connaît que
# l'agriculture scriptée ; le test swa est 100 % spontané et généraliste -> un LM à large
# couverture guide mieux le beam là où l'acoustique hésite. ~15-25 min.
import subprocess
import re as _re
subprocess.run(["pip", "install", "-q", "datasets"], check=False)
from datasets import load_dataset

_lmplz = "/content/kenlm/build/bin/lmplz"
_bbin = "/content/kenlm/build/bin/build_binary"
assert os.path.exists(_lmplz), "KenLM non compilé : exécuter 12.1b d'abord."
LM_WEB_DIR = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_web")
os.makedirs(LM_WEB_DIR, exist_ok=True)
LM_WEB_SW = os.path.join(LM_WEB_DIR, "sw.binary")
UNI_WEB_SW = os.path.join(LM_WEB_DIR, "sw_unigrams.txt")

if os.path.exists(LM_WEB_SW) and os.path.exists(UNI_WEB_SW):
    print("LM web swa déjà présent :", LM_WEB_SW)
else:
    print("Téléchargement Wikipedia swahili…")
    _wiki = load_dataset("wikimedia/wikipedia", "20231101.sw", split="train")
    _sents = [str(t) for t in manifest[manifest.split.isin(["train", "valid"])
                                       & (manifest.language == "sw")]["text_norm"]]
    _seen = set()
    for _a in _wiki["text"]:
        for _s in _re.split(r"[.!?\n]+", _a):
            _s = normalize_text(_s)
            if len(_s.split()) >= 3 and _s not in _seen:
                _seen.add(_s)
                _sents.append(_s)
    print(f"{len(_sents)} phrases (dont {len(_seen)} de Wikipedia)")
    _txt, _arpa = "/content/lm_web_sw.txt", "/content/lm_web_sw.arpa"
    with open(_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(_sents))
    with open(_txt, "rb") as fi, open(_arpa, "wb") as fo:
        _r = subprocess.run([_lmplz, "-o", "5", "--discount_fallback", "-S", "30%"],
                            stdin=fi, stdout=fo, stderr=subprocess.PIPE)
    assert _r.returncode == 0, _r.stderr[-600:].decode(errors="replace")
    subprocess.run([_bbin, _arpa, LM_WEB_SW], check=True)
    from collections import Counter
    _cnt = Counter(w for s in _sents for w in s.split())
    with open(UNI_WEB_SW, "w", encoding="utf-8") as f:      # 200k mots les plus fréquents
        f.write("\n".join(w for w, _ in _cnt.most_common(200_000)))
    print(f"LM web swa -> {LM_WEB_SW} ({os.path.getsize(LM_WEB_SW)//2**20} Mo), "
          f"vocab {min(len(_cnt), 200_000)} mots")
print("Suite : 14.4 (contrôle + bascule des réglages swa).")

In [ ]:
# 14.4 — Brancher le LM web sur swa : contrôle sur local_test (PROXY imparfait : il est scripté/
# agricole, le vrai juge du spontané est le LEADERBOARD), puis bascule des réglages swa.
# Critère : si le LM web tient à <1 pt du LM actuel sur le proxy, il mérite l'A/B en soumission
# (son avantage attendu est précisément hors domaine, invisible sur le proxy).
import json, glob, jiwer
from pyctcdecode import build_ctcdecoder
assert "LM_WEB_SW" in globals() and os.path.exists(LM_WEB_SW), "Exécuter 14.3 d'abord."
assert "capture_logprobs" in globals() and "LABELS" in globals(), "Exécuter 12.2c d'abord."

_uni_web = open(UNI_WEB_SW, encoding="utf-8").read().splitlines()
_val = (manifest[(manifest.split == "local_test") & (manifest.language == "sw")]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
refs = _val["text_norm"].tolist()
lp = capture_logprobs(_val["curated_audio_path"].tolist(), omni_lang("sw"))

_tun_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
_tun = json.load(open(_tun_p))
_cur = _tun["sw"]
_dec0 = build_ctcdecoder(LABELS, _cur["lm_bin"], unigrams=LM_UNIGRAMS["sw"],
                         alpha=float(_cur["alpha"]), beta=float(_cur["beta"]))
w_cur = jiwer.wer(refs, [normalize_text(_dec0.decode(x)) for x in lp])
print(f"swa ACTUEL ({_cur['lm']}, a={_cur['alpha']}, b={_cur['beta']}) : "
      f"WER local_test {w_cur:.4f}")
best = None
for a in (0.3, 0.5, 0.65):
    for b in (0.0, 1.0):
        _d = build_ctcdecoder(LABELS, LM_WEB_SW, unigrams=_uni_web, alpha=a, beta=b)
        w = jiwer.wer(refs, [normalize_text(_d.decode(x)) for x in lp])
        if best is None or w < best[0]:
            best = (w, a, b)
print(f"swa LM WEB : meilleur {best[0]:.4f} (a={best[1]}, b={best[2]})")

if best[0] <= w_cur + 0.01:
    _tun["sw"] = {"alpha": best[1], "beta": best[2], "beam": 100, "lm": "web",
                  "lm_bin": LM_WEB_SW, "wer_best": round(best[0], 4),
                  "wer_avant_polish": _cur.get("wer_best")}
    save_json(_tun, _tun_p)
    LM_BEST["sw"] = _tun["sw"]
    LM_PATHS["sw"] = LM_WEB_SW
    LM_UNIGRAMS["sw"] = _uni_web
    _lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
    _n = 0
    for f in glob.glob(os.path.join(_lm_dir, "swa__*.csv")):
        os.remove(f)
        _n += 1
    print(f"{_n} CSV swa invalidés -> RELANCER 12.3 puis SOUMETTRE (A/B leaderboard).")
    print("(Si le score se dégrade : remettre _tun['sw'] depuis kenlm_tuning_pre_polish/12.2c "
          "n'est PAS automatique — me prévenir, je te donne la restauration.)")
else:
    print("LM web nettement pire même sur le proxy scripté : ne pas déployer, me montrer ces chiffres.")

In [ ]:
# 14.5 — Corpus externes PUBLICS + LMs v4 pour sw/kik/kln/luo/som (deep-research 2026-07-06).
# Sources (autorisées : publiques/gratuites) : thinkKenya kln+luo (~35k/29k phrases, CC-BY-4.0),
# Bibles eBible kik/luo/som (~700k mots), Leipzig som (newscrawl 100K), Wikipedia sw (14.3).
# Recette : in-domain TRAIN dupliqué x4 + externe dédupliqué ; AUCUN texte de valid -> le
# réglage 14.6 se fait sur VALID (le proxy prouvé), sans fuite. mas : pas de source fiable.
import subprocess, zipfile, tarfile, json, glob
import io as _io
import re as _re
subprocess.run(["pip", "install", "-q", "datasets", "requests"], check=False)
import requests
from datasets import load_dataset

# Authentification HF (thinkKenya = dataset gated ; conditions à accepter UNE fois sur sa page
# web) : token lu depuis hf_token.json sur le Drive ({"name":"HF_TOKEN","key":"hf_..."}).
try:
    _tk = ["/content/afrivoices_project", os.path.join(DRIVE_ROOT, "hf_token.json")]
    _tp = next((p for p in _tk if os.path.exists(p)), None)
    if _tp is None:
        _hits = glob.glob("/content/persistent_storage/*/hf_token.json")
        _tp = _hits[0] if _hits else None
    if _tp:
        from huggingface_hub import login
        login(token=json.load(open(_tp))["key"])
        print("HF authentifié via", _tp)
    else:
        print("⚠️ hf_token.json introuvable sur le Drive : thinkKenya (gated) échouera.")
except Exception as _e:
    print(f"⚠️ login HF : {_e!r} — thinkKenya échouera probablement.")

_lmplz = "/content/kenlm/build/bin/lmplz"
_bbin = "/content/kenlm/build/bin/build_binary"
assert os.path.exists(_lmplz), "KenLM non compilé : exécuter 12.1b d'abord."
LM_V4_DIR = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_v4")
os.makedirs(LM_V4_DIR, exist_ok=True)
V4_LANGS = ["sw", "kik", "kln", "luo", "som"]

def _clean_ext(s):
    s = _re.sub(r"\d+", " ", str(s))          # les chiffres n'existent pas dans nos réfs ASR
    s = normalize_text(s)
    return s if len(s.split()) >= 3 else ""

_ext = {l: [] for l in V4_LANGS}

# 1) thinkKenya — paires parallèles kalenjin/dholuo <-> kiswahili
def _pick_text(row, cols, keys):
    for k in keys:
        if k in cols and isinstance(row.get(k), str):
            return row[k]
    for c in cols:
        v = row.get(c)
        if isinstance(v, dict):
            for k in keys:
                if isinstance(v.get(k), str):
                    return v[k]
    return None

for _lang, _cfg, _keys in (("kln", "kln_swa", ("kln", "kalenjin", "kal")),
                           ("luo", "luo_swa", ("luo", "dholuo"))):
    try:
        _ds = load_dataset("thinkKenya/kenyan-low-resource-language-data", _cfg)
        _n0 = len(_ext[_lang])
        for _split in _ds:
            _cols = _ds[_split].column_names
            for _row in _ds[_split]:
                _t = _pick_text(_row, _cols, _keys)
                if _t is None:
                    raise SystemExit(f"thinkKenya {_cfg}: colonne {_lang} introuvable dans "
                                     f"{_cols} — me montrer ce message et une ligne d'exemple.")
                _s = _clean_ext(_t)
                if _s:
                    _ext[_lang].append(_s)
        print(f"  thinkKenya {_lang}: {len(_ext[_lang]) - _n0} phrases")
    except SystemExit:
        raise
    except Exception as e:
        print(f"⚠️ thinkKenya {_cfg} : {e!r} (non bloquant, me le signaler)")

# 2) eBible — Bibles complètes redistributables (kik/luo/som)
for _lang, _tid in (("kik", "kik"), ("luo", "luo"), ("som", "som")):
    try:
        _r = requests.get(f"https://ebible.org/Scriptures/{_tid}_readaloud.zip", timeout=180)
        _r.raise_for_status()
        _z = zipfile.ZipFile(_io.BytesIO(_r.content))
        _n0 = len(_ext[_lang])
        for _nm in _z.namelist():
            if not _nm.lower().endswith(".txt"):
                continue
            for _line in _z.read(_nm).decode("utf-8", errors="replace").splitlines():
                _s = _clean_ext(_line)
                if _s:
                    _ext[_lang].append(_s)
        print(f"  eBible {_lang}: {len(_ext[_lang]) - _n0} lignes")
    except Exception as e:
        print(f"⚠️ eBible {_lang} : {e!r} (non bloquant, me le signaler)")

# 3) Leipzig — somali newscrawl 100K
try:
    _r = requests.get("https://downloads.wortschatz-leipzig.de/corpora/"
                      "som_newscrawl_2011_100K.tar.gz", timeout=300)
    _r.raise_for_status()
    _t = tarfile.open(fileobj=_io.BytesIO(_r.content), mode="r:gz")
    _n0 = len(_ext["som"])
    for _m in _t.getmembers():
        if _m.name.endswith("-sentences.txt"):
            for _line in _t.extractfile(_m).read().decode("utf-8", errors="replace").splitlines():
                _s = _clean_ext(_line.split("\t", 1)[-1])
                if _s:
                    _ext["som"].append(_s)
    print(f"  Leipzig som: {len(_ext['som']) - _n0} phrases")
except Exception as e:
    print(f"⚠️ Leipzig som : {e!r} (non bloquant)")

# 4) Wikipedia sw (SANS texte valid, contrairement au LM « web » de 14.3)
try:
    _wiki = load_dataset("wikimedia/wikipedia", "20231101.sw", split="train")
    for _a in _wiki["text"]:
        for _s0 in _re.split(r"[.!?\n]+", _a):
            _s = _clean_ext(_s0)
            if _s:
                _ext["sw"].append(_s)
    print(f"  Wikipedia sw: {len(_ext['sw'])} phrases")
except Exception as e:
    print(f"⚠️ Wikipedia sw : {e!r} (non bloquant)")

# 5) Construction des LMs v4 : train x4 + externe dédupliqué
from collections import Counter
LM_PATHS_V4, UNI_V4 = {}, {}
for lang in V4_LANGS:
    _extl = sorted(set(_ext[lang]))
    if not _extl:
        print(f"⚠️ {lang}: aucun texte externe -> v4 sautée pour cette langue")
        continue
    _bin = os.path.join(LM_V4_DIR, f"{lang}.binary")
    _unif = os.path.join(LM_V4_DIR, f"{lang}_unigrams.txt")
    _ind = [str(t) for t in manifest[(manifest.split == "train")
                                     & (manifest.language == lang)]["text_norm"]]
    # reconstruction si le binaire manque OU si le corpus externe a changé (ex. thinkKenya
    # débloqué après un premier passage sans authentification)
    _metaf = os.path.join(LM_V4_DIR, f"{lang}_meta.json")
    _old_n = json.load(open(_metaf)).get("externes", -1) if os.path.exists(_metaf) else -1
    if not (os.path.exists(_bin) and os.path.exists(_unif)) or _old_n != len(_extl):
        if _old_n >= 0 and _old_n != len(_extl):
            print(f"  {lang}: corpus externe {_old_n} -> {len(_extl)} phrases : reconstruction")
        _sents = _ind * 4 + _extl
        _txt, _arpa = f"/content/lm4_{lang}.txt", f"/content/lm4_{lang}.arpa"
        with open(_txt, "w", encoding="utf-8") as f:
            f.write("\n".join(_sents))
        def _run(order):
            with open(_txt, "rb") as fi, open(_arpa, "wb") as fo:
                return subprocess.run([_lmplz, "-o", str(order), "--discount_fallback",
                                       "-S", "30%"], stdin=fi, stdout=fo,
                                      stderr=subprocess.PIPE)
        _r = _run(5)
        if _r.returncode != 0:
            _r = _run(3)
        assert _r.returncode == 0, f"lmplz {lang}: {_r.stderr[-600:].decode(errors='replace')}"
        subprocess.run([_bbin, _arpa, _bin], check=True)
        _cnt = Counter(w for s in (_ind + _extl) for w in s.split())
        with open(_unif, "w", encoding="utf-8") as f:
            f.write("\n".join(w for w, _ in _cnt.most_common(200_000)))
        with open(_metaf, "w") as f:
            json.dump({"externes": len(_extl)}, f)
    LM_PATHS_V4[lang] = _bin
    UNI_V4[lang] = _unif
    print(f"  {lang}: LM v4 = {len(_ind)} in-domain x4 + {len(_extl)} externes "
          f"-> {os.path.getsize(_bin) // 2**20} Mo")
print("LMs v4 prêts -> 14.6 (arbitrage sur valid + seuils pyctcdecode).")

In [ ]:
# 14.6 — ARBITRE v4 : par langue (sw/kik/kln/luo/som), candidat A = config PRÉ-POLISH prouvée
# (v1, celle de v6=0.38201) vs candidat B = LM v4 (grille alpha/beta) ; puis micro-grille des
# SEUILS pyctcdecode (token_min_logp/beam_prune_logp/beam — jamais réglés, défauts
# calibrés pour l'anglais) sur le vainqueur. Réglage sur VALID (aucun corpus v4 n'en contient).
# Écrit kenlm_tuning.json + invalide les CSV -> RELANCER 12.3 (version mise à jour !) puis
# SOUMETTRE. mas reste inchangée (config identique depuis v6).
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
from pyctcdecode import build_ctcdecoder

assert "LM_PATHS_V4" in globals() and LM_PATHS_V4, "Exécuter 14.5 d'abord."
assert "capture_logprobs" in globals() and "LABELS" in globals(), "Exécuter 12.2d d'abord."

_prev_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_pre_polish.json")
_base = json.load(open(_prev_p if os.path.exists(_prev_p)
                       else os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
_tun_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
_tun = json.load(open(_tun_p))
_V1_BIN = {l: os.path.join(FT_CONFIG["experiment_run"], "kenlm_models", f"{l}.binary")
           for l in LANGUAGES}
_uniA = {}                                     # unigrammes v1 : train seul
for l in LANGUAGES:
    _w = set()
    for t in manifest[(manifest.split == "train") & (manifest.language == l)]["text_norm"]:
        _w.update(str(t).split())
    _uniA[l] = sorted(_w)

ALPHAS = [0.3, 0.5, 0.65, 0.8]
BETAS = [0.0, 1.0, 2.0]
KNOBS = ((-8.0, -10.0, 100), (-12.0, -15.0, 100), (-8.0, -10.0, 250))
_val = (manifest[manifest.split == "valid"].sample(frac=1.0, random_state=FT_CONFIG["seed"])
        .groupby("language").head(200))
_changed = []
for lang in [l for l in ("sw", "kik", "kln", "luo", "som") if l in LM_PATHS_V4]:
    _t0 = time.time()
    sub = _val[_val.language == lang]
    refs = sub["text_norm"].tolist()
    lp = capture_logprobs(sub["curated_audio_path"].tolist(), omni_lang(lang))
    _uniB = open(UNI_V4[lang], encoding="utf-8").read().splitlines()

    # A — config pré-polish (v1)
    _pp = _base[lang]
    _decA = build_ctcdecoder(LABELS, _V1_BIN[lang], unigrams=_uniA[lang],
                             alpha=float(_pp["alpha"]), beta=float(_pp["beta"]))
    wA = jiwer.wer(refs, [normalize_text(_decA.decode(
        x, beam_width=int(_pp.get("beam", 100)))) for x in lp])

    # B — LM v4, grille alpha/beta
    bestB = None
    for a in ALPHAS:
        for b in BETAS:
            _d = build_ctcdecoder(LABELS, LM_PATHS_V4[lang], unigrams=_uniB, alpha=a, beta=b)
            w = jiwer.wer(refs, [normalize_text(_d.decode(x)) for x in lp])
            if bestB is None or w < bestB[0]:
                bestB = (w, a, b)

    if bestB[0] < wA - 0.002:
        _win = {"alpha": bestB[1], "beta": bestB[2], "beam": 100, "lm": "v4",
                "lm_bin": LM_PATHS_V4[lang], "uni_file": UNI_V4[lang],
                "wer_best": round(bestB[0], 4)}
        _dec = build_ctcdecoder(LABELS, LM_PATHS_V4[lang], unigrams=_uniB,
                                alpha=bestB[1], beta=bestB[2])
        _wwin = bestB[0]
    else:
        _win = {"alpha": float(_pp["alpha"]), "beta": float(_pp["beta"]),
                "beam": int(_pp.get("beam", 100)), "lm": "v1", "lm_bin": _V1_BIN[lang],
                "wer_best": round(wA, 4)}
        _dec = _decA
        _wwin = wA

    # seuils pyctcdecode sur le vainqueur
    for tml, bpl, bw in KNOBS:
        w = jiwer.wer(refs, [normalize_text(_dec.decode(
            x, beam_width=max(bw, _win["beam"]), token_min_logp=tml,
            beam_prune_logp=bpl)) for x in lp])
        if w < _wwin - 0.002:
            _wwin = w
            _win.update({"tml": tml, "bpl": bpl, "beam": max(bw, _win["beam"]),
                         "wer_best": round(w, 4)})

    _tun[lang] = _win
    _changed.append(lang)
    print(f"{lang}: pré-polish(v1) {wA:.4f} | v4 {bestB[0]:.4f} (a={bestB[1]},b={bestB[2]}) "
          f"-> retenu {_win['lm']} WER {_win['wer_best']}"
          f"{' + seuils ' + str((_win.get('tml'), _win.get('bpl'), _win['beam'])) if 'tml' in _win else ''}"
          f"  [{(time.time() - _t0) / 60:.1f} min]")
    del lp

save_json(_tun, _tun_p)
LM_BEST = _tun
LM_PATHS = {l: LM_BEST[l].get("lm_bin") or _V1_BIN[l] for l in LANGUAGES}
LM_UNIGRAMS = {}
for l in LANGUAGES:
    _uf = LM_BEST.get(l, {}).get("uni_file")
    if _uf and os.path.exists(_uf):
        LM_UNIGRAMS[l] = open(_uf, encoding="utf-8").read().splitlines()
    elif LM_BEST.get(l, {}).get("lm") == "v3":
        _w = set()
        for t in manifest[manifest.split.isin(["train", "valid"])
                          & (manifest.language == l)]["text_norm"]:
            _w.update(str(t).split())
        LM_UNIGRAMS[l] = sorted(_w)
    else:
        LM_UNIGRAMS[l] = _uniA[l]

_lm_dir = os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm")
_MAN2DIR = {"sw": "swa"}
_n = 0
for l in _changed:
    for f in glob.glob(os.path.join(_lm_dir, f"{_MAN2DIR.get(l, l)}__*.csv")):
        os.remove(f)
        _n += 1
print(f"\n{_n} CSV invalidés ({_changed}). RELANCER 12.3 (VERSION MISE À JOUR — elle lit les "
      "seuils) puis SOUMETTRE. mas inchangée.")

In [ ]:
# TEMP — restaurer kln à la config pré-polish (v6) + invalider ses CSV
# (kln n'a pas de LM v4 : 14.6 ne l'a pas traitée, elle resterait sinon sur la config
# du polish v7 qui avait régressé)
import json, glob
_prev = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning_pre_polish.json")))
_tun = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
_tun["kln"] = _prev["kln"]
save_json(_tun, os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json"))

LM_BEST["kln"] = _prev["kln"]
LM_PATHS["kln"] = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models", "kln.binary")
_w = set()
for t in manifest[(manifest.split == "train") & (manifest.language == "kln")]["text_norm"]:
    _w.update(str(t).split())
LM_UNIGRAMS["kln"] = sorted(_w)

_n = 0
for f in glob.glob(os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm", "kln__*.csv")):
    os.remove(f)
    _n += 1
print(f"kln -> config pré-polish (a={_tun['kln']['alpha']}, b={_tun['kln']['beta']}, "
      f"beam={_tun['kln'].get('beam', 100)}) ; {_n} CSV kln invalidés")

In [ ]:
# 16.1 — swa SPONTANÉ : construire un dataset fairseq2 de ~120 h depuis Afrivoice_Swahili
# (domaines health/education/financial/government — l'agriculture est déjà dans le joint).
# Par archive : manifest_N.jsonl <-> audio_N.tar.xz (~25-30 h). Sélection par durée depuis les
# manifests SEULS, puis téléchargement ciblé, décodage webm->FLAC 16k mono (ffmpeg, 12 threads),
# parquet fairseq2 direct (train + dev ~3 h disjoint par locuteur). Vérifie la DISJONCTION avec
# les IDs du test Kaggle (règle 4b). REPRENABLE par archive ; cache Drive final. ~45-60 min.
import subprocess, tarfile, json, glob, shutil
import io as _io
import re as _re
import pyarrow as pa
import pyarrow.parquet as _pq
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download, login

SWA_TARGET_HOURS = 120
SWA_DEV_HOURS = 3
_DOMAINS = ["health", "education", "financial", "government"]
_REPO = "DigitalUmuganda/Afrivoice_Swahili"

_tp = next((p for p in ("/content/afrivoices_project",
                        os.path.join(DRIVE_ROOT, "hf_token.json")) if os.path.exists(p)), None)
assert _tp, "hf_token.json introuvable"
login(token=json.load(open(_tp))["key"])

SPONT_ROOT = "/content/topup_swa_spont"
SPONT_PARQ = os.path.join(SPONT_ROOT, "version=0")
SPONT_DRIVE = os.path.join(DRIVE_ROOT, "topup_swa_spont_cache")
_marker = os.path.join(SPONT_DRIVE, "_COMPLETE")
if os.path.exists(_marker) and not os.path.exists(os.path.join(SPONT_ROOT, "_COMPLETE")):
    print("Cache Drive trouvé : copie locale…")
    shutil.rmtree(SPONT_ROOT, ignore_errors=True)
    subprocess.run(["cp", "-r", SPONT_DRIVE, SPONT_ROOT], check=True)

if not os.path.exists(os.path.join(SPONT_ROOT, "_COMPLETE")):
    # IDs du test Kaggle (préuve de disjonction, règle 4b)
    _test_ids = set()
    for _l, _sh, _n in test_shards:
        if _l == "swa":
            for _i in _pq.read_table(_sh, columns=["id"])["id"].to_pylist():
                _test_ids.add(str(_i).replace(".wav", "").replace(".webm", ""))
    print(f"{len(_test_ids)} IDs test swa chargés (contrôle de disjonction)")

    # Phase A — sélection par manifests (round-robin sur les domaines)
    _sel, _hours = [], 0.0
    for _chunk in range(0, 20):
        for _dom in _DOMAINS:
            if _hours >= SWA_TARGET_HOURS + SWA_DEV_HOURS:
                break
            _dtrain = f"{_dom}_swahili_train"
            try:
                _mf = hf_hub_download(_REPO, f"{_dtrain}/manifest_{_chunk}.jsonl",
                                      repo_type="dataset")
            except Exception:
                continue
            _rows = []
            for _line in open(_mf, encoding="utf-8"):
                try:
                    r = json.loads(_line)
                except Exception:
                    continue
                _txt = _re.sub(r"\[cs\]", " ", str(r.get("normalized_transcription")
                                                   or r.get("transcription") or ""))
                _txt = normalize_text(_txt)
                _dur = float(r.get("duration") or 0)
                if _txt and len(_txt.split()) >= 2 and 0.5 <= _dur <= 38.0:
                    _rows.append({"key": r["key"], "file": r["audio_filepath"],
                                  "spk": r.get("voice_creator_id") or r.get("uid") or "?",
                                  "text": _txt, "dur": _dur})
            if not _rows:
                continue
            assert not (_test_ids & {r["key"] for r in _rows}), \
                f"CHEVAUCHEMENT avec le test détecté dans {_dtrain}/chunk{_chunk} — STOP (règle 4b)"
            _h = sum(r["dur"] for r in _rows) / 3600
            _sel.append((_dtrain, _chunk, _rows, _h))
            _hours += _h
            print(f"  + {_dtrain}/chunk{_chunk}: {len(_rows)} clips, {_h:.1f} h "
                  f"(cumul {_hours:.1f} h) — disjonction test OK")
        if _hours >= SWA_TARGET_HOURS + SWA_DEV_HOURS:
            break

    # dev disjoint par locuteur (~SWA_DEV_HOURS)
    _spk_h = {}
    for _, _, _rows, _ in _sel:
        for r in _rows:
            _spk_h[r["spk"]] = _spk_h.get(r["spk"], 0.0) + r["dur"] / 3600
    _dev_spk, _acc = set(), 0.0
    for _s, _h in sorted(_spk_h.items(), key=lambda kv: kv[1]):
        if _acc >= SWA_DEV_HOURS:
            break
        _dev_spk.add(_s)
        _acc += _h
    print(f"dev spontané : {len(_dev_spk)} locuteurs, {_acc:.1f} h")

    # Phase B — téléchargement + conversion, REPRENABLE par archive
    def _to_flac(args):
        _path, _text, _spk = args
        try:
            _p = subprocess.run(["ffmpeg", "-v", "quiet", "-nostdin", "-i", _path, "-f", "f32le",
                                 "-ac", "1", "-ar", "16000", "pipe:1"],
                                capture_output=True, check=True)
            wav = np.frombuffer(_p.stdout, dtype=np.float32)
            if len(wav) < 3200:
                return None
            _b = _io.BytesIO()
            sf.write(_b, wav, 16000, format="FLAC")
            return {"text": _text, "audio_bytes": _b.getvalue(), "audio_size": len(wav),
                    "_dev": _spk in _dev_spk}
        except Exception:
            return None

    _done_f = os.path.join(SPONT_ROOT, "chunks_done.json")
    _done = set(json.load(open(_done_f))) if os.path.exists(_done_f) else set()
    for _dtrain, _chunk, _rows, _h in _sel:
        _tag = f"{_dtrain}__c{_chunk}"
        if _tag in _done:
            print(f"  {_tag}: déjà converti")
            continue
        _t0 = time.time()
        _tarp = hf_hub_download(_REPO, f"{_dtrain}/audio/audio_{_chunk}.tar.xz",
                                repo_type="dataset")
        _ex = f"/content/_spont_extract_{_chunk}"
        shutil.rmtree(_ex, ignore_errors=True)
        with tarfile.open(_tarp, "r:xz") as _t:
            _t.extractall(_ex)
        _byname = {}
        for _root, _, _fs in os.walk(_ex):
            for _f in _fs:
                _byname[_f] = os.path.join(_root, _f)
        _jobs = [(_byname[r["file"]], r["text"], r["spk"]) for r in _rows if r["file"] in _byname]
        with ThreadPoolExecutor(max_workers=12) as _pool:
            _out = [x for x in _pool.map(_to_flac, _jobs) if x]
        for _split in ("train", "dev"):
            _sub = [{k: v for k, v in x.items() if k != "_dev"}
                    for x in _out if x["_dev"] == (_split == "dev")]
            if not _sub:
                continue
            _dir = os.path.join(SPONT_PARQ, "corpus=afrivoices", f"split={_split}",
                                "language=swh_Latn")
            os.makedirs(_dir, exist_ok=True)
            _pq.write_table(pa.Table.from_pylist(_sub),
                            os.path.join(_dir, f"part-{_tag}.parquet"), row_group_size=100)
        _done.add(_tag)
        save_json(sorted(_done), _done_f)
        shutil.rmtree(_ex, ignore_errors=True)
        os.remove(_tarp)
        print(f"  {_tag}: {len(_out)}/{len(_rows)} clips convertis en "
              f"{(time.time()-_t0)/60:.1f} min")

    # TSV fairseq2 + texte pour LM + marqueur + cache Drive
    _tot = 0
    for _f in glob.glob(os.path.join(SPONT_PARQ, "corpus=afrivoices", "split=train", "*", "*.parquet")):
        _tot += sum(_pq.read_table(_f, columns=["audio_size"])["audio_size"].to_pylist())
    with open(os.path.join(SPONT_ROOT, "language_distribution_0.tsv"), "w") as f:
        f.write("corpus\tlanguage\thours\n")
        f.write(f"afrivoices\tswh_Latn\t{_tot / 16000 / 3600:.4f}\n")
    _txts = []
    for _, _, _rows, _ in _sel:
        _txts += [r["text"] for r in _rows if r["spk"] not in _dev_spk]
    with open(os.path.join(SPONT_ROOT, "swa_spont_text.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(_txts))
    open(os.path.join(SPONT_ROOT, "_COMPLETE"), "w").write(now_iso())
    print(f"train spontané : {_tot / 16000 / 3600:.1f} h — mise en cache Drive…")
    shutil.rmtree(SPONT_DRIVE, ignore_errors=True)
    subprocess.run(["cp", "-r", SPONT_ROOT, SPONT_DRIVE], check=True)

print("Dataset swa spontané prêt :", SPONT_PARQ)
print("Suite : 16.2 (top-up spontané swa).")

In [ ]:
# 16.2 — TOP-UP SPONTANÉ swa : 2000 steps depuis la carte ft5000 sur les 145 h spontanées
# (16.1), validation pendant l'entraînement sur le DEV SPONTANÉ (split dev du parquet).
# Checkpoints à 1000/1500/2000 -> 16.3 arbitre. ~4h20 (A100). REPRENABLE (même cellule).
SPONT_STEPS = 2000
assert "TOPUP_SEED_CARD" in globals(), "Exécuter 13.1 d'abord (carte ft5000 + sonde, ~3 min)."
SPONT_ROOT = "/content/topup_swa_spont"
SPONT_PARQ = os.path.join(SPONT_ROOT, "version=0")
assert os.path.exists(os.path.join(SPONT_ROOT, "_COMPLETE")), \
    "Dataset spontané absent : exécuter 16.1 (restaure depuis le cache Drive si déjà construit)."

# libérer le GPU du noyau (cf. 13.2 : la recette a besoin des 40 GB entiers)
import gc
for _v in ("_pipe_joint", "_pipe_lm", "_pipe"):
    if _v in globals():
        del globals()[_v]
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.clear()
gc.collect()
torch.cuda.empty_cache()
print(f"GPU noyau libéré : {torch.cuda.memory_allocated() / 2**30:.1f} GB restants alloués")

write_dataset_card(data_root=SPONT_PARQ)   # 2.4 remettra la carte sur le dataset complet
_s_tsv = os.path.join(SPONT_ROOT, "language_distribution_0.tsv")
_s_out = os.path.join(FT_CONFIG["experiment_run"], "topup_swa_spont")
_s_yaml = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], "topup_swa_spont_recipe.yaml"),
                            TOPUP_SEED_CARD, _s_tsv, num_steps=SPONT_STEPS,
                            grad_accum=FT_CONFIG["grad_accum"],
                            max_num_elements=FT_CONFIG["max_num_elements"], ckpt_every=500)
rc, _log = run_recipe(_s_out, _s_yaml, "topup_swa_spont.log",
                      wall_limit_h=FT_CONFIG["max_wall_hours_per_session"])
print("Code :", rc, "| dernier checkpoint :", latest_ckpt(_s_out))
print("Suite : 16.3 (double thermomètre : dev spontané + valid agricole).")

In [ ]:
# 16.3 — DOUBLE THERMOMÈTRE swa : joint vs checkpoints spontanés (1000/1500/2000), mesurés
# avec le LM/réglages sw ACTUELS sur (a) 200 clips du DEV SPONTANÉ (16.1) et (b) 200 clips du
# valid agricole. Adoption si le spontané gagne SANS régression >1 pt sur l'agricole (le test
# swa est 100 % spontané, mais prudence). Écrit topup_choice_sw.json -> 13.4 -> 12.3. ~30 min.
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
import pyarrow.parquet as _pq
from pathlib import Path
from pyctcdecode import build_ctcdecoder

SPONT_ROOT = "/content/topup_swa_spont"
SPONT_PARQ = os.path.join(SPONT_ROOT, "version=0")
assert os.path.exists(os.path.join(SPONT_ROOT, "_COMPLETE")), "Exécuter 16.1 d'abord."

def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""
                return L
            except Exception:
                pass
    raise SystemExit("Vocab SentencePiece inaccessible.")
def _capture(pipe, inputs, omni_code):
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps

# jeu (a) : dev spontané — 200 clips depuis le parquet
_dev_rows = []
for _f in sorted(glob.glob(os.path.join(SPONT_PARQ, "corpus=afrivoices", "split=dev",
                                        "*", "*.parquet"))):
    _dev_rows += _pq.read_table(_f).to_pylist()
import random as _rnd
_rnd.Random(FT_CONFIG["seed"]).shuffle(_dev_rows)
_dev_rows = _dev_rows[:200]
_dev_in = [{"waveform": sf.read(io.BytesIO(r["audio_bytes"]), dtype="float32")[0],
            "sample_rate": 16000} for r in _dev_rows]
_dev_refs = [r["text"] for r in _dev_rows]
# jeu (b) : valid agricole (curation)
_agr = (manifest[(manifest.split == "valid") & (manifest.language == "sw")]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
_agr_refs = _agr["text_norm"].tolist()

# LM/réglages sw actuels (v4 Wikipedia, arbitrés en 14.6)
_tun = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
_cfg = _tun["sw"]
_uf = _cfg.get("uni_file")
_uni = (open(_uf, encoding="utf-8").read().splitlines() if _uf and os.path.exists(_uf)
        else sorted({w for t in manifest[(manifest.split == "train")
                                         & (manifest.language == "sw")]["text_norm"]
                     for w in str(t).split()}))
_dec = None                                  # construit au 1er tour (LABELS garanti)
_bw = int(_cfg.get("beam", 100))

_s_out = os.path.join(FT_CONFIG["experiment_run"], "topup_swa_spont")
_steps = sorted((d for d in Path(_s_out).rglob("step_*")
                 if d.is_dir() and not d.name.endswith(".tmp") and (d / "model").exists()),
                key=lambda d: int(d.name.split("_")[1]))
assert _steps, f"Aucun checkpoint sous {_s_out} : exécuter 16.2 d'abord."
_cands = [("joint", latest_ckpt(FT_CONFIG["output_dir"]))] + [(d.name, str(d)) for d in _steps]

_res = {}
for _name, _ck in _cands:
    _pipe = load_finetuned_pipeline(_ck)
    if "LABELS" not in globals():
        LABELS = _labels_from(_pipe)
    if _dec is None:
        _dec = build_ctcdecoder(LABELS, _cfg["lm_bin"], unigrams=_uni,
                                alpha=float(_cfg["alpha"]), beta=float(_cfg["beta"]))
    _lp_dev = _capture(_pipe, _dev_in, omni_lang("sw"))
    w_dev = jiwer.wer(_dev_refs, [normalize_text(_dec.decode(x, beam_width=_bw))
                                  for x in _lp_dev])
    _lp_agr = _capture(_pipe, _agr["curated_audio_path"].tolist(), omni_lang("sw"))
    w_agr = jiwer.wer(_agr_refs, [normalize_text(_dec.decode(x, beam_width=_bw))
                                  for x in _lp_agr])
    _res[_name] = {"ckpt": _ck, "wer_spont": round(w_dev, 4), "wer_agri": round(w_agr, 4)}
    print(f"sw/{_name}: SPONTANÉ {w_dev:.4f} | agricole {w_agr:.4f}")
    del _pipe, _lp_dev, _lp_agr
    torch.cuda.empty_cache()

_best = min(_res, key=lambda n: _res[n]["wer_spont"])
_dspont = _res["joint"]["wer_spont"] - _res[_best]["wer_spont"]
_dagri = _res[_best]["wer_agri"] - _res["joint"]["wer_agri"]
if _best != "joint" and _dspont > 0.005 and _dagri < 0.01:
    save_json({"language": "sw", "chosen": _best, "ckpt": _res[_best]["ckpt"],
               "wer_lm": _res[_best]["wer_spont"], "all": _res},
              os.path.join(FT_CONFIG["report_dir"], "topup_choice_sw.json"))
    print(f"\n✅ {_best} adopté : spontané {_dspont:+.4f}, agricole {_dagri:+.4f} "
          f"-> exécuter 13.4 (TOPUP_LANG_EVAL='sw') puis 12.3, puis SOUMETTRE.")
else:
    print(f"\n⚠️ Pas d'adoption (best={_best}, gain spontané {_dspont:+.4f}, "
          f"régression agricole {_dagri:+.4f}) — me montrer le tableau complet.")

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"   # désactive le backend Xet, cause des hangs à 0%

In [ ]:
# 16.4 — Dataset SPONTANÉ générique depuis anv_data_ke unscripted (kln/mas/som/kik/luo).
# Les shards train contiennent audio (WAV 44,1 k) + transcription -> lecture shard par shard,
# filtres, disjonction test (règle 4b), conversion 16 k FLAC (ffmpeg x12), parquet fairseq2
# train + dev ~2,5 h disjoint par locuteur. REPRENABLE par shard ; cache Drive. ~60-90 min.
SPONT_LANG = "som"            # "kln" puis "mas" puis "som" — vide = cellule inerte
SPONT_TARGET_HOURS = 80
SPONT_DEV_HOURS = 2.5
if not SPONT_LANG:
    raise SystemExit("SPONT_LANG vide : mettre 'kln'/'mas'/'som'/'kik'/'luo' puis relancer.")
assert SPONT_LANG in ("kln", "mas", "som", "kik", "luo"), SPONT_LANG
import subprocess, json, glob, shutil
os.environ["HF_HUB_DISABLE_XET"] = "1"   # hangs de téléchargement HF (backend Xet)
import io as _io
import pyarrow as pa
import pyarrow.parquet as _pq
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import HfApi, hf_hub_download, login

_REPO = "MCAA1-MSU/anv_data_ke"
_tp = next((p for p in ("/content/afrivoices_project",
                        os.path.join(DRIVE_ROOT, "hf_token.json")) if os.path.exists(p)), None)
assert _tp, "hf_token.json introuvable"
login(token=json.load(open(_tp))["key"])

S_ROOT = f"/content/topup_{SPONT_LANG}_spont"
S_PARQ = os.path.join(S_ROOT, "version=0")
S_DRIVE = os.path.join(DRIVE_ROOT, f"topup_{SPONT_LANG}_spont_cache")
if os.path.exists(os.path.join(S_DRIVE, "_COMPLETE")) \
        and not os.path.exists(os.path.join(S_ROOT, "_COMPLETE")):
    print("Cache Drive trouvé : copie locale…")
    shutil.rmtree(S_ROOT, ignore_errors=True)
    subprocess.run(["cp", "-r", S_DRIVE, S_ROOT], check=True)

if not os.path.exists(os.path.join(S_ROOT, "_COMPLETE")):
    _test_ids = set()
    for _l, _sh, _n in test_shards:
        if _l == SPONT_LANG:
            for _i in _pq.read_table(_sh, columns=["id"])["id"].to_pylist():
                _test_ids.add(os.path.basename(str(_i)))
    print(f"{len(_test_ids)} IDs test {SPONT_LANG} chargés (contrôle de disjonction)")

    _dir = f"{SPONT_LANG}/train/unscripted/audios"
    _shards = sorted(f.path for f in HfApi().list_repo_tree(_REPO, _dir, repo_type="dataset")
                     if f.path.endswith(".parquet"))
    print(f"{len(_shards)} shards unscripted disponibles pour {SPONT_LANG}")

    def _to_flac(args):
        raw, text, spk = args
        try:
            _p = subprocess.run(["ffmpeg", "-v", "quiet", "-nostdin", "-i", "pipe:0",
                                 "-f", "f32le", "-ac", "1", "-ar", "16000", "pipe:1"],
                                input=raw, capture_output=True, check=True)
            wav = np.frombuffer(_p.stdout, dtype=np.float32)
            if not (8000 <= len(wav) <= int(38.0 * 16000)):
                return None
            _b = _io.BytesIO()
            sf.write(_b, wav, 16000, format="FLAC")
            return {"text": text, "audio_bytes": _b.getvalue(), "audio_size": len(wav),
                    "_spk": spk}
        except Exception:
            return None

    _done_f = os.path.join(S_ROOT, "shards_done.json")
    os.makedirs(S_ROOT, exist_ok=True)
    _state = json.load(open(_done_f)) if os.path.exists(_done_f) else {"done": [], "hours": 0.0,
                                                                       "dev_spk": None}
    _dev_spk = set(_state["dev_spk"] or [])
    for _shp in _shards:
        if _state["hours"] >= SPONT_TARGET_HOURS + SPONT_DEV_HOURS:
            break
        _tag = os.path.basename(_shp).replace(".parquet", "")
        if _tag in _state["done"]:
            continue
        _t0 = time.time()
        _local = hf_hub_download(_REPO, _shp, repo_type="dataset")
        _tbl = _pq.read_table(_local).to_pylist()
        _jobs = []
        for r in _tbl:
            _fn = os.path.basename(str(r.get("filename") or ""))
            assert _fn not in _test_ids, f"CHEVAUCHEMENT test détecté ({_fn}) — STOP (règle 4b)"
            _txt = normalize_text(str(r.get("transcription") or ""))
            _au = r.get("audio")
            raw = _au.get("bytes") if isinstance(_au, dict) else _au
            if _txt and len(_txt.split()) >= 2 and raw:
                _jobs.append((bytes(raw), _txt, str(r.get("recorder_uuid") or "?")))
        with ThreadPoolExecutor(max_workers=12) as _pool:
            _out = [x for x in _pool.map(_to_flac, _jobs) if x]
        if _state["dev_spk"] is None:            # locuteurs dev fixés au 1er shard
            _sp = {}
            for x in _out:
                _sp[x["_spk"]] = _sp.get(x["_spk"], 0) + x["audio_size"]
            _acc, _dev_spk = 0.0, set()
            for s, n in sorted(_sp.items(), key=lambda kv: kv[1]):
                if _acc >= SPONT_DEV_HOURS:
                    break
                _dev_spk.add(s)
                _acc += n / 16000 / 3600
            _state["dev_spk"] = sorted(_dev_spk)
        for _split in ("train", "dev"):
            _sub = [{k: v for k, v in x.items() if k != "_spk"} for x in _out
                    if (x["_spk"] in _dev_spk) == (_split == "dev")]
            if not _sub:
                continue
            _d = os.path.join(S_PARQ, "corpus=afrivoices", f"split={_split}",
                              f"language={omni_lang(SPONT_LANG)}")
            os.makedirs(_d, exist_ok=True)
            _pq.write_table(pa.Table.from_pylist(_sub),
                            os.path.join(_d, f"part-{_tag}.parquet"), row_group_size=100)
        _h = sum(x["audio_size"] for x in _out if x["_spk"] not in _dev_spk) / 16000 / 3600
        _state["hours"] += _h
        _state["done"].append(_tag)
        save_json(_state, _done_f)
        # purge du cache HF : os.remove(_local) ne supprimait que le symlink, pas le blob
        # de ~700 Mo -> saturation du disque VM après quelques dizaines de shards
        shutil.rmtree(os.path.expanduser(
            "~/.cache/huggingface/hub/datasets--MCAA1-MSU--anv_data_ke"), ignore_errors=True)
        print(f"  {_tag}: {len(_out)}/{len(_tbl)} clips, +{_h:.1f} h "
              f"(cumul {_state['hours']:.1f} h) en {(time.time()-_t0)/60:.1f} min — disjonction OK")

    _tot = 0
    for _f in glob.glob(os.path.join(S_PARQ, "corpus=afrivoices", "split=train", "*", "*.parquet")):
        _tot += sum(_pq.read_table(_f, columns=["audio_size"])["audio_size"].to_pylist())
    with open(os.path.join(S_ROOT, "language_distribution_0.tsv"), "w") as f:
        f.write("corpus\tlanguage\thours\n")
        f.write(f"afrivoices\t{omni_lang(SPONT_LANG)}\t{_tot / 16000 / 3600:.4f}\n")
    open(os.path.join(S_ROOT, "_COMPLETE"), "w").write(now_iso())
    print(f"train spontané {SPONT_LANG} : {_tot / 16000 / 3600:.1f} h — cache Drive…")
    shutil.rmtree(S_DRIVE, ignore_errors=True)
    subprocess.run(["cp", "-r", S_ROOT, S_DRIVE], check=True)
print(f"Dataset spontané {SPONT_LANG} prêt : {S_PARQ} -> 16.5 (top-up)")


In [ ]:
import subprocess
subprocess.run(["cp", "-r", "/content/topup_mas_spont",
                "/content/afrivoices_project/topup_mas_spont_partial"],
               check=True)
print("partiel mas sauvegardé")

In [ ]:
# TEMP — libérer le disque VM
import shutil, subprocess, os
# datasets spontanés déjà en cache Drive -> inutiles en local
for p in ("/content/topup_kln_spont", "/content/topup_swa_spont"):
    shutil.rmtree(p, ignore_errors=True)
# LE vrai coupable : les blobs de shards accumulés dans le cache HF
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/datasets--MCAA1-MSU--anv_data_ke"),
              ignore_errors=True)
shutil.rmtree(os.path.expanduser(
    "~/.cache/huggingface/hub/datasets--DigitalUmuganda--Afrivoice_Swahili"), ignore_errors=True)
subprocess.run(["df", "-h", "/content"])

In [ ]:
# 16.5 — TOP-UP SPONTANÉ générique : 2000 steps depuis la carte ft5000 sur le dataset 16.4,
# validation d'entraînement sur le dev spontané. ~4h20. REPRENABLE (même cellule).
assert SPONT_LANG, "Exécuter 16.4 d'abord (SPONT_LANG)."
assert "TOPUP_SEED_CARD" in globals(), "Exécuter 13.1 d'abord (carte + sonde)."
S_ROOT = f"/content/topup_{SPONT_LANG}_spont"
assert os.path.exists(os.path.join(S_ROOT, "_COMPLETE")), "Exécuter 16.4 d'abord."
import gc
for _v in ("_pipe_joint", "_pipe_lm", "_pipe"):
    if _v in globals():
        del globals()[_v]
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.clear()
gc.collect()
torch.cuda.empty_cache()
write_dataset_card(data_root=os.path.join(S_ROOT, "version=0"))
_o = os.path.join(FT_CONFIG["experiment_run"], f"topup_{SPONT_LANG}_spont")
_y = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], f"topup_{SPONT_LANG}_spont_recipe.yaml"),
                       TOPUP_SEED_CARD, os.path.join(S_ROOT, "language_distribution_0.tsv"),
                       num_steps=2000, grad_accum=FT_CONFIG["grad_accum"],
                       max_num_elements=FT_CONFIG["max_num_elements"], ckpt_every=500)
rc, _log = run_recipe(_o, _y, f"topup_{SPONT_LANG}_spont.log",
                      wall_limit_h=FT_CONFIG["max_wall_hours_per_session"])
print("Code :", rc, "| dernier checkpoint :", latest_ckpt(_o), "-> 16.6")

In [ ]:
# 16.6 — Double thermomètre générique : joint + checkpoint DÉPLOYÉ (top-up scripté éventuel)
# + checkpoints spontanés, sur (a) dev spontané 16.4 et (b) valid scripté, avec le LM/réglages
# actuels de la langue. Adoption si le spontané gagne >1 pt sans régression scriptée >1,5 pt
# (l'unscripted pèse 67-83 % du test en durée). Écrit topup_choice_{lang}.json -> 13.4 -> 12.3.
assert SPONT_LANG, "Exécuter 16.4 d'abord (SPONT_LANG)."
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
import random as _rnd
import pyarrow.parquet as _pq
from pathlib import Path
from pyctcdecode import build_ctcdecoder
S_ROOT = f"/content/topup_{SPONT_LANG}_spont"
S_PARQ = os.path.join(S_ROOT, "version=0")
assert os.path.exists(os.path.join(S_ROOT, "_COMPLETE")), "Exécuter 16.4 d'abord."

def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""
                return L
            except Exception:
                pass
    raise SystemExit("Vocab inaccessible.")
def _capture(pipe, inputs, omni_code):
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps

_dev_rows = []
for _f in sorted(glob.glob(os.path.join(S_PARQ, "corpus=afrivoices", "split=dev", "*", "*.parquet"))):
    _dev_rows += _pq.read_table(_f).to_pylist()
_rnd.Random(FT_CONFIG["seed"]).shuffle(_dev_rows)
_dev_rows = _dev_rows[:200]
_dev_in = [{"waveform": sf.read(io.BytesIO(r["audio_bytes"]), dtype="float32")[0],
            "sample_rate": 16000} for r in _dev_rows]
_dev_refs = [r["text"] for r in _dev_rows]
_scr = (manifest[(manifest.split == "valid") & (manifest.language == SPONT_LANG)]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
_scr_refs = _scr["text_norm"].tolist()

_tun = json.load(open(os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")))
_cfg = _tun[SPONT_LANG]
# les entrées héritées du pré-polish n'ont pas de champ lm_bin -> binaire v1 par défaut
_lm_bin = _cfg.get("lm_bin") or os.path.join(FT_CONFIG["experiment_run"], "kenlm_models",
                                             f"{SPONT_LANG}.binary")
assert os.path.exists(_lm_bin), _lm_bin
_uf = _cfg.get("uni_file")
_uni = (open(_uf, encoding="utf-8").read().splitlines() if _uf and os.path.exists(_uf)
        else sorted({w for t in manifest[(manifest.split == "train")
                                         & (manifest.language == SPONT_LANG)]["text_norm"]
                     for w in str(t).split()}))
_bw = int(_cfg.get("beam", 100))

_cands = [("joint", latest_ckpt(FT_CONFIG["output_dir"]))]
_old_p = os.path.join(FT_CONFIG["report_dir"], f"topup_choice_{SPONT_LANG}.json")
_old = json.load(open(_old_p)) if os.path.exists(_old_p) else None
if _old and _old.get("chosen") != "joint" and os.path.exists(_old["ckpt"]):
    _cands.append((f"déployé({_old['chosen']})", _old["ckpt"]))
_o = os.path.join(FT_CONFIG["experiment_run"], f"topup_{SPONT_LANG}_spont")
_steps = sorted((d for d in Path(_o).rglob("step_*")
                 if d.is_dir() and not d.name.endswith(".tmp") and (d / "model").exists()),
                key=lambda d: int(d.name.split("_")[1]))
assert _steps, "Exécuter 16.5 d'abord."
_cands += [(f"spont_{d.name}", str(d)) for d in _steps]

_dec = None
_res = {}
for _name, _ck in _cands:
    _pipe = load_finetuned_pipeline(_ck)
    if "LABELS" not in globals():
        LABELS = _labels_from(_pipe)
    if _dec is None:
        _dec = build_ctcdecoder(LABELS, _lm_bin, unigrams=_uni,
                                alpha=float(_cfg["alpha"]), beta=float(_cfg["beta"]))
    w_sp = jiwer.wer(_dev_refs, [normalize_text(_dec.decode(x, beam_width=_bw))
                                 for x in _capture(_pipe, _dev_in, omni_lang(SPONT_LANG))])
    w_sc = jiwer.wer(_scr_refs, [normalize_text(_dec.decode(x, beam_width=_bw))
                                 for x in _capture(_pipe, _scr["curated_audio_path"].tolist(),
                                                   omni_lang(SPONT_LANG))])
    _res[_name] = {"ckpt": _ck, "wer_spont": round(w_sp, 4), "wer_script": round(w_sc, 4)}
    print(f"{SPONT_LANG}/{_name}: SPONTANÉ {w_sp:.4f} | scripté {w_sc:.4f}")
    del _pipe
    torch.cuda.empty_cache()

_ref = min((n for n in _res if not n.startswith("spont_")), key=lambda n: _res[n]["wer_spont"])
_best = min(_res, key=lambda n: _res[n]["wer_spont"])
_gain = _res[_ref]["wer_spont"] - _res[_best]["wer_spont"]
_reg = _res[_best]["wer_script"] - _res[_ref]["wer_script"]
if _best.startswith("spont_") and _gain > 0.01 and _reg < 0.015:
    save_json({"language": SPONT_LANG, "chosen": _best, "ckpt": _res[_best]["ckpt"],
               "wer_lm": _res[_best]["wer_spont"], "all": _res}, _old_p)
    print(f"\n✅ {_best} adopté (spontané {_gain:+.4f} vs {_ref}, scripté {_reg:+.4f}) -> "
          f"TOPUP_LANG_EVAL='{SPONT_LANG}' puis 13.4 puis 12.3 puis SOUMETTRE.")
else:
    print(f"\n⚠️ Pas d'adoption (best={_best}, gain {_gain:+.4f}, régression {_reg:+.4f}) — "
          "me montrer le tableau.")

In [ ]:
import json
print(json.dumps(_res, indent=2, ensure_ascii=False))

In [ ]:
# TEMP — adoption manuelle mas : spont_step_2000 domine strictement le déployé (2 thermomètres)
save_json({"language": "mas", "chosen": "spont_step_2000",
           "ckpt": _res["spont_step_2000"]["ckpt"],
           "wer_lm": _res["spont_step_2000"]["wer_spont"], "all": _res},
          os.path.join(FT_CONFIG["report_dir"], "topup_choice_mas.json"))
print("mas -> spont_step_2000 adopté")

In [ ]:
# TEMP — les textes spont kik sont-ils dégénérés ?
import pyarrow.parquet as pq, glob, collections
rows = []
for f in sorted(glob.glob("/content/topup_kik_spont/version=0/corpus=afrivoices/split=train/*/*.parquet"))[:6]:
    rows += pq.read_table(f, columns=["text"])["text"].to_pylist()
c = collections.Counter(rows)
print(f"{len(rows)} textes, {len(c)} uniques ({len(c)/len(rows):.0%})")
for t, n in c.most_common(5):
    print(f"  ×{n}: {t[:90]}")

In [ ]:
# 16.7 — LM v5 « spontané » (générique) : corpus = train scripté + transcriptions spontanées
# 16.4 (hors locuteurs du dev) [+ thinkKenya si accessible, kln/luo] ; réglage alpha/beta sur
# le DEV SPONTANÉ (67-83 % du test en durée) avec l'acoustique DÉPLOYÉE, garde sur le valid
# scripté ; adoption au critère pondéré par la composition du test. ~30-40 min.
assert SPONT_LANG, "Définir SPONT_LANG (16.4) d'abord."
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
import random as _rnd
import pyarrow.parquet as _pq
from pyctcdecode import build_ctcdecoder

_W_UNSCR = {"kln": 0.82, "mas": 0.83, "som": 0.82, "kik": 0.67, "luo": 0.67}[SPONT_LANG]
S_ROOT = f"/content/topup_{SPONT_LANG}_spont"
S_PARQ = os.path.join(S_ROOT, "version=0")
assert os.path.exists(os.path.join(S_ROOT, "_COMPLETE")), "Exécuter 16.4 d'abord."
_lmplz = "/content/kenlm/build/bin/lmplz"
_bbin = "/content/kenlm/build/bin/build_binary"
assert os.path.exists(_lmplz), "Exécuter 12.1b d'abord (compile lmplz)."

# corpus v5 : train scripté + texte spontané (train seulement) + thinkKenya éventuel
LM_V5_DIR = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_v5")
os.makedirs(LM_V5_DIR, exist_ok=True)
_bin5 = os.path.join(LM_V5_DIR, f"{SPONT_LANG}.binary")
_uni5f = os.path.join(LM_V5_DIR, f"{SPONT_LANG}_unigrams.txt")
_scr_txt = [str(t) for t in manifest[(manifest.split == "train")
                                     & (manifest.language == SPONT_LANG)]["text_norm"]]
_sp_txt = []
for _f in glob.glob(os.path.join(S_PARQ, "corpus=afrivoices", "split=train", "*", "*.parquet")):
    _sp_txt += [str(t) for t in _pq.read_table(_f, columns=["text"])["text"].to_pylist()]
_tk_txt = []
if SPONT_LANG in ("kln", "luo"):
    try:
        from datasets import load_dataset
        _ds = load_dataset("thinkKenya/kenyan-low-resource-language-data", f"{SPONT_LANG}_swa")
        for _split in _ds:
            _cols = _ds[_split].column_names
            for _row in _ds[_split]:
                for _k in (SPONT_LANG, "kalenjin", "dholuo"):
                    _v = _row.get(_k) if _k in _cols else None
                    if _v is None:
                        for c in _cols:
                            if isinstance(_row.get(c), dict) and isinstance(_row[c].get(_k), str):
                                _v = _row[c][_k]
                    if isinstance(_v, str):
                        _s = normalize_text(_v)
                        if len(_s.split()) >= 3:
                            _tk_txt.append(_s)
                        break
        print(f"thinkKenya : +{len(_tk_txt)} phrases")
    except Exception as _e:
        print(f"thinkKenya inaccessible ({type(_e).__name__}) — sans")
_sents = _scr_txt * 2 + sorted(set(_sp_txt)) * 2 + sorted(set(_tk_txt))
if not os.path.exists(_bin5):
    _txt, _arpa = f"/content/lm5_{SPONT_LANG}.txt", f"/content/lm5_{SPONT_LANG}.arpa"
    with open(_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(_sents))
    with open(_txt, "rb") as fi, open(_arpa, "wb") as fo:
        _r = subprocess.run([_lmplz, "-o", "5", "--discount_fallback", "-S", "30%"],
                            stdin=fi, stdout=fo, stderr=subprocess.PIPE)
    assert _r.returncode == 0, _r.stderr[-500:].decode(errors="replace")
    subprocess.run([_bbin, _arpa, _bin5], check=True)
    from collections import Counter
    _cnt = Counter(w for s in _sents for w in s.split())
    with open(_uni5f, "w", encoding="utf-8") as f:
        f.write("\n".join(w for w, _ in _cnt.most_common(200_000)))
print(f"LM v5 {SPONT_LANG} : {len(_scr_txt)} scripté + {len(set(_sp_txt))} spontané "
      f"+ {len(set(_tk_txt))} thinkKenya -> {os.path.getsize(_bin5) // 2**20} Mo")

# capture avec l'acoustique DÉPLOYÉE (topup_choice sinon joint)
def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""
                return L
            except Exception:
                pass
    raise SystemExit("Vocab inaccessible.")
def _capture(pipe, inputs, omni_code):
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps

_ch_p = os.path.join(FT_CONFIG["report_dir"], f"topup_choice_{SPONT_LANG}.json")
_ch = json.load(open(_ch_p)) if os.path.exists(_ch_p) else None
_ck = _ch["ckpt"] if _ch and _ch.get("chosen") != "joint" else latest_ckpt(FT_CONFIG["output_dir"])
_pipe = load_finetuned_pipeline(_ck)
if "LABELS" not in globals():
    LABELS = _labels_from(_pipe)

_dev_rows = []
for _f in sorted(glob.glob(os.path.join(S_PARQ, "corpus=afrivoices", "split=dev", "*", "*.parquet"))):
    _dev_rows += _pq.read_table(_f).to_pylist()
_rnd.Random(FT_CONFIG["seed"]).shuffle(_dev_rows)
_dev_rows = _dev_rows[:200]
_lp_dev = _capture(_pipe, [{"waveform": sf.read(io.BytesIO(r["audio_bytes"]), dtype="float32")[0],
                            "sample_rate": 16000} for r in _dev_rows], omni_lang(SPONT_LANG))
_dev_refs = [r["text"] for r in _dev_rows]
_scr = (manifest[(manifest.split == "valid") & (manifest.language == SPONT_LANG)]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(200))
_lp_scr = _capture(_pipe, _scr["curated_audio_path"].tolist(), omni_lang(SPONT_LANG))
_scr_refs = _scr["text_norm"].tolist()
del _pipe
torch.cuda.empty_cache()

_tun_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
_tun = json.load(open(_tun_p))
_cfg = _tun[SPONT_LANG]
_lm_cur = _cfg.get("lm_bin") or os.path.join(FT_CONFIG["experiment_run"], "kenlm_models",
                                             f"{SPONT_LANG}.binary")
_uni_cur = (open(_cfg["uni_file"], encoding="utf-8").read().splitlines()
            if _cfg.get("uni_file") and os.path.exists(_cfg.get("uni_file", ""))
            else sorted({w for t in _scr_txt for w in t.split()}))
_uni5 = open(_uni5f, encoding="utf-8").read().splitlines()
_bw = int(_cfg.get("beam", 100))

def _mix(dec, a=None, b=None):
    w_sp = jiwer.wer(_dev_refs, [normalize_text(dec.decode(x, beam_width=_bw)) for x in _lp_dev])
    w_sc = jiwer.wer(_scr_refs, [normalize_text(dec.decode(x, beam_width=_bw)) for x in _lp_scr])
    return w_sp, w_sc, _W_UNSCR * w_sp + (1 - _W_UNSCR) * w_sc

_dec0 = build_ctcdecoder(LABELS, _lm_cur, unigrams=_uni_cur,
                         alpha=float(_cfg["alpha"]), beta=float(_cfg["beta"]))
sp0, sc0, mix0 = _mix(_dec0)
print(f"ACTUEL (a={_cfg['alpha']}, b={_cfg['beta']}): spont {sp0:.4f} | script {sc0:.4f} "
      f"| mix {mix0:.4f}")
best = None
for _tag, _binx, _unix in (("cur", _lm_cur, _uni_cur), ("v5", _bin5, _uni5)):
    for a in (0.3, 0.5, 0.65, 0.8, 1.0):
        for b in (0.0, 1.0, 2.0):
            d = build_ctcdecoder(LABELS, _binx, unigrams=_unix, alpha=a, beta=b)
            sp, sc, mx = _mix(d)
            if best is None or mx < best[0]:
                best = (mx, _tag, _binx, a, b, sp, sc)
mx1, _tag, _binx, a1, b1, sp1, sc1 = best
print(f"MEILLEUR ({_tag}, a={a1}, b={b1}): spont {sp1:.4f} | script {sc1:.4f} | mix {mx1:.4f}")

if mx1 < mix0 - 0.005:
    _tun[SPONT_LANG] = {"alpha": a1, "beta": b1, "beam": _bw,
                        "lm": ("v5" if _tag == "v5" else _cfg.get("lm", "v1")),
                        "lm_bin": _binx,
                        **({"uni_file": _uni5f} if _tag == "v5" else
                           ({"uni_file": _cfg["uni_file"]} if _cfg.get("uni_file") else {})),
                        "wer_best": round(sp1, 4)}
    save_json(_tun, _tun_p)
    _n = 0
    _dir_code = {"sw": "swa"}.get(SPONT_LANG, SPONT_LANG)
    for f in glob.glob(os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm",
                                    f"{_dir_code}__*.csv")):
        os.remove(f)
        _n += 1
    print(f"\n✅ Adopté (mix {mix0:.4f} -> {mx1:.4f}) ; {_n} CSV invalidés -> "
          "13.4 (pour recharger) puis 12.3 puis SOUMETTRE.")
else:
    print(f"\n⚠️ Gain mix insuffisant ({mix0:.4f} -> {mx1:.4f}) : rien n'est modifié.")

In [ ]:
print(os.path.exists(os.path.join(FT_CONFIG["report_dir"], "topup_choice_sw.json")))

In [ ]:
# TEMP — revert kik -> config v4 (le texte spont kik est du mojibake source)
import json, glob
_tun_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
_tun = json.load(open(_tun_p))
_v4 = os.path.join(FT_CONFIG["experiment_run"], "kenlm_models_v4")
_tun["kik"] = {"alpha": 0.5, "beta": 0.0, "beam": 100, "lm": "v4",
               "lm_bin": os.path.join(_v4, "kik.binary"),
               "uni_file": os.path.join(_v4, "kik_unigrams.txt")}
save_json(_tun, _tun_p)
print("kik -> v4 (a=0.5, b=0.0)")

In [ ]:
# 16.8 — THERMOMÈTRE LONGS + re-arbitrage d'alpha (générique). Fabrique ~100 longs synthétiques
# (3 clips dev concaténés, ~60-90 s), les décode EXACTEMENT comme 12.3 v2 (fenêtres 38 s,
# recouvrement 6 s, logits tronqués/concaténés, une passe beam), et re-choisit alpha sur le mix
# longs(0.6)/spontané court(0.25)/scripté(0.15) — le juge qui manquait (verdict point 2).
# LM et acoustique = ceux actuellement déployés pour la langue. Invalide les CSV si changement.
assert SPONT_LANG, "Définir SPONT_LANG d'abord."
import subprocess, json, glob
try:
    import kenlm, pyctcdecode
except ImportError:
    subprocess.run(["pip", "install", "-q", "pyctcdecode", "kenlm"], check=False)
    import kenlm, pyctcdecode
import jiwer
import random as _rnd
import pyarrow.parquet as _pq
from pyctcdecode import build_ctcdecoder

S_PARQ = os.path.join(f"/content/topup_{SPONT_LANG}_spont", "version=0")
assert os.path.exists(os.path.join(f"/content/topup_{SPONT_LANG}_spont", "_COMPLETE")), \
    "Exécuter 16.4 d'abord (restauration cache)."

def _labels_from(pipe):
    _tkm = getattr(pipe.tokenizer, "_model", None)
    for _m in ("index_to_token", "id_to_piece", "IdToPiece"):
        if _tkm is not None and hasattr(_tkm, _m):
            try:
                L = [str(getattr(_tkm, _m)(i)) for i in range(pipe.tokenizer.vocab_info.size)]
                assert len(L) == len(set(L))
                L[0] = ""
                return L
            except Exception:
                pass
    raise SystemExit("Vocab inaccessible.")
def _capture(pipe, inputs, omni_code):
    caps = []
    _orig = pipe.model.forward
    def _spy(src, bl, *a, **kw):
        out = _orig(src, bl, *a, **kw)
        lg = out[0] if isinstance(out, tuple) else out
        caps.append(torch.log_softmax(lg[0].detach().float(), -1).cpu().numpy())
        return out
    pipe.model.forward = _spy
    try:
        pipe.transcribe(inputs, lang=[omni_code] * len(inputs), batch_size=1)
    finally:
        pipe.model.forward = _orig
    return caps

# acoustique déployée (topup_choice sinon joint)
_ch_p = os.path.join(FT_CONFIG["report_dir"], f"topup_choice_{SPONT_LANG}.json")
_ch = json.load(open(_ch_p)) if os.path.exists(_ch_p) else None
_ck = _ch["ckpt"] if _ch and _ch.get("chosen") != "joint" else latest_ckpt(FT_CONFIG["output_dir"])
_pipe = load_finetuned_pipeline(_ck)
if "LABELS" not in globals():
    LABELS = _labels_from(_pipe)

# jeux : dev court, longs synthétiques (3 clips concaténés), scripté
_dev = []
for _f in sorted(glob.glob(os.path.join(S_PARQ, "corpus=afrivoices", "split=dev", "*", "*.parquet"))):
    _dev += _pq.read_table(_f).to_pylist()
_rnd.Random(FT_CONFIG["seed"]).shuffle(_dev)
_short = _dev[:150]
_short_in = [{"waveform": sf.read(io.BytesIO(r["audio_bytes"]), dtype="float32")[0],
              "sample_rate": 16000} for r in _short]
_short_ref = [r["text"] for r in _short]
_longs, _long_ref = [], []
for i in range(0, min(len(_dev), 300) - 2, 3):
    _ws = [sf.read(io.BytesIO(_dev[i + j]["audio_bytes"]), dtype="float32")[0] for j in range(3)]
    _gap = np.zeros(int(0.3 * 16000), dtype=np.float32)
    _longs.append(np.concatenate([_ws[0], _gap, _ws[1], _gap, _ws[2]]))
    _long_ref.append(" ".join(_dev[i + j]["text"] for j in range(3)))
_longs, _long_ref = _longs[:100], _long_ref[:100]
_scr = (manifest[(manifest.split == "valid") & (manifest.language == SPONT_LANG)]
        .sample(frac=1.0, random_state=FT_CONFIG["seed"]).head(150))
print(f"jeux : {len(_short)} courts | {len(_longs)} longs synthétiques "
      f"(~{np.mean([len(w) for w in _longs]) / 16000:.0f} s) | {len(_scr)} scriptés")

_omni = omni_lang(SPONT_LANG)
_lp_short = _capture(_pipe, _short_in, _omni)
_lp_scr = _capture(_pipe, _scr["curated_audio_path"].tolist(), _omni)
_scr_ref = _scr["text_norm"].tolist()

# fenêtres + logits tronqués/concaténés pour les longs (identique à 12.3 v2)
_WIN_S, _OVL_S = 38.0, 6.0
_HOP_S, _TRIM_S = _WIN_S - _OVL_S, _OVL_S / 2
_long_lp = []
for wav in _longs:
    n, off, segs = len(wav), 0, []
    if n <= int(39.5 * 16000):
        segs = [(wav, False, False)]
    else:
        while off < n:
            seg = wav[off:off + int(_WIN_S * 16000)]
            last = off + int(_WIN_S * 16000) >= n
            if len(seg) >= int(0.2 * 16000):
                segs.append((seg, off > 0, not last))
            if last:
                break
            off += int(_HOP_S * 16000)
    lps = _capture(_pipe, [{"waveform": s, "sample_rate": 16000} for s, _, _ in segs], _omni)
    parts = []
    for (s, t0, t1), lp in zip(segs, lps):
        fps = len(lp) / (len(s) / 16000.0)
        a = int(round(_TRIM_S * fps)) if t0 else 0
        b = len(lp) - (int(round(_TRIM_S * fps)) if t1 else 0)
        parts.append(lp[a:b])
    _long_lp.append(parts[0] if len(parts) == 1 else np.concatenate(parts, axis=0))
del _pipe
torch.cuda.empty_cache()

# grille d'alpha sur le LM déployé, juge = mix longs/court/scripté
_tun_p = os.path.join(FT_CONFIG["report_dir"], "kenlm_tuning.json")
_tun = json.load(open(_tun_p))
_cfg = _tun[SPONT_LANG]
_lm_bin = _cfg.get("lm_bin") or os.path.join(FT_CONFIG["experiment_run"], "kenlm_models",
                                             f"{SPONT_LANG}.binary")
_uf = _cfg.get("uni_file")
_uni = (open(_uf, encoding="utf-8").read().splitlines() if _uf and os.path.exists(_uf)
        else sorted({w for t in manifest[(manifest.split == "train")
                                         & (manifest.language == SPONT_LANG)]["text_norm"]
                     for w in str(t).split()}))
_bw = int(_cfg.get("beam", 100))
_W = (0.60, 0.25, 0.15)
best = None
for a in (0.3, 0.4, 0.5, 0.65, 0.8):
    d = build_ctcdecoder(LABELS, _lm_bin, unigrams=_uni, alpha=a, beta=float(_cfg["beta"]))
    wl = jiwer.wer(_long_ref, [normalize_text(d.decode(x, beam_width=_bw)) for x in _long_lp])
    ws = jiwer.wer(_short_ref, [normalize_text(d.decode(x, beam_width=_bw)) for x in _lp_short])
    wc = jiwer.wer(_scr_ref, [normalize_text(d.decode(x, beam_width=_bw)) for x in _lp_scr])
    mx = _W[0] * wl + _W[1] * ws + _W[2] * wc
    _cur = " <- ACTUEL" if abs(a - float(_cfg["alpha"])) < 1e-9 else ""
    print(f"  a={a}: LONGS {wl:.4f} | court {ws:.4f} | scripté {wc:.4f} | mix {mx:.4f}{_cur}")
    if best is None or mx < best[0]:
        best = (mx, a, wl, ws, wc)
    if abs(a - float(_cfg["alpha"])) < 1e-9:
        _mx_cur = mx

mx1, a1, wl1, ws1, wc1 = best
if a1 != float(_cfg["alpha"]) and mx1 < _mx_cur - 0.003:
    _tun[SPONT_LANG]["alpha"] = a1
    _tun[SPONT_LANG]["wer_best"] = round(ws1, 4)
    save_json(_tun, _tun_p)
    _n = 0
    _dir_code = {"sw": "swa"}.get(SPONT_LANG, SPONT_LANG)
    for f in glob.glob(os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm",
                                    f"{_dir_code}__*.csv")):
        os.remove(f)
        _n += 1
    print(f"\n✅ alpha {SPONT_LANG}: {_cfg['alpha']} -> {a1} (mix longs {_mx_cur:.4f} -> {mx1:.4f}) ; "
          f"{_n} CSV invalidés.")
else:
    print(f"\n= alpha {SPONT_LANG} confirmé à {_cfg['alpha']} par le thermomètre longs.")

In [ ]:
# TEMP — invalidation générale : le décodage v2 (recouvrement) change tous les clips longs,
# donc TOUTES les langues doivent être ré-inférées
import glob
_n = 0
for f in glob.glob(os.path.join(FT_CONFIG["report_dir"], "test_predictions_lm", "*.csv")):
    os.remove(f)
    _n += 1
print(f"{_n} CSV invalidés -> ré-inférence complète en 12.3 v2")

In [ ]:
# 16.9 — TOP-UP MIXED-BATCH (générique) : dataset = spontané 16.4 (copie) + ~30 % de scripté
# (tiré du train curé) pour limiter le catastrophic forgetting (verdict point 3, littérature
# « damage control »), puis 2000 steps depuis la carte ft5000. Sortie : topup_{lang}_mix/.
# À lancer dans une session GPU DÉDIÉE (parallèle-sûr : n'écrit que dans ses dossiers).
assert SPONT_LANG, "Définir SPONT_LANG d'abord."
MIX_SCRIPT_RATIO = 0.30                      # part de scripté visée dans le mélange
assert "TOPUP_SEED_CARD" in globals(), "Exécuter 13.1 d'abord."
assert "build_parquet" not in (None,), ""
import glob, shutil, subprocess
import pyarrow as pa
import pyarrow.parquet as _pq

S_ROOT = f"/content/topup_{SPONT_LANG}_spont"
assert os.path.exists(os.path.join(S_ROOT, "_COMPLETE")), "Exécuter 16.4 d'abord (cache)."
M_ROOT = f"/content/topup_{SPONT_LANG}_mix"
M_PARQ = os.path.join(M_ROOT, "version=0")

if not os.path.exists(os.path.join(M_ROOT, "_COMPLETE")):
    shutil.rmtree(M_ROOT, ignore_errors=True)
    # 1) copie du spontané (train + dev)
    shutil.copytree(os.path.join(S_ROOT, "version=0"), M_PARQ)
    _sp_sz = 0
    for _f in glob.glob(os.path.join(M_PARQ, "corpus=afrivoices", "split=train", "*", "*.parquet")):
        _sp_sz += sum(_pq.read_table(_f, columns=["audio_size"])["audio_size"].to_pylist())
    _sp_h = _sp_sz / 16000 / 3600
    # 2) scripté : ~ ratio/(1-ratio) x heures spontanées, tiré du train curé
    _need_h = _sp_h * MIX_SCRIPT_RATIO / (1 - MIX_SCRIPT_RATIO)
    _grp = (manifest[(manifest.split == "train") & (manifest.language == SPONT_LANG)]
            .sample(frac=1.0, random_state=FT_CONFIG["seed"]))
    _rows_meta, _acc = [], 0.0
    for _, r in _grp.iterrows():
        if _acc >= _need_h * 3600:
            break
        _rows_meta.append(r)
        _acc += float(r["duration"])
    print(f"mix {SPONT_LANG}: {_sp_h:.1f} h spontané + {_acc / 3600:.1f} h scripté "
          f"({len(_rows_meta)} clips)")
    from concurrent.futures import ThreadPoolExecutor
    def _read(pth):
        with open(pth, "rb") as fh:
            return fh.read()
    with ThreadPoolExecutor(max_workers=32) as _pool:
        _blobs = list(_pool.map(_read, [r["curated_audio_path"] for r in _rows_meta]))
    _dir = os.path.join(M_PARQ, "corpus=afrivoices", "split=train",
                        f"language={omni_lang(SPONT_LANG)}")
    _buf, _part = [], 0
    for r, blob in zip(_rows_meta, _blobs):
        _frames = int(sf.info(io.BytesIO(blob)).frames)
        _buf.append({"text": r["text_norm"], "audio_bytes": blob, "audio_size": _frames})
        if len(_buf) >= 2000:
            _pq.write_table(pa.Table.from_pylist(_buf),
                            os.path.join(_dir, f"part-scripted-{_part:03d}.parquet"),
                            row_group_size=100)
            _part += 1
            _buf = []
    if _buf:
        _pq.write_table(pa.Table.from_pylist(_buf),
                        os.path.join(_dir, f"part-scripted-{_part:03d}.parquet"),
                        row_group_size=100)
    _tot = 0
    for _f in glob.glob(os.path.join(M_PARQ, "corpus=afrivoices", "split=train", "*", "*.parquet")):
        _tot += sum(_pq.read_table(_f, columns=["audio_size"])["audio_size"].to_pylist())
    with open(os.path.join(M_ROOT, "language_distribution_0.tsv"), "w") as f:
        f.write("corpus\tlanguage\thours\n")
        f.write(f"afrivoices\t{omni_lang(SPONT_LANG)}\t{_tot / 16000 / 3600:.4f}\n")
    open(os.path.join(M_ROOT, "_COMPLETE"), "w").write(now_iso())
    print(f"dataset mix : {_tot / 16000 / 3600:.1f} h -> {M_PARQ}")

# 3) entraînement
import gc
for _v in ("_pipe_joint", "_pipe_lm", "_pipe"):
    if _v in globals():
        del globals()[_v]
if "TOPUP_PIPES" in globals():
    TOPUP_PIPES.clear()
gc.collect()
torch.cuda.empty_cache()
write_dataset_card(data_root=M_PARQ)
_o = os.path.join(FT_CONFIG["experiment_run"], f"topup_{SPONT_LANG}_mix")
_y = write_recipe_yaml(os.path.join(FT_CONFIG["experiment_run"], f"topup_{SPONT_LANG}_mix_recipe.yaml"),
                       TOPUP_SEED_CARD, os.path.join(M_ROOT, "language_distribution_0.tsv"),
                       num_steps=2000, grad_accum=FT_CONFIG["grad_accum"],
                       max_num_elements=FT_CONFIG["max_num_elements"], ckpt_every=500)
rc, _log = run_recipe(_o, _y, f"topup_{SPONT_LANG}_mix.log",
                      wall_limit_h=FT_CONFIG["max_wall_hours_per_session"])
print("Code :", rc, "| dernier checkpoint :", latest_ckpt(_o))
print(f"Suite : 16.6 avec _t_out pointé sur topup_{SPONT_LANG}_mix — ou dis-le à Claude.")